# FPV Reconstruction

**Use:** paste a direct FPV video URL below, then choose **Runtime → Run all**.

The notebook automatically saves the reconstruction to Google Drive. When it finishes, the **Results** section shows the GLB, CSV files, reconstruction images, trajectory, quality status, and a ZIP containing the important deliverables.

Before the first run:

1. Accept the access conditions for **facebook/VGGT-Omega** on Hugging Face.
2. Create a Hugging Face **read** token.
3. In Colab, open **Secrets** (key icon), add the token with the exact name **`HF_TOKEN`**, and enable notebook access. Never paste the token into a code cell.
4. Select a **T4 GPU** or better.

> XYZ values are relative reconstruction units—not metres, latitude, longitude, or altitude.


In [ ]:
#@title Setup { display-mode: "form" }
from pathlib import Path
from urllib.parse import urlparse
from datetime import datetime, timezone
from IPython.display import display, Video

import json
import platform
import re
import shutil
import subprocess
import sys

import requests
import cv2
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

try:
    from google.colab import drive, files
except ImportError as error:
    raise RuntimeError(
        "המחברת הזאת מיועדת להרצה ב-Google Colab."
    ) from error

FFMPEG_PATH = shutil.which("ffmpeg")

if FFMPEG_PATH is None:
    raise RuntimeError(
        "FFmpeg לא נמצא בסביבת Colab."
    )

environment_df = pd.DataFrame([
    {
        "Python": platform.python_version(),
        "NumPy": np.__version__,
        "pandas": pd.__version__,
        "OpenCV": cv2.__version__,
        "Matplotlib": matplotlib.__version__,
        "FFmpeg": FFMPEG_PATH
    }
])

display(environment_df)


DRIVE_MOUNT_DIR = Path("/content/drive")

drive.mount(
    str(DRIVE_MOUNT_DIR),
    force_remount=False
)

MY_DRIVE_DIR = (
    DRIVE_MOUNT_DIR
    / "MyDrive"
)

if not MY_DRIVE_DIR.is_dir():
    raise FileNotFoundError(
        "Google Drive לא חובר בהצלחה."
    )


PROJECT_FOLDER_NAME = "FPV_Drone_Project"

STAGE1_VERSION = "stage1_candidate_v5_post_replay_scene_gate"

PARENT_BASELINE_VERSION = "stage1_baseline_v1_frozen"

PARENT_CANDIDATE_VERSION = "stage1_candidate_v4_replay_transition_gate"

PARENT_CANDIDATE_NOTEBOOK_SHA256 = "ac50bbc6bf1ed827f7a77fa8ada0d9e55cb0782b4ecb9e190e611a65c8c9f42a"

INPUT_MODE = globals().get("INPUT_MODE", "url")  # url | drive | upload

VIDEO_URL = globals().get("VIDEO_URL", "https://d2fioemadmrru3.cloudfront.net/videos/2026-05-05_strike_on_merkava_tank.mp4")

DRIVE_VIDEO_PATH = globals().get("DRIVE_VIDEO_PATH", "")

FORCE_DOWNLOAD = globals().get("FORCE_DOWNLOAD", False)

RUN_SUFFIX = globals().get("RUN_SUFFIX", "")

PARENT_BASELINE_LOGIC_SHA256 = "c7f8ee6e17359a4f2b5c9e23440f11e077421c8875b81c7a9d017cffc7a1af43"

if INPUT_MODE not in {"url", "drive", "upload"}:
    raise ValueError(
        "INPUT_MODE חייב להיות 'url', 'drive' או 'upload'."
    )


PROJECT_DIR = (
    MY_DRIVE_DIR
    / PROJECT_FOLDER_NAME
)

DRIVE_VIDEOS_DIR = (
    PROJECT_DIR
    / "videos"
)

DRIVE_RUNS_DIR = (
    PROJECT_DIR
    / "runs"
    / STAGE1_VERSION
)

RUNTIME_DIR = Path(
    "/content/fpv_stage1_runtime"
)

INPUT_DIR = RUNTIME_DIR / "input"
FEATURES_DIR = RUNTIME_DIR / "features"
OUTPUT_DIR = RUNTIME_DIR / "output"
TEMP_DIR = RUNTIME_DIR / "temp"

for folder in [
    PROJECT_DIR,
    DRIVE_VIDEOS_DIR,
    DRIVE_RUNS_DIR,
    INPUT_DIR,
    FEATURES_DIR,
    OUTPUT_DIR,
    TEMP_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


## Video

Paste one direct video URL. The recommended baseline settings are already selected.


In [ ]:
#@title Paste video URL { display-mode: "form" }
VIDEO_URL = "" #@param {type:"string"}
if VIDEO_URL.strip():
    print("Ready. Choose Runtime -> Run all.")
else:
    print("Paste a direct video URL, then choose Runtime -> Run all.")


In [ ]:
#@title Advanced options (optional) { display-mode: "form" }


## Run

Choose **Runtime → Run all**. Processing and diagnostics remain collapsed. The final deliverables appear in **Results**.


In [ ]:
def make_safe_filename(filename):

    filename = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        filename
    ).strip("._")

    if not filename:
        filename = "input_video.mp4"

    if not filename.lower().endswith(".mp4"):
        filename += ".mp4"

    return filename

def download_video(
    url,
    output_dir,
    force_download=False
):

    filename = Path(
        urlparse(url).path
    ).name

    filename = make_safe_filename(
        filename
    )

    output_path = output_dir / filename
    temporary_path = output_dir / f"{filename}.part"

    if (
        output_path.exists()
        and output_path.stat().st_size > 0
        and not force_download
    ):
        print("הסרטון כבר קיים בסביבת העבודה:")
        print(output_path)
        return output_path

    if temporary_path.exists():
        temporary_path.unlink()

    print("מוריד את הסרטון...")

    with requests.get(
        url,
        stream=True,
        timeout=(20, 180),
        headers={"User-Agent": "Mozilla/5.0"}
    ) as response:
        response.raise_for_status()

        with open(temporary_path, "wb") as file:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    file.write(chunk)

    if (
        not temporary_path.exists()
        or temporary_path.stat().st_size == 0
    ):
        raise RuntimeError(
            "הסרטון שהורד ריק."
        )

    temporary_path.replace(
        output_path
    )

    print("ההורדה הסתיימה:")
    print(output_path)
    return output_path

def resolve_drive_video_path(path_text):

    drive_path = Path(
        path_text
    ).expanduser()

    if not drive_path.is_absolute():
        drive_path = (
            MY_DRIVE_DIR
            / drive_path
        )

    drive_path = drive_path.resolve()

    if not drive_path.is_file():
        raise FileNotFoundError(
            f"קובץ הסרטון לא נמצא: {drive_path}"
        )

    return drive_path


In [ ]:
if INPUT_MODE == "url":
    if not VIDEO_URL.strip():
        raise ValueError(
            "יש להדביק קישור ישיר בתוך VIDEO_URL."
        )

    VIDEO_PATH = download_video(
        VIDEO_URL,
        INPUT_DIR,
        force_download=FORCE_DOWNLOAD
    )

elif INPUT_MODE == "drive":
    if not DRIVE_VIDEO_PATH.strip():
        raise ValueError(
            "יש להכניס נתיב בתוך DRIVE_VIDEO_PATH."
        )

    source_drive_path = resolve_drive_video_path(
        DRIVE_VIDEO_PATH
    )

    VIDEO_PATH = (
        INPUT_DIR
        / make_safe_filename(source_drive_path.name)
    )

    shutil.copy2(
        source_drive_path,
        VIDEO_PATH
    )

    print("הסרטון הועתק מ-Drive אל /content:")
    print(VIDEO_PATH)

else:
    print("בחר קובץ MP4 אחד להעלאה.")
    uploaded_files = files.upload()

    if len(uploaded_files) != 1:
        raise ValueError(
            "יש להעלות קובץ וידאו אחד בלבד."
        )

    uploaded_name, uploaded_data = next(
        iter(uploaded_files.items())
    )

    VIDEO_PATH = (
        INPUT_DIR
        / make_safe_filename(uploaded_name)
    )

    VIDEO_PATH.write_bytes(
        uploaded_data
    )

VIDEO_PATH = Path(VIDEO_PATH)

if (
    not VIDEO_PATH.is_file()
    or VIDEO_PATH.stat().st_size == 0
):
    raise RuntimeError(
        "קובץ הקלט לא נוצר כראוי."
    )

run_name_parts = [
    VIDEO_PATH.stem
]

if RUN_SUFFIX.strip():
    run_name_parts.append(
        make_safe_filename(RUN_SUFFIX).removesuffix(".mp4")
    )

RUN_NAME = "_".join(run_name_parts)

DRIVE_RUN_DIR = (
    DRIVE_RUNS_DIR
    / RUN_NAME
)

DRIVE_FEATURES_DIR = (
    DRIVE_RUN_DIR
    / "features"
)

DRIVE_OUTPUT_DIR = (
    DRIVE_RUN_DIR
    / "output"
)

for folder in [
    DRIVE_RUN_DIR,
    DRIVE_FEATURES_DIR,
    DRIVE_OUTPUT_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


In [ ]:
def get_video_info(video_path):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise ValueError(
            "OpenCV לא הצליח לפתוח את הסרטון."
        )

    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    width = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    height = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )

    cap.release()

    if fps <= 0:
        raise ValueError(
            "ערך ה-FPS אינו תקין."
        )

    if frame_count <= 0:
        raise ValueError(
            "מספר הפריימים אינו תקין."
        )

    if width <= 0 or height <= 0:
        raise ValueError(
            "רזולוציית הסרטון אינה תקינה."
        )

    duration_sec = (
        frame_count / fps
    )

    return {
        "video_path": str(video_path),
        "fps": fps,
        "frame_count": frame_count,
        "duration_sec": duration_sec,
        "width": width,
        "height": height
    }

VIDEO_INFO = get_video_info(
    VIDEO_PATH
)

video_info_df = pd.DataFrame(
    [VIDEO_INFO]
)

display(video_info_df)


In [ ]:
def read_frame_at_time(
    video_path,
    time_sec
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise ValueError(
            "לא הצלחתי לפתוח את הסרטון."
        )

    cap.set(
        cv2.CAP_PROP_POS_MSEC,
        float(time_sec) * 1000
    )

    success, frame = cap.read()
    cap.release()

    if not success or frame is None:
        return None

    return cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

def show_contact_sheet(
    video_path,
    duration_sec,
    sample_count=12,
    columns=4
):

    times = np.linspace(
        0,
        max(0, duration_sec - 0.1),
        sample_count
    )

    rows = int(
        np.ceil(sample_count / columns)
    )

    fig, axes = plt.subplots(
        rows,
        columns,
        figsize=(16, rows * 3.2)
    )

    axes = np.asarray(
        axes
    ).reshape(-1)

    for axis in axes:
        axis.axis("off")

    for index, time_sec in enumerate(times):
        frame = read_frame_at_time(
            video_path,
            time_sec
        )

        if frame is None:
            axes[index].set_title(
                f"{time_sec:.2f}s\nשגיאת קריאה"
            )
            continue

        axes[index].imshow(
            frame
        )

        axes[index].set_title(
            f"{time_sec:.2f} שניות"
        )

        axes[index].axis("off")

    plt.tight_layout()
    plt.show()

show_contact_sheet(
    VIDEO_PATH,
    VIDEO_INFO["duration_sec"]
)


In [ ]:
SAMPLE_FPS = 3

ANALYSIS_WIDTH = 360

DARK_PIXEL_THRESHOLD = 50

def resize_for_analysis(frame, target_width):

    height, width = frame.shape[:2]

    if width <= target_width:
        return frame.copy()

    scale = target_width / width
    target_height = int(round(height * scale))

    return cv2.resize(
        frame,
        (target_width, target_height),
        interpolation=cv2.INTER_AREA
    )

def make_tracking_mask(height, width):

    mask = np.full(
        (height, width),
        255,
        dtype=np.uint8
    )

    mask[
        int(height * 0.62):height,
        0:int(width * 0.22)
    ] = 0

    mask[
        int(height * 0.42):int(height * 0.58),
        int(width * 0.42):int(width * 0.58)
    ] = 0

    mask[
        0:int(height * 0.25),
        int(width * 0.70):width
    ] = 0

    return mask

def compute_phash_hex(gray_frame):

    small = cv2.resize(
        gray_frame,
        (32, 32),
        interpolation=cv2.INTER_AREA
    )

    dct = cv2.dct(
        small.astype(np.float32)
    )

    low_frequency = dct[:8, :8].flatten()

    median_value = float(
        np.median(low_frequency[1:])
    )

    bits = low_frequency > median_value

    hash_value = 0

    for bit in bits:
        hash_value = (
            hash_value << 1
        ) | int(bit)

    return f"{hash_value:016x}"


In [ ]:
def extract_video_features(video_path):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise ValueError(
            "לא הצלחתי לפתוח את הסרטון."
        )

    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if fps <= 0 or frame_count <= 0:
        cap.release()

        raise ValueError(
            "נתוני הסרטון אינם תקינים."
        )

    sample_step = max(
        1,
        int(round(fps / SAMPLE_FPS))
    )

    estimated_samples = int(
        np.ceil(frame_count / sample_step)
    )

    rows = []

    previous_gray = None
    previous_mask = None

    frame_index = 0
    sample_index = 0
    next_progress = 10

    while True:
        success, frame = cap.read()

        if not success:
            break

        should_sample = (
            frame_index % sample_step == 0
        )

        if not should_sample:
            frame_index += 1
            continue

        time_sec = frame_index / fps

        small_frame = resize_for_analysis(
            frame,
            ANALYSIS_WIDTH
        )

        gray = cv2.cvtColor(
            small_frame,
            cv2.COLOR_BGR2GRAY
        )

        height, width = gray.shape

        tracking_mask = make_tracking_mask(
            height,
            width
        )

        brightness = float(
            gray.mean()
        )

        contrast = float(
            gray.std()
        )

        dark_pixel_ratio = float(
            np.mean(
                gray < DARK_PIXEL_THRESHOLD
            )
        )

        phash_hex = compute_phash_hex(
            gray
        )

        frame_diff = 0.0
        detected_points = 0
        tracked_points = 0
        track_ratio = 0.0
        camera_motion_px = 0.0

        if previous_gray is not None:
            frame_diff = float(
                cv2.absdiff(
                    gray,
                    previous_gray
                ).mean()
            )

            points = cv2.goodFeaturesToTrack(
                previous_gray,
                maxCorners=350,
                qualityLevel=0.01,
                minDistance=7,
                mask=previous_mask
            )

            if points is not None:
                detected_points = len(points)

                next_points, status, _ = (
                    cv2.calcOpticalFlowPyrLK(
                        previous_gray,
                        gray,
                        points,
                        None,
                        winSize=(21, 21),
                        maxLevel=3,
                        criteria=(
                            cv2.TERM_CRITERIA_EPS
                            | cv2.TERM_CRITERIA_COUNT,
                            30,
                            0.01
                        )
                    )
                )

                if (
                    next_points is not None
                    and status is not None
                ):
                    old_points = points.reshape(
                        -1,
                        2
                    )

                    new_points = next_points.reshape(
                        -1,
                        2
                    )

                    valid = (
                        status.reshape(-1) == 1
                    )

                    valid = (
                        valid
                        & np.isfinite(old_points).all(axis=1)
                        & np.isfinite(new_points).all(axis=1)
                    )

                    old_points = old_points[valid]
                    new_points = new_points[valid]

                    tracked_points = len(
                        old_points
                    )

                    if detected_points > 0:
                        track_ratio = (
                            tracked_points
                            / detected_points
                        )

                    if tracked_points >= 5:
                        movement = np.linalg.norm(
                            new_points - old_points,
                            axis=1
                        )

                        if len(movement) > 10:
                            upper_limit = np.percentile(
                                movement,
                                95
                            )

                            movement = movement[
                                movement <= upper_limit
                            ]

                        if len(movement) > 0:
                            camera_motion_px = float(
                                np.median(movement)
                            )

        rows.append({
            "sample_index": sample_index,
            "frame_index": frame_index,
            "time_sec": time_sec,
            "brightness": brightness,
            "contrast": contrast,
            "dark_pixel_ratio": dark_pixel_ratio,
            "frame_diff": frame_diff,
            "detected_points": detected_points,
            "tracked_points": tracked_points,
            "track_ratio": track_ratio,
            "camera_motion_px": camera_motion_px,
            "phash_hex": phash_hex
        })

        previous_gray = gray
        previous_mask = tracking_mask

        sample_index += 1

        progress = int(
            100
            * sample_index
            / max(estimated_samples, 1)
        )

        if progress >= next_progress:
            print(
                f"סריקה: {min(progress, 100)}%"
            )

            next_progress += 10

        frame_index += 1

    cap.release()

    features_df = pd.DataFrame(
        rows
    )

    if features_df.empty:
        raise ValueError(
            "לא נמצאו פריימים לניתוח."
        )

    return features_df


In [ ]:
features_df = extract_video_features(
    VIDEO_PATH
)

required_columns = [
    "time_sec",
    "brightness",
    "contrast",
    "dark_pixel_ratio",
    "frame_diff",
    "detected_points",
    "tracked_points",
    "track_ratio",
    "camera_motion_px",
    "phash_hex"
]

missing_columns = [
    column
    for column in required_columns
    if column not in features_df.columns
]

if missing_columns:
    raise ValueError(
        f"חסרות עמודות: {missing_columns}"
    )

if not features_df[
    "time_sec"
].is_monotonic_increasing:

    raise ValueError(
        "עמודת הזמן אינה מסודרת."
    )

if features_df[
    "phash_hex"
].isna().any():

    raise ValueError(
        "נמצאו ערכי pHash חסרים."
    )

time_differences = (
    features_df["time_sec"]
    .diff()
    .dropna()
)

if time_differences.empty:
    raise ValueError(
        "לא נמצאו מספיק דגימות."
    )

sample_interval_sec = float(
    time_differences.median()
)

actual_sample_fps = (
    1.0 / sample_interval_sec
)

FEATURES_CSV_PATH = (
    FEATURES_DIR
    / f"{VIDEO_PATH.stem}_features.csv"
)

features_df.to_csv(
    FEATURES_CSV_PATH,
    index=False
)

DRIVE_FEATURES_CSV_PATH = (
    DRIVE_FEATURES_DIR
    / FEATURES_CSV_PATH.name
)

shutil.copy2(
    FEATURES_CSV_PATH,
    DRIVE_FEATURES_CSV_PATH
)

display(
    features_df.head()
)

display(pd.DataFrame([{
    "samples": len(features_df),
    "sample_interval_sec": round(sample_interval_sec, 3),
    "sample_fps": round(actual_sample_fps, 3),
    "features_csv": str(DRIVE_FEATURES_CSV_PATH),
}]))


In [ ]:
plt.figure(figsize=(15, 4))

plt.plot(
    features_df["time_sec"],
    features_df["camera_motion_px"]
)

plt.title("Camera motion")
plt.xlabel("Time in seconds")
plt.ylabel("Median movement in pixels")
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(15, 4))

plt.plot(
    features_df["time_sec"],
    features_df["frame_diff"]
)

plt.title("Frame difference")
plt.xlabel("Time in seconds")
plt.ylabel("Mean pixel difference")
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(15, 4))

plt.plot(
    features_df["time_sec"],
    features_df["dark_pixel_ratio"]
)

plt.title("Dark pixel ratio")
plt.xlabel("Time in seconds")
plt.ylabel("Ratio")
plt.ylim(0, 1)
plt.grid(alpha=0.25)
plt.show()

feature_summary = features_df[
    [
        "brightness",
        "contrast",
        "dark_pixel_ratio",
        "frame_diff",
        "detected_points",
        "tracked_points",
        "track_ratio",
        "camera_motion_px"
    ]
].describe().round(3)

display(feature_summary)


In [ ]:
SMOOTH_SECONDS = 1.0
MIN_RELIABLE_TRACKED_POINTS = 20

smooth_window = max(
    3,
    int(round(
        SMOOTH_SECONDS / sample_interval_sec
    ))
)

if smooth_window % 2 == 0:
    smooth_window += 1

analysis_df = features_df.copy()

analysis_df["camera_motion_smooth"] = (
    analysis_df["camera_motion_px"]
    .rolling(
        window=smooth_window,
        center=True,
        min_periods=1
    )
    .median()
)

analysis_df["frame_diff_smooth"] = (
    analysis_df["frame_diff"]
    .rolling(
        window=smooth_window,
        center=True,
        min_periods=1
    )
    .median()
)

analysis_df["dark_pixel_ratio_smooth"] = (
    analysis_df["dark_pixel_ratio"]
    .rolling(
        window=smooth_window,
        center=True,
        min_periods=1
    )
    .median()
)

analysis_df["tracking_reliable"] = (
    analysis_df["tracked_points"]
    >= MIN_RELIABLE_TRACKED_POINTS
)

def find_boolean_intervals(
    time_values,
    flag_values,
    sample_interval
):

    time_values = np.asarray(
        time_values,
        dtype=float
    )

    flag_values = np.asarray(
        flag_values,
        dtype=bool
    )

    intervals = []
    start_sec = None

    for index, flag in enumerate(flag_values):
        current_time = float(
            time_values[index]
        )

        if flag and start_sec is None:
            start_sec = current_time

        interval_ended = (
            start_sec is not None
            and (
                not flag
                or index == len(flag_values) - 1
            )
        )

        if not interval_ended:
            continue

        if flag and index == len(flag_values) - 1:
            end_sec = current_time + sample_interval
        else:
            end_sec = (
                float(time_values[index - 1])
                + sample_interval
            )

        intervals.append({
            "start_sec": float(start_sec),
            "end_sec": float(end_sec)
        })

        start_sec = None

    return intervals

def merge_close_intervals(
    intervals,
    max_gap_sec
):

    if not intervals:
        return []

    sorted_intervals = sorted(
        intervals,
        key=lambda item: item["start_sec"]
    )

    merged = [
        sorted_intervals[0].copy()
    ]

    for interval in sorted_intervals[1:]:
        previous = merged[-1]

        gap_sec = (
            float(interval["start_sec"])
            - float(previous["end_sec"])
        )

        if gap_sec <= max_gap_sec:
            previous["end_sec"] = max(
                float(previous["end_sec"]),
                float(interval["end_sec"])
            )
        else:
            merged.append(
                interval.copy()
            )

    return merged


In [ ]:
DARK_SECTION_THRESHOLD = 0.85
MIN_DARK_SECTION_SECONDS = 1.5
MAX_DARK_GAP_SECONDS = 0.7

MIN_CHAPTER_SECONDS = 2.0

analysis_df["is_dark_sample"] = (
    analysis_df["dark_pixel_ratio_smooth"]
    >= DARK_SECTION_THRESHOLD
)

dark_intervals = find_boolean_intervals(
    analysis_df["time_sec"].to_numpy(),
    analysis_df["is_dark_sample"].to_numpy(),
    sample_interval_sec
)

dark_intervals = merge_close_intervals(
    dark_intervals,
    MAX_DARK_GAP_SECONDS
)

dark_intervals = [
    interval
    for interval in dark_intervals
    if (
        float(interval["end_sec"])
        - float(interval["start_sec"])
    ) >= MIN_DARK_SECTION_SECONDS
]

video_duration_sec = float(
    VIDEO_INFO["duration_sec"]
)

for interval in dark_intervals:
    interval["duration_sec"] = (
        float(interval["end_sec"])
        - float(interval["start_sec"])
    )

    if float(interval["start_sec"]) <= 1.0:
        interval["section_type"] = "INTRO"

    elif (
        float(interval["end_sec"])
        >= video_duration_sec - 1.0
    ):
        interval["section_type"] = "OUTRO"

    else:
        interval["section_type"] = (
            "CHAPTER_SEPARATOR"
        )

dark_intervals_df = pd.DataFrame(
    dark_intervals,
    columns=[
        "start_sec",
        "end_sec",
        "duration_sec",
        "section_type"
    ]
)

chapters = []
chapter_start_sec = 0.0

for interval in dark_intervals:
    chapter_end_sec = float(
        interval["start_sec"]
    )

    chapter_duration_sec = (
        chapter_end_sec
        - chapter_start_sec
    )

    if (
        chapter_duration_sec
        >= MIN_CHAPTER_SECONDS
    ):
        chapters.append({
            "chapter_index": len(chapters),
            "start_sec": chapter_start_sec,
            "end_sec": chapter_end_sec,
            "duration_sec": chapter_duration_sec
        })

    chapter_start_sec = max(
        chapter_start_sec,
        float(interval["end_sec"])
    )

remaining_duration_sec = (
    video_duration_sec
    - chapter_start_sec
)

if remaining_duration_sec >= MIN_CHAPTER_SECONDS:
    chapters.append({
        "chapter_index": len(chapters),
        "start_sec": chapter_start_sec,
        "end_sec": video_duration_sec,
        "duration_sec": remaining_duration_sec
    })

chapters_df = pd.DataFrame(
    chapters,
    columns=[
        "chapter_index",
        "start_sec",
        "end_sec",
        "duration_sec"
    ]
)

display(dark_intervals_df.style.set_caption("אזורים כהים"))

display(chapters_df.style.set_caption("פרקי תוכן"))


In [ ]:
plt.figure(figsize=(15, 4))

plt.plot(
    analysis_df["time_sec"],
    analysis_df["dark_pixel_ratio_smooth"],
    label="dark pixel ratio"
)

plt.axhline(
    DARK_SECTION_THRESHOLD,
    linestyle="--",
    label="dark threshold"
)

for interval in dark_intervals:
    plt.axvspan(
        interval["start_sec"],
        interval["end_sec"],
        alpha=0.2
    )

plt.title("Sustained dark sections")
plt.xlabel("Time in seconds")
plt.ylabel("Dark pixel ratio")
plt.ylim(0, 1)
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
FREEZE_MOTION_MAX_PX = 0.35
FREEZE_FRAME_DIFF_MAX = 2.0

MIN_FREEZE_SECONDS = 0.8
MAX_FREEZE_GAP_SECONDS = 0.40

analysis_df["in_chapter"] = False

for _, chapter in chapters_df.iterrows():
    inside_chapter = (
        (analysis_df["time_sec"] >= chapter["start_sec"])
        & (analysis_df["time_sec"] < chapter["end_sec"])
    )

    analysis_df.loc[
        inside_chapter,
        "in_chapter"
    ] = True

analysis_df["is_freeze_sample"] = (
    analysis_df["in_chapter"]
    & (~analysis_df["is_dark_sample"])
    & analysis_df["tracking_reliable"]
    & (
        analysis_df["camera_motion_smooth"]
        <= FREEZE_MOTION_MAX_PX
    )
    & (
        analysis_df["frame_diff_smooth"]
        <= FREEZE_FRAME_DIFF_MAX
    )
)

freeze_intervals = find_boolean_intervals(
    analysis_df["time_sec"].to_numpy(),
    analysis_df["is_freeze_sample"].to_numpy(),
    sample_interval_sec
)

freeze_intervals = merge_close_intervals(
    freeze_intervals,
    MAX_FREEZE_GAP_SECONDS
)

freeze_intervals = [
    interval
    for interval in freeze_intervals
    if (
        float(interval["end_sec"])
        - float(interval["start_sec"])
    ) >= MIN_FREEZE_SECONDS
]

for interval in freeze_intervals:
    interval["duration_sec"] = (
        float(interval["end_sec"])
        - float(interval["start_sec"])
    )

freeze_intervals_df = pd.DataFrame(
    freeze_intervals,
    columns=[
        "start_sec",
        "end_sec",
        "duration_sec"
    ]
)

display(freeze_intervals_df)


In [ ]:
plt.figure(figsize=(15, 4))

plt.plot(
    analysis_df["time_sec"],
    analysis_df["camera_motion_smooth"],
    label="camera motion"
)

plt.axhline(
    FREEZE_MOTION_MAX_PX,
    linestyle="--",
    label="freeze motion threshold"
)

for interval in freeze_intervals:
    plt.axvspan(
        interval["start_sec"],
        interval["end_sec"],
        alpha=0.2
    )

plt.title("Freeze detection — camera motion")
plt.xlabel("Time in seconds")
plt.ylabel("Movement in pixels")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

plt.figure(figsize=(15, 4))

plt.plot(
    analysis_df["time_sec"],
    analysis_df["frame_diff_smooth"],
    label="frame difference"
)

plt.axhline(
    FREEZE_FRAME_DIFF_MAX,
    linestyle="--",
    label="freeze difference threshold"
)

for interval in freeze_intervals:
    plt.axvspan(
        interval["start_sec"],
        interval["end_sec"],
        alpha=0.2
    )

plt.title("Freeze detection — frame difference")
plt.xlabel("Time in seconds")
plt.ylabel("Mean pixel difference")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
REPLAY_SUPPORT_HASH_DISTANCE = 10
REPLAY_STRONG_HASH_DISTANCE = 8

MIN_REPLAY_TIME_GAP_SEC = 3.5

analysis_df["chapter_index"] = -1

for _, chapter in chapters_df.iterrows():
    inside_chapter = (
        (analysis_df["time_sec"] >= chapter["start_sec"])
        & (analysis_df["time_sec"] < chapter["end_sec"])
    )

    analysis_df.loc[
        inside_chapter,
        "chapter_index"
    ] = int(chapter["chapter_index"])

hash_values = np.array(
    [
        int(hash_text, 16)
        for hash_text in analysis_df["phash_hex"]
    ],
    dtype=object
)

time_values = analysis_df[
    "time_sec"
].to_numpy(dtype=float)

dark_flags = analysis_df[
    "is_dark_sample"
].to_numpy(dtype=bool)

chapter_values = analysis_df[
    "chapter_index"
].to_numpy(dtype=int)

replay_hash_distance = np.full(
    len(analysis_df),
    np.nan
)

replay_match_time_sec = np.full(
    len(analysis_df),
    np.nan
)

for chapter_index in chapters_df["chapter_index"]:
    chapter_indices = np.flatnonzero(
        chapter_values == int(chapter_index)
    )

    for position, target_index in enumerate(
        chapter_indices
    ):
        if dark_flags[target_index]:
            continue

        previous_indices = chapter_indices[
            :position
        ]

        if len(previous_indices) == 0:
            continue

        valid_previous_indices = previous_indices[
            (
                time_values[target_index]
                - time_values[previous_indices]
            ) >= MIN_REPLAY_TIME_GAP_SEC
        ]

        valid_previous_indices = (
            valid_previous_indices[
                ~dark_flags[valid_previous_indices]
            ]
        )

        if len(valid_previous_indices) == 0:
            continue

        distances = np.array(
            [
                (
                    int(hash_values[target_index])
                    ^ int(hash_values[source_index])
                ).bit_count()
                for source_index
                in valid_previous_indices
            ],
            dtype=np.int16
        )

        best_position = int(
            np.argmin(distances)
        )

        best_source_index = int(
            valid_previous_indices[best_position]
        )

        replay_hash_distance[target_index] = float(
            distances[best_position]
        )

        replay_match_time_sec[target_index] = float(
            time_values[best_source_index]
        )

analysis_df["replay_hash_distance"] = (
    replay_hash_distance
)

analysis_df["replay_match_time_sec"] = (
    replay_match_time_sec
)

analysis_df["is_replay_match"] = (
    analysis_df["replay_hash_distance"]
    <= REPLAY_SUPPORT_HASH_DISTANCE
)

analysis_df["is_replay_strong_match"] = (
    analysis_df["replay_hash_distance"]
    <= REPLAY_STRONG_HASH_DISTANCE
)


In [ ]:
# מועמדי Replay שנדחו נשמרים לאבחון אך אינם משפיעים על הסיווג.
MIN_REPLAY_SEQUENCE_SECONDS = 1.5
MAX_REPLAY_SEQUENCE_GAP_SECONDS = 0.70

MIN_REPLAY_MATCH_RATIO = 0.60
MIN_REPLAY_STRONG_MATCH_COUNT = 2
MIN_REPLAY_SOURCE_SPAN_SECONDS = 0.50
MIN_REPLAY_ORDER_RATIO = 0.55

MAX_REPLAY_FREEZE_OVERLAP_RATIO = 0.50

REPLAY_ORDER_TOLERANCE_SEC = (
    sample_interval_sec + 0.01
)

REPLAY_GATE_TARGET_FPS = 25.0
REPLAY_GATE_REFERENCE_SECONDS = 4.0
REPLAY_GATE_LOOKBACK_SECONDS = 1.5
REPLAY_GATE_LOOKAHEAD_SECONDS = 1.5

REPLAY_GATE_MIN_FRAME_DIFF = 24.0
REPLAY_GATE_MAD_MULTIPLIER = 6.0

REPLAY_GATE_DARK_RATIO = 0.70
REPLAY_GATE_MIN_DARK_SECONDS = 0.20

def measure_replay_transition(
    video_path,
    core_start_sec,
    chapter_start_sec,
    chapter_end_sec
):

    default_result = {
        "transition_confirmed": False,
        "transition_method": "insufficient_scan_data",
        "transition_peak_time_sec": np.nan,
        "transition_peak_frame_diff": np.nan,
        "transition_threshold": np.nan,
        "transition_reference_median": np.nan,
        "transition_reference_mad": np.nan,
        "transition_dark_duration_sec": 0.0,
        "transition_effective_fps": np.nan
    }

    scan_start_sec = max(
        float(chapter_start_sec),
        float(core_start_sec)
        - REPLAY_GATE_REFERENCE_SECONDS
    )

    gate_start_sec = max(
        float(chapter_start_sec),
        float(core_start_sec)
        - REPLAY_GATE_LOOKBACK_SECONDS
    )

    gate_end_sec = min(
        float(chapter_end_sec),
        float(core_start_sec)
        + REPLAY_GATE_LOOKAHEAD_SECONDS
    )

    scan_end_sec = gate_end_sec

    if scan_end_sec <= scan_start_sec:
        return default_result

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        return default_result

    fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    frame_count = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    if fps <= 0 or frame_count <= 0:
        cap.release()
        return default_result

    sample_step = max(
        1,
        int(round(
            fps / REPLAY_GATE_TARGET_FPS
        ))
    )

    effective_fps = (
        fps / sample_step
    )

    first_frame_index = max(
        0,
        int(np.floor(
            scan_start_sec * fps
        ))
        - sample_step
    )

    last_frame_index = min(
        frame_count - 1,
        int(np.ceil(
            scan_end_sec * fps
        ))
        + sample_step
    )

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        first_frame_index
    )

    rows = []
    previous_gray = None
    frame_index = first_frame_index

    while frame_index <= last_frame_index:
        success, frame = cap.read()

        if not success:
            break

        if frame_index % sample_step == 0:
            small_frame = resize_for_analysis(
                frame,
                ANALYSIS_WIDTH
            )

            gray = cv2.cvtColor(
                small_frame,
                cv2.COLOR_BGR2GRAY
            )

            if previous_gray is not None:
                time_sec = frame_index / fps

                rows.append({
                    "time_sec": time_sec,
                    "frame_diff": float(
                        cv2.absdiff(
                            gray,
                            previous_gray
                        ).mean()
                    ),
                    "dark_pixel_ratio": float(
                        np.mean(
                            gray < DARK_PIXEL_THRESHOLD
                        )
                    )
                })

            previous_gray = gray

        frame_index += 1

    cap.release()

    scan_df = pd.DataFrame(
        rows
    )

    if scan_df.empty:
        return default_result

    reference_rows = scan_df[
        (scan_df["time_sec"] >= scan_start_sec)
        & (scan_df["time_sec"] < gate_start_sec)
    ].copy()

    gate_rows = scan_df[
        (scan_df["time_sec"] >= gate_start_sec)
        & (scan_df["time_sec"] <= gate_end_sec)
    ].copy()

    if gate_rows.empty:
        return default_result

    if reference_rows.empty:
        reference_median = 0.0
        reference_mad = 0.0
    else:
        reference_values = reference_rows[
            "frame_diff"
        ].to_numpy(dtype=float)

        reference_median = float(
            np.median(reference_values)
        )

        reference_mad = float(
            np.median(
                np.abs(
                    reference_values
                    - reference_median
                )
            )
        )

    transition_threshold = max(
        REPLAY_GATE_MIN_FRAME_DIFF,
        reference_median
        + REPLAY_GATE_MAD_MULTIPLIER
        * reference_mad
    )

    peak_index = gate_rows[
        "frame_diff"
    ].idxmax()

    peak_row = gate_rows.loc[
        peak_index
    ]

    peak_time_sec = float(
        peak_row["time_sec"]
    )

    peak_frame_diff = float(
        peak_row["frame_diff"]
    )

    has_strong_change = (
        peak_frame_diff
        >= transition_threshold
    )

    dark_flags = (
        gate_rows["dark_pixel_ratio"]
        >= REPLAY_GATE_DARK_RATIO
    )

    dark_intervals = find_boolean_intervals(
        gate_rows["time_sec"].to_numpy(),
        dark_flags.to_numpy(),
        1.0 / effective_fps
    )

    if dark_intervals:
        dark_duration_sec = max(
            float(interval["end_sec"])
            - float(interval["start_sec"])
            for interval in dark_intervals
        )
    else:
        dark_duration_sec = 0.0

    has_dark_transition = (
        dark_duration_sec
        >= REPLAY_GATE_MIN_DARK_SECONDS
    )

    transition_confirmed = (
        has_strong_change
        or has_dark_transition
    )

    if has_strong_change and has_dark_transition:
        transition_method = (
            "strong_change_and_dark"
        )
    elif has_strong_change:
        transition_method = (
            "strong_frame_change"
        )
    elif has_dark_transition:
        transition_method = (
            "short_dark_transition"
        )
    else:
        transition_method = (
            "no_edit_transition"
        )

    return {
        "transition_confirmed": bool(
            transition_confirmed
        ),
        "transition_method": transition_method,
        "transition_peak_time_sec": peak_time_sec,
        "transition_peak_frame_diff": peak_frame_diff,
        "transition_threshold": float(
            transition_threshold
        ),
        "transition_reference_median": (
            reference_median
        ),
        "transition_reference_mad": reference_mad,
        "transition_dark_duration_sec": float(
            dark_duration_sec
        ),
        "transition_effective_fps": float(
            effective_fps
        )
    }

replay_intervals = []
rejected_freeze_overlap_replays = []
rejected_no_transition_replays = []

for _, chapter in chapters_df.iterrows():
    chapter_index = int(
        chapter["chapter_index"]
    )

    chapter_rows = analysis_df[
        analysis_df["chapter_index"]
        == chapter_index
    ].copy()

    if chapter_rows.empty:
        continue

    raw_intervals = find_boolean_intervals(
        chapter_rows["time_sec"].to_numpy(),
        chapter_rows["is_replay_match"].to_numpy(),
        sample_interval_sec
    )

    candidate_intervals = merge_close_intervals(
        raw_intervals,
        MAX_REPLAY_SEQUENCE_GAP_SECONDS
    )

    for candidate in candidate_intervals:
        start_sec = float(
            candidate["start_sec"]
        )

        end_sec = min(
            float(candidate["end_sec"]),
            float(chapter["end_sec"])
        )

        duration_sec = (
            end_sec - start_sec
        )

        if (
            duration_sec
            < MIN_REPLAY_SEQUENCE_SECONDS
        ):
            continue

        candidate_rows = chapter_rows[
            (chapter_rows["time_sec"] >= start_sec)
            & (chapter_rows["time_sec"] < end_sec)
        ].copy()

        if candidate_rows.empty:
            continue

        match_ratio = float(
            candidate_rows[
                "is_replay_match"
            ].mean()
        )

        strong_match_count = int(
            candidate_rows[
                "is_replay_strong_match"
            ].sum()
        )

        matched_rows = candidate_rows[
            candidate_rows["is_replay_match"]
            & candidate_rows[
                "replay_match_time_sec"
            ].notna()
        ].copy()

        if len(matched_rows) < 2:
            continue

        source_times = matched_rows[
            "replay_match_time_sec"
        ].to_numpy(dtype=float)

        source_start_sec = float(
            source_times.min()
        )

        source_end_sec = float(
            source_times.max()
        )

        source_span_sec = (
            source_end_sec
            - source_start_sec
        )

        source_differences = np.diff(
            source_times
        )

        if len(source_differences) == 0:
            forward_ratio = 0.0
            reverse_ratio = 0.0

        else:
            forward_ratio = float(
                np.mean(
                    source_differences
                    >= -REPLAY_ORDER_TOLERANCE_SEC
                )
            )

            reverse_ratio = float(
                np.mean(
                    source_differences
                    <= REPLAY_ORDER_TOLERANCE_SEC
                )
            )

        if forward_ratio >= reverse_ratio:
            order_ratio = forward_ratio
            direction = "forward"

        else:
            order_ratio = reverse_ratio
            direction = "reverse"

        freeze_overlap_sec = 0.0

        for freeze_interval in freeze_intervals:
            overlap_start_sec = max(
                start_sec,
                float(freeze_interval["start_sec"])
            )

            overlap_end_sec = min(
                end_sec,
                float(freeze_interval["end_sec"])
            )

            freeze_overlap_sec += max(
                0.0,
                overlap_end_sec - overlap_start_sec
            )

        freeze_overlap_ratio = min(
            1.0,
            freeze_overlap_sec
            / max(duration_sec, 1e-9)
        )

        has_replay_evidence = (
            match_ratio
            >= MIN_REPLAY_MATCH_RATIO
            and strong_match_count
            >= MIN_REPLAY_STRONG_MATCH_COUNT
            and source_span_sec
            >= MIN_REPLAY_SOURCE_SPAN_SECONDS
            and order_ratio
            >= MIN_REPLAY_ORDER_RATIO
        )

        blocked_by_freeze_overlap = (
            freeze_overlap_ratio
            >= MAX_REPLAY_FREEZE_OVERLAP_RATIO
        )

        transition_metrics = {
            "transition_confirmed": False,
            "transition_method": "not_evaluated",
            "transition_peak_time_sec": np.nan,
            "transition_peak_frame_diff": np.nan,
            "transition_threshold": np.nan,
            "transition_reference_median": np.nan,
            "transition_reference_mad": np.nan,
            "transition_dark_duration_sec": 0.0,
            "transition_effective_fps": np.nan
        }

        if (
            has_replay_evidence
            and not blocked_by_freeze_overlap
        ):
            transition_metrics = (
                measure_replay_transition(
                    VIDEO_PATH,
                    start_sec,
                    float(chapter["start_sec"]),
                    float(chapter["end_sec"])
                )
            )

        has_transition_evidence = bool(
            transition_metrics[
                "transition_confirmed"
            ]
        )

        is_valid_replay = (
            has_replay_evidence
            and not blocked_by_freeze_overlap
            and has_transition_evidence
        )

        if (
            has_replay_evidence
            and blocked_by_freeze_overlap
        ):
            rejected_freeze_overlap_replays.append({
                "chapter_index": chapter_index,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "duration_sec": duration_sec,
                "match_ratio": match_ratio,
                "strong_match_count": strong_match_count,
                "source_start_sec": source_start_sec,
                "source_end_sec": source_end_sec,
                "freeze_overlap_sec": freeze_overlap_sec,
                "freeze_overlap_ratio": freeze_overlap_ratio,
                "rejection_reason": "majority_freeze_overlap"
            })

        if (
            has_replay_evidence
            and not blocked_by_freeze_overlap
            and not has_transition_evidence
        ):
            rejected_no_transition_replays.append({
                "chapter_index": chapter_index,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "duration_sec": duration_sec,
                "match_ratio": match_ratio,
                "strong_match_count": strong_match_count,
                "source_start_sec": source_start_sec,
                "source_end_sec": source_end_sec,
                "source_span_sec": source_span_sec,
                "order_ratio": order_ratio,
                **transition_metrics,
                "rejection_reason": (
                    "missing_edit_transition"
                )
            })

        if not is_valid_replay:
            continue

        replay_intervals.append({
            "chapter_index": chapter_index,
            "start_sec": start_sec,
            "end_sec": end_sec,
            "duration_sec": duration_sec,
            "match_ratio": match_ratio,
            "strong_match_count": strong_match_count,
            "source_start_sec": source_start_sec,
            "source_end_sec": source_end_sec,
            "source_span_sec": source_span_sec,
            "freeze_overlap_sec": freeze_overlap_sec,
            "freeze_overlap_ratio": freeze_overlap_ratio,
            "order_ratio": order_ratio,
            "direction": direction,
            **transition_metrics
        })

replay_intervals_df = pd.DataFrame(
    replay_intervals,
    columns=[
        "chapter_index",
        "start_sec",
        "end_sec",
        "duration_sec",
        "match_ratio",
        "strong_match_count",
        "source_start_sec",
        "source_end_sec",
        "source_span_sec",
        "freeze_overlap_sec",
        "freeze_overlap_ratio",
        "order_ratio",
        "direction",
        "transition_confirmed",
        "transition_method",
        "transition_peak_time_sec",
        "transition_peak_frame_diff",
        "transition_threshold",
        "transition_reference_median",
        "transition_reference_mad",
        "transition_dark_duration_sec",
        "transition_effective_fps"
    ]
)

rejected_freeze_overlap_replays_df = pd.DataFrame(
    rejected_freeze_overlap_replays,
    columns=[
        "chapter_index",
        "start_sec",
        "end_sec",
        "duration_sec",
        "match_ratio",
        "strong_match_count",
        "source_start_sec",
        "source_end_sec",
        "freeze_overlap_sec",
        "freeze_overlap_ratio",
        "rejection_reason"
    ]
)

rejected_no_transition_replays_df = pd.DataFrame(
    rejected_no_transition_replays,
    columns=[
        "chapter_index",
        "start_sec",
        "end_sec",
        "duration_sec",
        "match_ratio",
        "strong_match_count",
        "source_start_sec",
        "source_end_sec",
        "source_span_sec",
        "order_ratio",
        "transition_confirmed",
        "transition_method",
        "transition_peak_time_sec",
        "transition_peak_frame_diff",
        "transition_threshold",
        "transition_reference_median",
        "transition_reference_mad",
        "transition_dark_duration_sec",
        "transition_effective_fps",
        "rejection_reason"
    ]
)

analysis_df["is_replay_match_for_classification"] = (
    analysis_df["is_replay_match"].copy()
)

for rejected in rejected_no_transition_replays:
    inside_rejected_candidate = (
        (
            analysis_df["chapter_index"]
            == rejected["chapter_index"]
        )
        & (
            analysis_df["time_sec"]
            >= rejected["start_sec"]
        )
        & (
            analysis_df["time_sec"]
            < rejected["end_sec"]
        )
    )

    analysis_df.loc[
        inside_rejected_candidate,
        "is_replay_match_for_classification"
    ] = False

analysis_df["is_replay_core_sample"] = False

for interval in replay_intervals:
    inside_replay = (
        (
            analysis_df["chapter_index"]
            == interval["chapter_index"]
        )
        & (
            analysis_df["time_sec"]
            >= interval["start_sec"]
        )
        & (
            analysis_df["time_sec"]
            < interval["end_sec"]
        )
    )

    analysis_df.loc[
        inside_replay,
        "is_replay_core_sample"
    ] = True

display(replay_intervals_df.style.set_caption("Replay מאושר"))

display(rejected_freeze_overlap_replays_df.style.set_caption("נדחה: חפיפה ל־Freeze"))

display(rejected_no_transition_replays_df.style.set_caption("נדחה: ללא מעבר עריכה"))


In [ ]:
plt.figure(figsize=(15, 4))

plt.plot(
    analysis_df["time_sec"],
    analysis_df["replay_hash_distance"],
    marker=".",
    markersize=3,
    linewidth=1,
    label="best previous pHash distance"
)

plt.axhline(
    REPLAY_SUPPORT_HASH_DISTANCE,
    linestyle="--",
    label="supporting match"
)

plt.axhline(
    REPLAY_STRONG_HASH_DISTANCE,
    linestyle=":",
    label="strong match"
)

for interval in replay_intervals:
    plt.axvspan(
        interval["start_sec"],
        interval["end_sec"],
        alpha=0.2
    )

plt.title("Replay sequence detection")
plt.xlabel("Time in seconds")
plt.ylabel("pHash distance")
plt.ylim(0, 32)
plt.legend()
plt.grid(alpha=0.25)
plt.show()


In [ ]:
REPLAY_ENVELOPE_LOOKBACK_SECONDS = 3.5

REPLAY_TRANSITION_DARK_RATIO = 0.70
REPLAY_TRANSITION_MIN_DARK_SECONDS = 0.25
REPLAY_TRANSITION_MAX_GAP_SECONDS = 0.35
REPLAY_TRANSITION_MAX_GAP_TO_CORE_SECONDS = 2.5

REPLAY_EDIT_MIN_FRAME_DIFF = 45.0
REPLAY_EDIT_MAD_MULTIPLIER = 5.0
REPLAY_EDIT_MAX_TRACK_RATIO = 0.70

MAX_REPLAY_ENVELOPE_GAP_SECONDS = 0.70

replay_envelopes_raw = []

for replay_index, replay in replay_intervals_df.iterrows():
    chapter_index = int(
        replay["chapter_index"]
    )

    core_start_sec = float(
        replay["start_sec"]
    )

    core_end_sec = float(
        replay["end_sec"]
    )

    search_start_sec = max(
        float(
            chapters_df.loc[
                chapters_df["chapter_index"] == chapter_index,
                "start_sec"
            ].iloc[0]
        ),
        core_start_sec
        - REPLAY_ENVELOPE_LOOKBACK_SECONDS
    )

    lookback_rows = analysis_df[
        (analysis_df["chapter_index"] == chapter_index)
        & (analysis_df["time_sec"] >= search_start_sec)
        & (analysis_df["time_sec"] < core_start_sec)
    ].copy()

    envelope_start_sec = core_start_sec
    start_method = "replay_core_fallback"

    if not lookback_rows.empty:
        dark_transition_flags = (
            lookback_rows["dark_pixel_ratio_smooth"]
            >= REPLAY_TRANSITION_DARK_RATIO
        )

        dark_transition_intervals = find_boolean_intervals(
            lookback_rows["time_sec"].to_numpy(),
            dark_transition_flags.to_numpy(),
            sample_interval_sec
        )

        dark_transition_intervals = merge_close_intervals(
            dark_transition_intervals,
            REPLAY_TRANSITION_MAX_GAP_SECONDS
        )

        valid_dark_transitions = [
            interval
            for interval in dark_transition_intervals
            if (
                float(interval["end_sec"])
                - float(interval["start_sec"])
            ) >= REPLAY_TRANSITION_MIN_DARK_SECONDS
            and (
                core_start_sec
                - float(interval["end_sec"])
            ) <= REPLAY_TRANSITION_MAX_GAP_TO_CORE_SECONDS
        ]

        if valid_dark_transitions:
            selected_dark_transition = max(
                valid_dark_transitions,
                key=lambda interval: interval["end_sec"]
            )

            envelope_start_sec = float(
                selected_dark_transition["start_sec"]
            )

            start_method = "short_dark_transition"

    if (
        start_method == "replay_core_fallback"
        and not lookback_rows.empty
    ):
        local_frame_diff = (
            lookback_rows["frame_diff_smooth"]
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(local_frame_diff) > 0:
            local_median = float(
                np.median(local_frame_diff)
            )

            local_mad = float(
                np.median(
                    np.abs(
                        local_frame_diff
                        - local_median
                    )
                )
            )

            edit_threshold = max(
                REPLAY_EDIT_MIN_FRAME_DIFF,
                local_median
                + REPLAY_EDIT_MAD_MULTIPLIER
                * local_mad
            )

            strong_edit_rows = lookback_rows[
                (
                    lookback_rows["frame_diff_smooth"]
                    >= edit_threshold
                )
                & (
                    lookback_rows["track_ratio"]
                    <= REPLAY_EDIT_MAX_TRACK_RATIO
                )
            ]

            if not strong_edit_rows.empty:
                selected_edit_row = (
                    strong_edit_rows
                    .sort_values("time_sec")
                    .iloc[-1]
                )

                envelope_start_sec = float(
                    selected_edit_row["time_sec"]
                )

                start_method = "strong_edit_transition"

    replay_envelopes_raw.append({
        "replay_index": int(replay_index),
        "chapter_index": chapter_index,
        "core_start_sec": core_start_sec,
        "core_end_sec": core_end_sec,
        "envelope_start_sec": envelope_start_sec,
        "envelope_end_sec": core_end_sec,
        "start_extension_sec": (
            core_start_sec
            - envelope_start_sec
        ),
        "match_ratio": float(
            replay["match_ratio"]
        ),
        "strong_match_count": int(
            replay["strong_match_count"]
        ),
        "start_method": start_method
    })

replay_envelopes_df = pd.DataFrame(
    replay_envelopes_raw,
    columns=[
        "replay_index",
        "chapter_index",
        "core_start_sec",
        "core_end_sec",
        "envelope_start_sec",
        "envelope_end_sec",
        "start_extension_sec",
        "match_ratio",
        "strong_match_count",
        "start_method"
    ]
)

replay_envelopes = []

for chapter_index in chapters_df["chapter_index"]:
    chapter_envelopes = [
        {
            "chapter_index": int(chapter_index),
            "start_sec": float(row["envelope_start_sec"]),
            "end_sec": float(row["envelope_end_sec"])
        }
        for _, row in replay_envelopes_df[
            replay_envelopes_df["chapter_index"]
            == int(chapter_index)
        ].iterrows()
    ]

    chapter_envelopes = merge_close_intervals(
        chapter_envelopes,
        MAX_REPLAY_ENVELOPE_GAP_SECONDS
    )

    for interval in chapter_envelopes:
        interval["chapter_index"] = int(
            chapter_index
        )

        replay_envelopes.append(
            interval
        )

analysis_df["is_replay_envelope_sample"] = False

for interval in replay_envelopes:
    inside_replay_envelope = (
        (
            analysis_df["chapter_index"]
            == interval["chapter_index"]
        )
        & (
            analysis_df["time_sec"]
            >= interval["start_sec"]
        )
        & (
            analysis_df["time_sec"]
            < interval["end_sec"]
        )
    )

    analysis_df.loc[
        inside_replay_envelope,
        "is_replay_envelope_sample"
    ] = True

display(replay_envelopes_df)


In [ ]:
analysis_df["state"] = "OUTSIDE_CHAPTER"

analysis_df.loc[
    analysis_df["in_chapter"],
    "state"
] = "UNCLASSIFIED"

for interval in dark_intervals:
    inside_dark_section = (
        (analysis_df["time_sec"] >= interval["start_sec"])
        & (analysis_df["time_sec"] < interval["end_sec"])
    )

    analysis_df.loc[
        inside_dark_section,
        "state"
    ] = "TITLE_OUTRO"

analysis_df.loc[
    analysis_df["is_freeze_sample"],
    "state"
] = "FREEZE"

analysis_df.loc[
    analysis_df["is_replay_envelope_sample"],
    "state"
] = "REPLAY"

state_intervals = []

current_state = str(
    analysis_df.iloc[0]["state"]
)

current_chapter_index = int(
    analysis_df.iloc[0]["chapter_index"]
)

interval_start_sec = float(
    analysis_df.iloc[0]["time_sec"]
)

for row_index in range(
    1,
    len(analysis_df)
):
    row_state = str(
        analysis_df.iloc[row_index]["state"]
    )

    row_chapter_index = int(
        analysis_df.iloc[row_index]["chapter_index"]
    )

    state_changed = (
        row_state != current_state
    )

    chapter_changed = (
        row_chapter_index
        != current_chapter_index
    )

    if not (
        state_changed
        or chapter_changed
    ):
        continue

    interval_end_sec = float(
        analysis_df.iloc[row_index]["time_sec"]
    )

    state_intervals.append({
        "chapter_index": current_chapter_index,
        "start_sec": interval_start_sec,
        "end_sec": interval_end_sec,
        "duration_sec": (
            interval_end_sec
            - interval_start_sec
        ),
        "state": current_state
    })

    interval_start_sec = interval_end_sec
    current_state = row_state
    current_chapter_index = row_chapter_index

final_interval_end_sec = min(
    float(VIDEO_INFO["duration_sec"]),
    float(
        analysis_df.iloc[-1]["time_sec"]
    )
    + sample_interval_sec
)

state_intervals.append({
    "chapter_index": current_chapter_index,
    "start_sec": interval_start_sec,
    "end_sec": final_interval_end_sec,
    "duration_sec": (
        final_interval_end_sec
        - interval_start_sec
    ),
    "state": current_state
})

state_intervals_df = pd.DataFrame(
    state_intervals,
    columns=[
        "chapter_index",
        "start_sec",
        "end_sec",
        "duration_sec",
        "state"
    ]
)

display(state_intervals_df.style.set_caption("מקטעי המצבים"))

display(
    analysis_df["state"]
    .value_counts()
    .rename_axis("state")
    .reset_index(name="sample_count")
)


In [ ]:
SCENE_CUT_MIN_DIFF = 40.0
SCENE_CUT_QUANTILE = 0.985
SCENE_CUT_CLUSTER_GAP_SECONDS = 0.70

analysis_df["is_scene_cut_sample"] = False

scene_cut_records = []

for _, chapter in chapters_df.iterrows():
    chapter_index = int(
        chapter["chapter_index"]
    )

    chapter_rows = analysis_df[
        analysis_df["chapter_index"] == chapter_index
    ].sort_values("time_sec").copy()

    if len(chapter_rows) < 2:
        continue

    eligible_rows = chapter_rows.iloc[1:].copy()

    valid_differences = (
        eligible_rows["frame_diff"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .to_numpy(dtype=float)
    )

    if len(valid_differences) == 0:
        continue

    adaptive_threshold = float(
        np.quantile(
            valid_differences,
            SCENE_CUT_QUANTILE
        )
    )

    scene_cut_threshold = max(
        SCENE_CUT_MIN_DIFF,
        adaptive_threshold
    )

    candidate_rows = eligible_rows[
        eligible_rows["frame_diff"]
        >= scene_cut_threshold
    ].copy()

    if candidate_rows.empty:
        continue

    candidate_indices = list(
        candidate_rows.index
    )

    clusters = []
    current_cluster = [
        candidate_indices[0]
    ]

    for row_index in candidate_indices[1:]:
        previous_index = current_cluster[-1]

        gap_sec = (
            float(
                analysis_df.loc[
                    row_index,
                    "time_sec"
                ]
            )
            - float(
                analysis_df.loc[
                    previous_index,
                    "time_sec"
                ]
            )
        )

        if gap_sec <= SCENE_CUT_CLUSTER_GAP_SECONDS:
            current_cluster.append(
                row_index
            )
        else:
            clusters.append(
                current_cluster
            )

            current_cluster = [
                row_index
            ]

    clusters.append(
        current_cluster
    )

    for cluster in clusters:
        selected_index = max(
            cluster,
            key=lambda index: float(
                analysis_df.loc[
                    index,
                    "frame_diff"
                ]
            )
        )

        analysis_df.loc[
            selected_index,
            "is_scene_cut_sample"
        ] = True

        scene_cut_records.append({
            "chapter_index": chapter_index,
            "time_sec": float(
                analysis_df.loc[
                    selected_index,
                    "time_sec"
                ]
            ),
            "frame_diff": float(
                analysis_df.loc[
                    selected_index,
                    "frame_diff"
                ]
            ),
            "frame_diff_smooth": float(
                analysis_df.loc[
                    selected_index,
                    "frame_diff_smooth"
                ]
            ),
            "track_ratio": float(
                analysis_df.loc[
                    selected_index,
                    "track_ratio"
                ]
            ),
            "threshold": scene_cut_threshold
        })

scene_cuts_df = pd.DataFrame(
    scene_cut_records,
    columns=[
        "chapter_index",
        "time_sec",
        "frame_diff",
        "frame_diff_smooth",
        "track_ratio",
        "threshold"
    ]
).sort_values(
    ["chapter_index", "time_sec"],
    ignore_index=True
)

display(scene_cuts_df)


In [ ]:
block_boundaries = [0]

for row_position in range(
    1,
    len(analysis_df)
):
    current_row = analysis_df.iloc[
        row_position
    ]

    previous_row = analysis_df.iloc[
        row_position - 1
    ]

    state_changed = (
        str(current_row["state"])
        != str(previous_row["state"])
    )

    chapter_changed = (
        int(current_row["chapter_index"])
        != int(previous_row["chapter_index"])
    )

    starts_with_scene_cut = bool(
        current_row["is_scene_cut_sample"]
    )

    if (
        state_changed
        or chapter_changed
        or starts_with_scene_cut
    ):
        block_boundaries.append(
            row_position
        )

block_boundaries.append(
    len(analysis_df)
)

blocks = []

video_duration_sec = float(
    VIDEO_INFO["duration_sec"]
)

for block_number in range(
    len(block_boundaries) - 1
):
    start_position = block_boundaries[
        block_number
    ]

    end_position = block_boundaries[
        block_number + 1
    ]

    block_rows = analysis_df.iloc[
        start_position:end_position
    ].copy()

    if block_rows.empty:
        continue

    start_sec = float(
        block_rows.iloc[0]["time_sec"]
    )

    if end_position < len(analysis_df):
        end_sec = float(
            analysis_df.iloc[
                end_position
            ]["time_sec"]
        )
    else:
        end_sec = min(
            video_duration_sec,
            float(
                block_rows.iloc[-1]["time_sec"]
            )
            + sample_interval_sec
        )

    duration_sec = (
        end_sec - start_sec
    )

    if duration_sec <= 0:
        continue

    reliable_motion = block_rows.loc[
        block_rows["tracking_reliable"],
        "camera_motion_smooth"
    ]

    if not reliable_motion.empty:
        median_camera_motion = float(
            reliable_motion.median()
        )
    else:
        median_camera_motion = float(
            block_rows[
                "camera_motion_smooth"
            ].median()
        )

    median_frame_diff = float(
        block_rows[
            "frame_diff_smooth"
        ].median()
    )

    median_dark_ratio = float(
        block_rows[
            "dark_pixel_ratio_smooth"
        ].median()
    )

    reliable_tracking_ratio = float(
        block_rows[
            "tracking_reliable"
        ].mean()
    )

    replay_match_ratio = float(
        block_rows[
            "is_replay_match_for_classification"
        ].mean()
    )

    blocks.append({
        "block_id": len(blocks) + 1,
        "chapter_index": int(
            block_rows.iloc[0][
                "chapter_index"
            ]
        ),
        "start_sec": start_sec,
        "end_sec": end_sec,
        "duration_sec": duration_sec,
        "state": str(
            block_rows.iloc[0]["state"]
        ),
        "starts_with_scene_cut": bool(
            block_rows.iloc[0][
                "is_scene_cut_sample"
            ]
        ),
        "median_camera_motion": (
            median_camera_motion
        ),
        "median_frame_diff": (
            median_frame_diff
        ),
        "median_dark_ratio": (
            median_dark_ratio
        ),
        "reliable_tracking_ratio": (
            reliable_tracking_ratio
        ),
        "replay_match_ratio": (
            replay_match_ratio
        )
    })

blocks_df = pd.DataFrame(
    blocks,
    columns=[
        "block_id",
        "chapter_index",
        "start_sec",
        "end_sec",
        "duration_sec",
        "state",
        "starts_with_scene_cut",
        "median_camera_motion",
        "median_frame_diff",
        "median_dark_ratio",
        "reliable_tracking_ratio",
        "replay_match_ratio"
    ]
)

display(blocks_df)


In [ ]:
MIN_FLIGHT_BLOCK_SECONDS = 2.0
MIN_FLIGHT_MOTION_PX = 0.50
MIN_FLIGHT_TRACKING_RATIO = 0.65
MAX_FLIGHT_DARK_RATIO = 0.65
MAX_FLIGHT_REPLAY_MATCH_RATIO = 0.25

TRANSITION_MAX_SECONDS = 3.0
TRANSITION_MIN_FRAME_DIFF = 45.0
TRANSITION_MAX_TRACKING_RATIO = 0.75
TRANSITION_MIN_DARK_RATIO = 0.25
TRANSITION_MIN_SIGNALS = 2

POST_REPLAY_NEW_FLIGHT_MIN_SECONDS = 6.0
POST_REPLAY_MIN_MOTION_PX = 1.0
POST_REPLAY_MIN_TRACKING_RATIO = 0.80
POST_REPLAY_MAX_DARK_RATIO = 0.25
POST_REPLAY_MAX_MATCH_RATIO = 0.15

POST_REPLAY_REQUIRE_NEW_SCENE = True

RESUME_MAX_FREEZE_TO_REPLAY_GAP_SEC = 1.0
RESUME_MAX_REPLAY_DURATION_SEC = 3.0
RESUME_MAX_DELAY_AFTER_REPLAY_SEC = 1.5

RESUME_FLIGHT_MIN_SECONDS = 3.0
RESUME_TAIL_SECONDS = 1.5

RESUME_TAIL_MIN_UNIQUE_RATIO = 0.80
RESUME_TAIL_MIN_MOTION_PX = 0.80
RESUME_TAIL_MIN_TRACKING_RATIO = 0.75
RESUME_TAIL_MAX_DARK_RATIO = 0.30

def get_block_tail_metrics(
    start_sec,
    end_sec,
    tail_seconds
):
    tail_start_sec = max(
        float(start_sec),
        float(end_sec) - tail_seconds
    )

    tail_rows = analysis_df[
        (analysis_df["time_sec"] >= tail_start_sec)
        & (analysis_df["time_sec"] < float(end_sec))
    ].copy()

    if tail_rows.empty:
        return {
            "unique_ratio": 0.0,
            "motion": 0.0,
            "tracking_ratio": 0.0,
            "dark_ratio": 1.0
        }

    replay_flags = (
        tail_rows["is_replay_match_for_classification"]
        .fillna(False)
        .astype(bool)
    )

    unique_ratio = float(
        (~replay_flags).mean()
    )

    reliable_motion = tail_rows.loc[
        tail_rows["tracking_reliable"],
        "camera_motion_smooth"
    ]

    if reliable_motion.empty:
        motion = 0.0
    else:
        motion = float(
            reliable_motion.median()
        )

    tracking_ratio = float(
        tail_rows["tracking_reliable"].mean()
    )

    dark_ratio = float(
        tail_rows[
            "dark_pixel_ratio_smooth"
        ].median()
    )

    return {
        "unique_ratio": unique_ratio,
        "motion": motion,
        "tracking_ratio": tracking_ratio,
        "dark_ratio": dark_ratio
    }


In [ ]:
# אחרי Replay נדרשת סצנה חדשה; המשך מאומת אחרי Freeze נשמר בנפרד.
classified_blocks_df = (
    blocks_df
    .sort_values(
        ["chapter_index", "start_sec"]
    )
    .reset_index(drop=True)
    .copy()
)

classified_blocks_df["coarse_state"] = (
    classified_blocks_df["state"]
)

classified_blocks_df["classification_reason"] = ""
classified_blocks_df["post_replay_active"] = False
classified_blocks_df["resume_overlap_active"] = False
classified_blocks_df["post_replay_transition_seen"] = False
classified_blocks_df["post_replay_new_scene_evidence"] = False

classified_blocks_df["tail_unique_ratio"] = np.nan
classified_blocks_df["tail_motion"] = np.nan
classified_blocks_df["tail_tracking_ratio"] = np.nan
classified_blocks_df["tail_dark_ratio"] = np.nan

current_chapter = None

post_replay_active = False
resume_overlap_active = False
post_replay_transition_seen = False

last_freeze_end_sec = None
last_replay_end_sec = None

for position in range(
    len(classified_blocks_df)
):
    row = classified_blocks_df.iloc[
        position
    ]

    chapter_index = int(
        row["chapter_index"]
    )

    if chapter_index != current_chapter:
        current_chapter = chapter_index

        post_replay_active = False
        resume_overlap_active = False
        post_replay_transition_seen = False

        last_freeze_end_sec = None
        last_replay_end_sec = None

    original_state = str(
        row["state"]
    )

    if original_state != "UNCLASSIFIED":
        classified_blocks_df.loc[
            position,
            "coarse_state"
        ] = original_state

        classified_blocks_df.loc[
            position,
            "classification_reason"
        ] = "already_detected"

        if original_state == "FREEZE":
            last_freeze_end_sec = float(
                row["end_sec"]
            )

        elif original_state == "REPLAY":
            replay_start_sec = float(
                row["start_sec"]
            )

            replay_end_sec = float(
                row["end_sec"]
            )

            replay_duration_sec = float(
                row["duration_sec"]
            )

            post_replay_active = True
            post_replay_transition_seen = False
            last_replay_end_sec = replay_end_sec

            freeze_to_replay_gap_sec = np.inf

            if last_freeze_end_sec is not None:
                freeze_to_replay_gap_sec = (
                    replay_start_sec
                    - last_freeze_end_sec
                )

            resume_overlap_active = (
                0.0
                <= freeze_to_replay_gap_sec
                <= RESUME_MAX_FREEZE_TO_REPLAY_GAP_SEC
                and replay_duration_sec
                <= RESUME_MAX_REPLAY_DURATION_SEC
            )

        elif original_state in {
            "TITLE_OUTRO",
            "OUTSIDE_CHAPTER"
        }:
            post_replay_active = False
            resume_overlap_active = False
            post_replay_transition_seen = False

            last_freeze_end_sec = None
            last_replay_end_sec = None

        classified_blocks_df.loc[
            position,
            "post_replay_active"
        ] = post_replay_active

        classified_blocks_df.loc[
            position,
            "resume_overlap_active"
        ] = resume_overlap_active

        classified_blocks_df.loc[
            position,
            "post_replay_transition_seen"
        ] = post_replay_transition_seen

        continue

    duration_sec = float(
        row["duration_sec"]
    )

    median_motion = float(
        row["median_camera_motion"]
    )

    median_frame_diff = float(
        row["median_frame_diff"]
    )

    median_dark_ratio = float(
        row["median_dark_ratio"]
    )

    tracking_ratio = float(
        row["reliable_tracking_ratio"]
    )

    replay_match_ratio = float(
        row["replay_match_ratio"]
    )

    starts_with_scene_cut = bool(
        row["starts_with_scene_cut"]
    )

    has_post_replay_new_scene_evidence = (
        starts_with_scene_cut
        or post_replay_transition_seen
    )

    classified_blocks_df.loc[
        position,
        "post_replay_new_scene_evidence"
    ] = has_post_replay_new_scene_evidence

    tail_metrics = get_block_tail_metrics(
        row["start_sec"],
        row["end_sec"],
        RESUME_TAIL_SECONDS
    )

    classified_blocks_df.loc[
        position,
        "tail_unique_ratio"
    ] = tail_metrics["unique_ratio"]

    classified_blocks_df.loc[
        position,
        "tail_motion"
    ] = tail_metrics["motion"]

    classified_blocks_df.loc[
        position,
        "tail_tracking_ratio"
    ] = tail_metrics["tracking_ratio"]

    classified_blocks_df.loc[
        position,
        "tail_dark_ratio"
    ] = tail_metrics["dark_ratio"]

    is_strong_flight_candidate = (
        duration_sec >= MIN_FLIGHT_BLOCK_SECONDS
        and median_motion >= MIN_FLIGHT_MOTION_PX
        and tracking_ratio >= MIN_FLIGHT_TRACKING_RATIO
        and median_dark_ratio < MAX_FLIGHT_DARK_RATIO
    )

    transition_signal_count = 0

    if median_frame_diff >= TRANSITION_MIN_FRAME_DIFF:
        transition_signal_count += 1

    if tracking_ratio <= TRANSITION_MAX_TRACKING_RATIO:
        transition_signal_count += 1

    if median_dark_ratio >= TRANSITION_MIN_DARK_RATIO:
        transition_signal_count += 1

    is_transition = (
        starts_with_scene_cut
        and duration_sec <= TRANSITION_MAX_SECONDS
        and transition_signal_count
        >= TRANSITION_MIN_SIGNALS
    )

    is_new_flight_after_replay = (
        duration_sec
        >= POST_REPLAY_NEW_FLIGHT_MIN_SECONDS
        and median_motion
        >= POST_REPLAY_MIN_MOTION_PX
        and tracking_ratio
        >= POST_REPLAY_MIN_TRACKING_RATIO
        and median_dark_ratio
        < POST_REPLAY_MAX_DARK_RATIO
        and replay_match_ratio
        < POST_REPLAY_MAX_MATCH_RATIO
        and (
            not POST_REPLAY_REQUIRE_NEW_SCENE
            or has_post_replay_new_scene_evidence
        )
    )

    replay_to_block_gap_sec = np.inf

    if last_replay_end_sec is not None:
        replay_to_block_gap_sec = (
            float(row["start_sec"])
            - last_replay_end_sec
        )

    is_resume_after_freeze_replay = (
        resume_overlap_active
        and 0.0
        <= replay_to_block_gap_sec
        <= RESUME_MAX_DELAY_AFTER_REPLAY_SEC
        and duration_sec
        >= RESUME_FLIGHT_MIN_SECONDS
        and is_strong_flight_candidate
        and tail_metrics["unique_ratio"]
        >= RESUME_TAIL_MIN_UNIQUE_RATIO
        and tail_metrics["motion"]
        >= RESUME_TAIL_MIN_MOTION_PX
        and tail_metrics["tracking_ratio"]
        >= RESUME_TAIL_MIN_TRACKING_RATIO
        and tail_metrics["dark_ratio"]
        < RESUME_TAIL_MAX_DARK_RATIO
    )

    if is_transition:
        coarse_state = "TRANSITION"

        if post_replay_active:
            post_replay_transition_seen = True

        reason = (
            f"scene_cut_with_"
            f"{transition_signal_count}_signals"
        )

    elif post_replay_active:

        if is_resume_after_freeze_replay:
            coarse_state = "FLIGHT"
            reason = (
                "confirmed_resume_after_"
                "freeze_replay"
            )

            post_replay_active = False
            resume_overlap_active = False
            post_replay_transition_seen = False

        elif (
            is_strong_flight_candidate
            and is_new_flight_after_replay
        ):
            coarse_state = "FLIGHT"
            reason = (
                "confirmed_new_flight_"
                "after_replay"
            )

            post_replay_active = False
            resume_overlap_active = False
            post_replay_transition_seen = False

        else:
            coarse_state = "EDIT"

            if not has_post_replay_new_scene_evidence:
                reason = "post_replay_without_new_scene"
            else:
                reason = "post_replay_content"

    elif (
        is_strong_flight_candidate
        and replay_match_ratio
        < MAX_FLIGHT_REPLAY_MATCH_RATIO
    ):
        coarse_state = "FLIGHT"
        reason = "strong_flight_candidate"

    else:
        coarse_state = "EDIT"
        reason = "not_enough_flight_evidence"

    classified_blocks_df.loc[
        position,
        "coarse_state"
    ] = coarse_state

    classified_blocks_df.loc[
        position,
        "classification_reason"
    ] = reason

    classified_blocks_df.loc[
        position,
        "post_replay_active"
    ] = post_replay_active

    classified_blocks_df.loc[
        position,
        "resume_overlap_active"
    ] = resume_overlap_active

    classified_blocks_df.loc[
        position,
        "post_replay_transition_seen"
    ] = post_replay_transition_seen

columns_to_show = [
    "block_id",
    "chapter_index",
    "start_sec",
    "end_sec",
    "duration_sec",
    "state",
    "coarse_state",
    "classification_reason",
    "starts_with_scene_cut",
    "post_replay_active",
    "resume_overlap_active",
    "post_replay_transition_seen",
    "post_replay_new_scene_evidence",
    "median_camera_motion",
    "median_frame_diff",
    "median_dark_ratio",
    "reliable_tracking_ratio",
    "replay_match_ratio"
]

display(
    classified_blocks_df[
        columns_to_show
    ]
)


In [ ]:
BOUNDARY_REFINE_FPS = 12
BOUNDARY_REFINE_WIDTH = 360

BOUNDARY_LOOKBACK_SECONDS = 1.5
BOUNDARY_LOOKAHEAD_SECONDS = 0.8

BOUNDARY_MIN_FRAME_DIFF = 45.0
BOUNDARY_MAD_MULTIPLIER = 5.0
BOUNDARY_MAX_TRACK_RATIO = 0.75

BOUNDARY_DARK_RATIO = 0.70
BOUNDARY_DARK_MIN_FRAME_DIFF = 10.0

def scan_boundary_area(
    video_path,
    start_sec,
    end_sec
):
    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise ValueError(
            "לא הצלחתי לפתוח את הסרטון."
        )

    native_fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    if native_fps <= 0:
        cap.release()

        raise ValueError(
            "ערך ה-FPS אינו תקין."
        )

    sample_step = max(
        1,
        int(round(
            native_fps / BOUNDARY_REFINE_FPS
        ))
    )

    start_frame = max(
        0,
        int(np.floor(
            start_sec * native_fps
        ))
    )

    end_frame = int(np.ceil(
        end_sec * native_fps
    ))

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        start_frame
    )

    rows = []

    previous_gray = None
    previous_mask = None

    frame_index = start_frame

    while frame_index <= end_frame:
        success, frame = cap.read()

        if not success:
            break

        should_sample = (
            (frame_index - start_frame)
            % sample_step == 0
        )

        if not should_sample:
            frame_index += 1
            continue

        small_frame = resize_for_analysis(
            frame,
            BOUNDARY_REFINE_WIDTH
        )

        gray = cv2.cvtColor(
            small_frame,
            cv2.COLOR_BGR2GRAY
        )

        height, width = gray.shape

        tracking_mask = make_tracking_mask(
            height,
            width
        )

        frame_diff = np.nan
        track_ratio = 0.0

        if previous_gray is not None:
            frame_diff = float(
                cv2.absdiff(
                    gray,
                    previous_gray
                ).mean()
            )

            points = cv2.goodFeaturesToTrack(
                previous_gray,
                maxCorners=350,
                qualityLevel=0.01,
                minDistance=7,
                mask=previous_mask
            )

            if points is not None:
                next_points, status, _ = (
                    cv2.calcOpticalFlowPyrLK(
                        previous_gray,
                        gray,
                        points,
                        None,
                        winSize=(21, 21),
                        maxLevel=3,
                        criteria=(
                            cv2.TERM_CRITERIA_EPS
                            | cv2.TERM_CRITERIA_COUNT,
                            30,
                            0.01
                        )
                    )
                )

                if (
                    next_points is not None
                    and status is not None
                ):
                    detected_points = len(points)

                    tracked_points = int(
                        np.sum(
                            status.reshape(-1) == 1
                        )
                    )

                    if detected_points > 0:
                        track_ratio = (
                            tracked_points
                            / detected_points
                        )

        rows.append({
            "time_sec": (
                frame_index / native_fps
            ),
            "frame_diff": frame_diff,
            "track_ratio": track_ratio,
            "dark_pixel_ratio": float(
                np.mean(
                    gray < DARK_PIXEL_THRESHOLD
                )
            )
        })

        previous_gray = gray
        previous_mask = tracking_mask

        frame_index += 1

    cap.release()

    return pd.DataFrame(rows)


In [ ]:
sorted_blocks = (
    classified_blocks_df
    .sort_values(
        ["chapter_index", "start_sec"]
    )
    .reset_index(drop=True)
)

refined_boundaries = []

for position in range(
    len(sorted_blocks) - 1
):
    current_block = sorted_blocks.iloc[
        position
    ]

    next_block = sorted_blocks.iloc[
        position + 1
    ]

    if (
        int(current_block["chapter_index"])
        != int(next_block["chapter_index"])
    ):
        continue

    if (
        str(current_block["coarse_state"])
        != "FLIGHT"
    ):
        continue

    if (
        str(next_block["coarse_state"])
        == "FLIGHT"
    ):
        continue

    coarse_cut_sec = float(
        current_block["end_sec"]
    )

    search_start_sec = max(
        float(current_block["start_sec"]),
        coarse_cut_sec
        - BOUNDARY_LOOKBACK_SECONDS
    )

    search_end_sec = min(
        float(next_block["end_sec"]),
        coarse_cut_sec
        + BOUNDARY_LOOKAHEAD_SECONDS
    )

    scan_df = scan_boundary_area(
        VIDEO_PATH,
        search_start_sec,
        search_end_sec
    )

    refined_cut_sec = coarse_cut_sec
    trigger = "coarse_boundary_fallback"
    frame_diff_threshold = (
        BOUNDARY_MIN_FRAME_DIFF
    )

    if not scan_df.empty:
        baseline_values = (
            scan_df.loc[
                scan_df["time_sec"]
                < coarse_cut_sec,
                "frame_diff"
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan
            )
            .dropna()
            .to_numpy(dtype=float)
        )

        if len(baseline_values) > 0:
            baseline_median = float(
                np.median(
                    baseline_values
                )
            )

            baseline_mad = float(
                np.median(
                    np.abs(
                        baseline_values
                        - baseline_median
                    )
                )
            )

            frame_diff_threshold = max(
                BOUNDARY_MIN_FRAME_DIFF,
                baseline_median
                + BOUNDARY_MAD_MULTIPLIER
                * baseline_mad
            )

        candidate_rows = scan_df[
            (
                (
                    scan_df["frame_diff"]
                    >= frame_diff_threshold
                )
                & (
                    scan_df["track_ratio"]
                    <= BOUNDARY_MAX_TRACK_RATIO
                )
            )
            |
            (
                (
                    scan_df["dark_pixel_ratio"]
                    >= BOUNDARY_DARK_RATIO
                )
                & (
                    scan_df["frame_diff"]
                    >= BOUNDARY_DARK_MIN_FRAME_DIFF
                )
            )
        ].copy()

        if not candidate_rows.empty:
            selected_row = (
                candidate_rows
                .sort_values("time_sec")
                .iloc[0]
            )

            refined_cut_sec = float(
                selected_row["time_sec"]
            )

            if (
                selected_row["dark_pixel_ratio"]
                >= BOUNDARY_DARK_RATIO
            ):
                trigger = "dark_transition"

            else:
                trigger = "strong_edit_transition"

    refined_boundaries.append({
        "previous_flight_block_id": int(
            current_block["block_id"]
        ),
        "next_block_id": int(
            next_block["block_id"]
        ),
        "coarse_cut_sec": coarse_cut_sec,
        "refined_cut_sec": refined_cut_sec,
        "adjustment_sec": (
            refined_cut_sec
            - coarse_cut_sec
        ),
        "frame_diff_threshold": (
            frame_diff_threshold
        ),
        "trigger": trigger
    })

refined_boundaries_df = pd.DataFrame(
    refined_boundaries,
    columns=[
        "previous_flight_block_id",
        "next_block_id",
        "coarse_cut_sec",
        "refined_cut_sec",
        "adjustment_sec",
        "frame_diff_threshold",
        "trigger"
    ]
)

display(refined_boundaries_df)


In [ ]:
FREEZE_REFINE_FPS = 12
FREEZE_REFINE_WIDTH = 360

FREEZE_REFINE_LOOKBACK_SECONDS = 1.6
FREEZE_REFINE_LOOKAHEAD_SECONDS = 1.6

FREEZE_REFINE_MOTION_MAX_PX = 0.45
FREEZE_REFINE_FRAME_DIFF_MAX = 3.0
FREEZE_REFINE_MIN_TRACKED_POINTS = 20

FREEZE_REFINE_MAX_GAP_SECONDS = 0.25
FREEZE_REFINE_MIN_SECONDS = 0.50

def scan_freeze_area(
    video_path,
    start_sec,
    end_sec
):

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise ValueError(
            "לא הצלחתי לפתוח את הסרטון."
        )

    native_fps = float(
        cap.get(cv2.CAP_PROP_FPS)
    )

    if native_fps <= 0:
        cap.release()

        raise ValueError(
            "ערך ה-FPS אינו תקין."
        )

    sample_step = max(
        1,
        int(round(
            native_fps / FREEZE_REFINE_FPS
        ))
    )

    start_frame = max(
        0,
        int(np.floor(
            start_sec * native_fps
        ))
    )

    end_frame = int(np.ceil(
        end_sec * native_fps
    ))

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        start_frame
    )

    rows = []

    previous_gray = None
    previous_mask = None

    frame_index = start_frame

    while frame_index <= end_frame:
        success, frame = cap.read()

        if not success:
            break

        should_sample = (
            (frame_index - start_frame)
            % sample_step == 0
        )

        if not should_sample:
            frame_index += 1
            continue

        small_frame = resize_for_analysis(
            frame,
            FREEZE_REFINE_WIDTH
        )

        gray = cv2.cvtColor(
            small_frame,
            cv2.COLOR_BGR2GRAY
        )

        height, width = gray.shape

        tracking_mask = make_tracking_mask(
            height,
            width
        )

        frame_diff = np.nan
        camera_motion_px = np.nan

        detected_points = 0
        tracked_points = 0
        track_ratio = 0.0

        if previous_gray is not None:
            frame_diff = float(
                cv2.absdiff(
                    gray,
                    previous_gray
                ).mean()
            )

            points = cv2.goodFeaturesToTrack(
                previous_gray,
                maxCorners=350,
                qualityLevel=0.01,
                minDistance=7,
                mask=previous_mask
            )

            if points is not None:
                detected_points = len(points)

                next_points, status, _ = (
                    cv2.calcOpticalFlowPyrLK(
                        previous_gray,
                        gray,
                        points,
                        None,
                        winSize=(21, 21),
                        maxLevel=3,
                        criteria=(
                            cv2.TERM_CRITERIA_EPS
                            | cv2.TERM_CRITERIA_COUNT,
                            30,
                            0.01
                        )
                    )
                )

                if (
                    next_points is not None
                    and status is not None
                ):
                    old_points = points.reshape(
                        -1,
                        2
                    )

                    new_points = next_points.reshape(
                        -1,
                        2
                    )

                    valid = (
                        status.reshape(-1) == 1
                    )

                    valid = (
                        valid
                        & np.isfinite(old_points).all(axis=1)
                        & np.isfinite(new_points).all(axis=1)
                    )

                    old_points = old_points[valid]
                    new_points = new_points[valid]

                    tracked_points = len(
                        old_points
                    )

                    if detected_points > 0:
                        track_ratio = (
                            tracked_points
                            / detected_points
                        )

                    if tracked_points >= 5:
                        movement = np.linalg.norm(
                            new_points - old_points,
                            axis=1
                        )

                        if len(movement) > 10:
                            movement_limit = np.percentile(
                                movement,
                                95
                            )

                            movement = movement[
                                movement <= movement_limit
                            ]

                        if len(movement) > 0:
                            camera_motion_px = float(
                                np.median(movement)
                            )

        rows.append({
            "time_sec": (
                frame_index / native_fps
            ),
            "frame_diff": frame_diff,
            "camera_motion_px": camera_motion_px,
            "detected_points": detected_points,
            "tracked_points": tracked_points,
            "track_ratio": track_ratio,
            "tracking_reliable": (
                tracked_points
                >= FREEZE_REFINE_MIN_TRACKED_POINTS
            )
        })

        previous_gray = gray
        previous_mask = tracking_mask

        frame_index += 1

    cap.release()

    return pd.DataFrame(rows)


In [ ]:
refined_freezes = []

for freeze_index, freeze in freeze_intervals_df.iterrows():
    coarse_start_sec = float(
        freeze["start_sec"]
    )

    coarse_end_sec = float(
        freeze["end_sec"]
    )

    matching_chapters = chapters_df[
        (chapters_df["start_sec"] <= coarse_start_sec)
        & (chapters_df["end_sec"] >= coarse_end_sec)
    ]

    if matching_chapters.empty:
        chapter_start_sec = 0.0
        chapter_end_sec = float(
            VIDEO_INFO["duration_sec"]
        )

    else:
        matching_chapter = matching_chapters.iloc[0]

        chapter_start_sec = float(
            matching_chapter["start_sec"]
        )

        chapter_end_sec = float(
            matching_chapter["end_sec"]
        )

    search_start_sec = max(
        chapter_start_sec,
        coarse_start_sec
        - FREEZE_REFINE_LOOKBACK_SECONDS
    )

    search_end_sec = min(
        chapter_end_sec,
        coarse_end_sec
        + FREEZE_REFINE_LOOKAHEAD_SECONDS
    )

    scan_df = scan_freeze_area(
        VIDEO_PATH,
        search_start_sec,
        search_end_sec
    )

    refined_start_sec = coarse_start_sec
    refined_end_sec = coarse_end_sec
    method = "coarse_freeze_fallback"

    if len(scan_df) >= 2:
        local_sample_interval = float(
            scan_df["time_sec"]
            .diff()
            .dropna()
            .median()
        )

        scan_df["is_freeze_like"] = (
            scan_df["tracking_reliable"]
            & (
                scan_df["camera_motion_px"]
                <= FREEZE_REFINE_MOTION_MAX_PX
            )
            & (
                scan_df["frame_diff"]
                <= FREEZE_REFINE_FRAME_DIFF_MAX
            )
        )

        local_intervals = find_boolean_intervals(
            scan_df["time_sec"].to_numpy(),
            scan_df["is_freeze_like"].to_numpy(),
            local_sample_interval
        )

        local_intervals = merge_close_intervals(
            local_intervals,
            FREEZE_REFINE_MAX_GAP_SECONDS
        )

        overlapping_intervals = []

        for interval in local_intervals:
            interval_start_sec = float(
                interval["start_sec"]
            )

            interval_end_sec = float(
                interval["end_sec"]
            )

            interval_duration_sec = (
                interval_end_sec
                - interval_start_sec
            )

            overlap_sec = max(
                0.0,
                min(
                    interval_end_sec,
                    coarse_end_sec
                )
                - max(
                    interval_start_sec,
                    coarse_start_sec
                )
            )

            if (
                interval_duration_sec
                >= FREEZE_REFINE_MIN_SECONDS
                and overlap_sec > 0
            ):
                overlapping_intervals.append({
                    "start_sec": interval_start_sec,
                    "end_sec": interval_end_sec,
                    "overlap_sec": overlap_sec
                })

        if overlapping_intervals:
            selected_interval = max(
                overlapping_intervals,
                key=lambda interval: (
                    interval["overlap_sec"],
                    interval["end_sec"]
                    - interval["start_sec"]
                )
            )

            refined_start_sec = float(
                selected_interval["start_sec"]
            )

            refined_end_sec = float(
                selected_interval["end_sec"]
            )

            method = "high_fps_refinement"

    refined_freezes.append({
        "freeze_index": int(freeze_index),
        "coarse_start_sec": coarse_start_sec,
        "coarse_end_sec": coarse_end_sec,
        "refined_start_sec": refined_start_sec,
        "refined_end_sec": refined_end_sec,
        "start_adjustment_sec": (
            refined_start_sec
            - coarse_start_sec
        ),
        "end_adjustment_sec": (
            refined_end_sec
            - coarse_end_sec
        ),
        "duration_sec": (
            refined_end_sec
            - refined_start_sec
        ),
        "method": method
    })

refined_freezes_df = pd.DataFrame(
    refined_freezes,
    columns=[
        "freeze_index",
        "coarse_start_sec",
        "coarse_end_sec",
        "refined_start_sec",
        "refined_end_sec",
        "start_adjustment_sec",
        "end_adjustment_sec",
        "duration_sec",
        "method"
    ]
)

display(refined_freezes_df)


In [ ]:
FREEZE_ENVELOPE_SCAN_BEFORE_SEC = 2.0
FREEZE_ENVELOPE_SCAN_AFTER_SEC = 1.5

FREEZE_ENTRY_SEARCH_BEFORE_SEC = 1.2
FREEZE_ENTRY_STRONG_CHANGE_LOOKBACK_SEC = 0.55

FREEZE_ENTRY_DIFF_ABS_MIN = 12.0
FREEZE_ENTRY_DIFF_MAD_MULTIPLIER = 4.0

FREEZE_ENTRY_MOTION_MIN_THRESHOLD = 0.5
FREEZE_ENTRY_MOTION_MAX_THRESHOLD = 2.5
FREEZE_ENTRY_MOTION_FRACTION = 0.25

FREEZE_ENTRY_MIN_LOW_MOTION_SEC = 0.25
FREEZE_ENTRY_MAX_GAP_SEC = 0.18
FREEZE_ENTRY_MIN_TRACKED_POINTS = 10

FREEZE_RECOVERY_MOTION_MIN = 0.8
FREEZE_RECOVERY_MOTION_MAX = 3.0
FREEZE_RECOVERY_MOTION_FRACTION = 0.35

FREEZE_RECOVERY_DIFF_MIN = 3.0
FREEZE_RECOVERY_DIFF_MAX = 10.0
FREEZE_RECOVERY_DIFF_FRACTION = 0.25

FREEZE_RECOVERY_MIN_TRACKED_POINTS = 10
FREEZE_RECOVERY_MIN_SECONDS = 0.25

def safe_median(values, default_value):

    clean_values = (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if clean_values.empty:
        return float(default_value)

    return float(clean_values.median())

def median_absolute_deviation(
    values,
    default_value=1.0
):

    clean_values = (
        pd.Series(values)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    if clean_values.empty:
        return float(default_value)

    median_value = float(
        clean_values.median()
    )

    mad_value = float(
        (clean_values - median_value)
        .abs()
        .median()
    )

    return max(
        mad_value,
        float(default_value)
    )

def find_first_true_run(
    time_values,
    flag_values,
    minimum_duration_sec
):

    time_values = np.asarray(
        time_values,
        dtype=float
    )

    flag_values = np.asarray(
        flag_values,
        dtype=bool
    )

    if len(time_values) == 0:
        return None

    if len(time_values) > 1:
        local_interval = float(
            np.median(
                np.diff(time_values)
            )
        )
    else:
        local_interval = (
            1.0 / FREEZE_REFINE_FPS
        )

    run_start_index = None

    for index, flag in enumerate(flag_values):
        if flag and run_start_index is None:
            run_start_index = index

        run_ended = (
            run_start_index is not None
            and (
                not flag
                or index == len(flag_values) - 1
            )
        )

        if not run_ended:
            continue

        if flag and index == len(flag_values) - 1:
            run_end_index = index
        else:
            run_end_index = index - 1

        duration_sec = (
            time_values[run_end_index]
            - time_values[run_start_index]
            + local_interval
        )

        if duration_sec >= minimum_duration_sec:
            return {
                "start_sec": float(
                    time_values[run_start_index]
                ),
                "end_sec": float(
                    time_values[run_end_index]
                    + local_interval
                ),
                "duration_sec": float(
                    duration_sec
                )
            }

        run_start_index = None

    return None


In [ ]:
freeze_envelopes = []

for _, freeze_result in refined_freezes_df.iterrows():
    freeze_index = int(
        freeze_result["freeze_index"]
    )

    core_start_sec = float(
        freeze_result["refined_start_sec"]
    )

    core_end_sec = float(
        freeze_result["refined_end_sec"]
    )

    matching_chapters = chapters_df[
        (chapters_df["start_sec"] <= core_start_sec)
        & (chapters_df["end_sec"] >= core_end_sec)
    ]

    if matching_chapters.empty:
        chapter_start_sec = 0.0
        chapter_end_sec = float(
            VIDEO_INFO["duration_sec"]
        )

    else:
        chapter = matching_chapters.iloc[0]

        chapter_start_sec = float(
            chapter["start_sec"]
        )

        chapter_end_sec = float(
            chapter["end_sec"]
        )

    scan_start_sec = max(
        chapter_start_sec,
        core_start_sec
        - FREEZE_ENVELOPE_SCAN_BEFORE_SEC
    )

    scan_end_sec = min(
        chapter_end_sec,
        core_end_sec
        + FREEZE_ENVELOPE_SCAN_AFTER_SEC
    )

    local_df = scan_freeze_area(
        VIDEO_PATH,
        scan_start_sec,
        scan_end_sec
    )

    if len(local_df) < 2:
        freeze_envelopes.append({
            "freeze_index": freeze_index,
            "core_start_sec": core_start_sec,
            "core_end_sec": core_end_sec,
            "envelope_start_sec": core_start_sec,
            "envelope_end_sec": core_end_sec,
            "start_extension_sec": 0.0,
            "end_extension_sec": 0.0,
            "reference_motion": np.nan,
            "reference_diff": np.nan,
            "entry_motion_threshold": np.nan,
            "entry_diff_threshold": np.nan,
            "recovery_motion_threshold": np.nan,
            "recovery_diff_threshold": np.nan,
            "start_method": "insufficient_samples",
            "end_method": "insufficient_samples"
        })

        continue

    local_df = (
        local_df
        .sort_values("time_sec")
        .reset_index(drop=True)
    )

    local_sample_interval = float(
        local_df["time_sec"]
        .diff()
        .dropna()
        .median()
    )

    smooth_window = max(
        3,
        int(round(
            0.25 / local_sample_interval
        ))
    )

    if smooth_window % 2 == 0:
        smooth_window += 1

    local_df["camera_motion_smooth"] = (
        local_df["camera_motion_px"]
        .rolling(
            window=smooth_window,
            center=True,
            min_periods=1
        )
        .median()
    )

    local_df["frame_diff_smooth"] = (
        local_df["frame_diff"]
        .rolling(
            window=smooth_window,
            center=True,
            min_periods=1
        )
        .median()
    )

    local_df["valid_pair"] = (
        local_df["camera_motion_smooth"].notna()
        & local_df["frame_diff_smooth"].notna()
    )

    reference_end_sec = (
        core_start_sec
        - FREEZE_ENTRY_SEARCH_BEFORE_SEC
    )

    reference_df = local_df[
        local_df["time_sec"]
        < reference_end_sec
    ].copy()

    if len(reference_df) < 3:
        reference_df = local_df[
            local_df["time_sec"]
            < core_start_sec - 0.4
        ].copy()

    reference_motion = safe_median(
        reference_df["camera_motion_smooth"],
        4.0
    )

    reference_diff = safe_median(
        reference_df["frame_diff_smooth"],
        15.0
    )

    reference_diff_mad = (
        median_absolute_deviation(
            reference_df["frame_diff_smooth"],
            default_value=1.0
        )
    )

    entry_motion_threshold = float(
        np.clip(
            reference_motion
            * FREEZE_ENTRY_MOTION_FRACTION,
            FREEZE_ENTRY_MOTION_MIN_THRESHOLD,
            FREEZE_ENTRY_MOTION_MAX_THRESHOLD
        )
    )

    entry_diff_threshold = max(
        FREEZE_ENTRY_DIFF_ABS_MIN,
        reference_diff
        + FREEZE_ENTRY_DIFF_MAD_MULTIPLIER
        * reference_diff_mad
    )

    entry_window_start_sec = max(
        scan_start_sec,
        core_start_sec
        - FREEZE_ENTRY_SEARCH_BEFORE_SEC
    )

    entry_df = local_df[
        (
            local_df["time_sec"]
            >= entry_window_start_sec
        )
        & (
            local_df["time_sec"]
            <= core_start_sec
        )
    ].copy()

    entry_df["low_motion"] = (
        entry_df["valid_pair"]
        & (
            entry_df["tracked_points"]
            >= FREEZE_ENTRY_MIN_TRACKED_POINTS
        )
        & (
            entry_df["camera_motion_smooth"]
            <= entry_motion_threshold
        )
    )

    entry_df["strong_change"] = (
        entry_df["valid_pair"]
        & (
            entry_df["frame_diff_smooth"]
            >= entry_diff_threshold
        )
    )

    low_motion_intervals = find_boolean_intervals(
        entry_df["time_sec"].to_numpy(),
        entry_df["low_motion"].to_numpy(),
        local_sample_interval
    )

    low_motion_intervals = merge_close_intervals(
        low_motion_intervals,
        FREEZE_ENTRY_MAX_GAP_SEC
    )

    low_motion_intervals = [
        interval
        for interval in low_motion_intervals
        if (
            float(interval["end_sec"])
            - float(interval["start_sec"])
        ) >= FREEZE_ENTRY_MIN_LOW_MOTION_SEC
    ]

    low_motion_start_sec = None

    for interval in low_motion_intervals:
        reaches_core = (
            float(interval["end_sec"])
            >= (
                core_start_sec
                - 2 * local_sample_interval
            )
        )

        if reaches_core:
            low_motion_start_sec = float(
                interval["start_sec"]
            )

            break

    if low_motion_start_sec is None:
        low_motion_start_sec = core_start_sec

    strong_change_search_start_sec = max(
        entry_window_start_sec,
        low_motion_start_sec
        - FREEZE_ENTRY_STRONG_CHANGE_LOOKBACK_SEC
    )

    strong_change_candidates = entry_df[
        (
            entry_df["time_sec"]
            >= strong_change_search_start_sec
        )
        & (
            entry_df["time_sec"]
            <= low_motion_start_sec
        )
        & entry_df["strong_change"]
    ].copy()

    if not strong_change_candidates.empty:
        envelope_start_sec = float(
            strong_change_candidates
            .sort_values("time_sec")
            .iloc[0]["time_sec"]
        )

        start_method = "entry_scene_change"

    elif low_motion_start_sec < core_start_sec:
        envelope_start_sec = (
            low_motion_start_sec
        )

        start_method = "sustained_motion_drop"

    else:
        envelope_start_sec = (
            core_start_sec
        )

        start_method = "core_start_fallback"

    recovery_motion_threshold = float(
        np.clip(
            reference_motion
            * FREEZE_RECOVERY_MOTION_FRACTION,
            FREEZE_RECOVERY_MOTION_MIN,
            FREEZE_RECOVERY_MOTION_MAX
        )
    )

    recovery_diff_threshold = float(
        np.clip(
            reference_diff
            * FREEZE_RECOVERY_DIFF_FRACTION,
            FREEZE_RECOVERY_DIFF_MIN,
            FREEZE_RECOVERY_DIFF_MAX
        )
    )

    recovery_df = local_df[
        local_df["time_sec"]
        >= core_end_sec
    ].copy()

    recovery_df["stable_motion_return"] = (
        recovery_df["valid_pair"]
        & (
            recovery_df["tracked_points"]
            >= FREEZE_RECOVERY_MIN_TRACKED_POINTS
        )
        & (
            recovery_df["camera_motion_smooth"]
            >= recovery_motion_threshold
        )
        & (
            recovery_df["frame_diff_smooth"]
            >= recovery_diff_threshold
        )
    )

    recovery_run = find_first_true_run(
        recovery_df["time_sec"].to_numpy(),
        recovery_df[
            "stable_motion_return"
        ].to_numpy(),
        FREEZE_RECOVERY_MIN_SECONDS
    )

    if recovery_run is not None:
        envelope_end_sec = float(
            recovery_run["start_sec"]
        )

        end_method = "stable_flight_return"

    else:
        envelope_end_sec = core_end_sec
        end_method = "core_end_fallback"

    envelope_start_sec = max(
        scan_start_sec,
        min(
            envelope_start_sec,
            core_start_sec
        )
    )

    envelope_end_sec = min(
        scan_end_sec,
        max(
            envelope_end_sec,
            core_end_sec
        )
    )

    freeze_envelopes.append({
        "freeze_index": freeze_index,
        "core_start_sec": core_start_sec,
        "core_end_sec": core_end_sec,
        "envelope_start_sec": envelope_start_sec,
        "envelope_end_sec": envelope_end_sec,
        "start_extension_sec": (
            core_start_sec
            - envelope_start_sec
        ),
        "end_extension_sec": (
            envelope_end_sec
            - core_end_sec
        ),
        "reference_motion": reference_motion,
        "reference_diff": reference_diff,
        "entry_motion_threshold": (
            entry_motion_threshold
        ),
        "entry_diff_threshold": (
            entry_diff_threshold
        ),
        "recovery_motion_threshold": (
            recovery_motion_threshold
        ),
        "recovery_diff_threshold": (
            recovery_diff_threshold
        ),
        "start_method": start_method,
        "end_method": end_method
    })

freeze_envelope_columns = [
    "freeze_index",
    "core_start_sec",
    "core_end_sec",
    "envelope_start_sec",
    "envelope_end_sec",
    "start_extension_sec",
    "end_extension_sec",
    "reference_motion",
    "reference_diff",
    "entry_motion_threshold",
    "entry_diff_threshold",
    "recovery_motion_threshold",
    "recovery_diff_threshold",
    "start_method",
    "end_method"
]

freeze_envelopes_df = pd.DataFrame(
    freeze_envelopes,
    columns=freeze_envelope_columns
)

display(freeze_envelopes_df)


In [ ]:
MONTAGE_SEARCH_SECONDS_BEFORE_END = 10.0
MONTAGE_EVENT_WINDOW_SECONDS = 5.0
MONTAGE_ENTRY_LOOKBACK_SECONDS = 0.80

MONTAGE_HOLD_MOTION_MAX_PX = 0.50
MONTAGE_HOLD_FRAME_DIFF_MAX = 6.0
MONTAGE_MIN_HOLD_SECONDS = 0.55
MONTAGE_MAX_HOLD_GAP_SECONDS = 0.20

MONTAGE_HARD_CHANGE_MIN_DIFF = 45.0
MONTAGE_HARD_CHANGE_MAX_TRACK_RATIO = 0.80

MONTAGE_BRIGHT_FLASH_MIN = 210.0
MONTAGE_DARK_FLASH_MAX = 25.0

MONTAGE_MIN_HOLD_RUNS = 2
MONTAGE_MIN_HARD_CHANGES = 2
MONTAGE_MIN_FLIGHT_BLOCK_SECONDS = 4.0

edit_montage_boundaries = []

for _, flight_block in classified_blocks_df.iterrows():

    if str(flight_block["coarse_state"]) != "FLIGHT":
        continue

    block_id = int(
        flight_block["block_id"]
    )

    chapter_index = int(
        flight_block["chapter_index"]
    )

    block_start_sec = float(
        flight_block["start_sec"]
    )

    block_end_sec = float(
        flight_block["end_sec"]
    )

    block_duration_sec = float(
        flight_block["duration_sec"]
    )

    if (
        block_duration_sec
        < MONTAGE_MIN_FLIGHT_BLOCK_SECONDS
    ):
        continue

    search_start_sec = max(
        block_start_sec,
        block_end_sec
        - MONTAGE_SEARCH_SECONDS_BEFORE_END
    )

    block_rows = analysis_df[
        (analysis_df["chapter_index"] == chapter_index)
        & (analysis_df["time_sec"] >= search_start_sec)
        & (analysis_df["time_sec"] < block_end_sec)
    ].copy()

    if len(block_rows) < 3:
        continue

    block_rows = (
        block_rows
        .sort_values("time_sec")
        .reset_index(drop=True)
    )

    block_rows["is_montage_hold"] = (
        block_rows["tracking_reliable"]
        & (
            block_rows["camera_motion_smooth"]
            <= MONTAGE_HOLD_MOTION_MAX_PX
        )
        & (
            block_rows["frame_diff_smooth"]
            <= MONTAGE_HOLD_FRAME_DIFF_MAX
        )
    )

    hold_intervals = find_boolean_intervals(
        block_rows["time_sec"].to_numpy(),
        block_rows["is_montage_hold"].to_numpy(),
        sample_interval_sec
    )

    hold_intervals = merge_close_intervals(
        hold_intervals,
        MONTAGE_MAX_HOLD_GAP_SECONDS
    )

    hold_intervals = [
        interval
        for interval in hold_intervals
        if (
            float(interval["end_sec"])
            - float(interval["start_sec"])
        ) >= MONTAGE_MIN_HOLD_SECONDS
    ]

    if (
        len(hold_intervals)
        < MONTAGE_MIN_HOLD_RUNS
    ):
        continue

    strong_change = (
        (
            block_rows["frame_diff"]
            >= MONTAGE_HARD_CHANGE_MIN_DIFF
        )
        & (
            block_rows["track_ratio"]
            <= MONTAGE_HARD_CHANGE_MAX_TRACK_RATIO
        )
    )

    flash_change = (
        (
            (
                block_rows["brightness"]
                >= MONTAGE_BRIGHT_FLASH_MIN
            )
            |
            (
                block_rows["brightness"]
                <= MONTAGE_DARK_FLASH_MAX
            )
        )
        & (
            block_rows["frame_diff"]
            >= MONTAGE_HARD_CHANGE_MIN_DIFF
        )
    )

    block_rows["is_montage_hard_change"] = (
        strong_change | flash_change
    )

    montage_found = False

    for first_hold_index in range(
        len(hold_intervals)
    ):
        first_hold = hold_intervals[
            first_hold_index
        ]

        grouped_holds = [
            first_hold
        ]

        for next_hold in hold_intervals[
            first_hold_index + 1:
        ]:
            time_from_first_hold = (
                float(next_hold["start_sec"])
                - float(first_hold["start_sec"])
            )

            if (
                time_from_first_hold
                <= MONTAGE_EVENT_WINDOW_SECONDS
            ):
                grouped_holds.append(
                    next_hold
                )
            else:
                break

        if (
            len(grouped_holds)
            < MONTAGE_MIN_HOLD_RUNS
        ):
            continue

        first_hold_start_sec = float(
            grouped_holds[0]["start_sec"]
        )

        last_hold_end_sec = float(
            grouped_holds[-1]["end_sec"]
        )

        event_start_sec = max(
            search_start_sec,
            first_hold_start_sec
            - MONTAGE_ENTRY_LOOKBACK_SECONDS
        )

        event_rows = block_rows[
            (block_rows["time_sec"] >= event_start_sec)
            & (block_rows["time_sec"] <= last_hold_end_sec)
        ].copy()

        hard_change_rows = event_rows[
            event_rows[
                "is_montage_hard_change"
            ]
        ].copy()

        if (
            len(hard_change_rows)
            < MONTAGE_MIN_HARD_CHANGES
        ):
            continue

        entry_change_rows = hard_change_rows[
            hard_change_rows["time_sec"]
            <= first_hold_start_sec
        ]

        if not entry_change_rows.empty:
            montage_start_sec = float(
                entry_change_rows[
                    "time_sec"
                ].min()
            )

            start_method = (
                "hard_change_before_holds"
            )

        else:
            montage_start_sec = (
                first_hold_start_sec
            )

            start_method = (
                "first_hold_start"
            )

        seconds_before_block_end = (
            block_end_sec
            - montage_start_sec
        )

        edit_montage_boundaries.append({
            "flight_block_id": block_id,
            "chapter_index": chapter_index,
            "block_start_sec": block_start_sec,
            "block_end_sec": block_end_sec,
            "montage_start_sec": montage_start_sec,
            "seconds_before_block_end": (
                seconds_before_block_end
            ),
            "hold_runs_count": len(
                grouped_holds
            ),
            "hard_change_count": len(
                hard_change_rows
            ),
            "first_hold_start_sec": (
                first_hold_start_sec
            ),
            "last_hold_end_sec": (
                last_hold_end_sec
            ),
            "start_method": start_method
        })

        montage_found = True
        break

edit_montage_boundaries_df = pd.DataFrame(
    edit_montage_boundaries,
    columns=[
        "flight_block_id",
        "chapter_index",
        "block_start_sec",
        "block_end_sec",
        "montage_start_sec",
        "seconds_before_block_end",
        "hold_runs_count",
        "hard_change_count",
        "first_hold_start_sec",
        "last_hold_end_sec",
        "start_method"
    ]
)

display(edit_montage_boundaries_df)


In [ ]:
MIN_FINAL_FLIGHT_SECONDS = 1.0
TIME_EPSILON = 0.001

sorted_classified_blocks = (
    classified_blocks_df
    .sort_values(
        ["chapter_index", "start_sec"]
    )
    .reset_index(drop=True)
)

flight_groups = []
current_group = []

for _, block in sorted_classified_blocks.iterrows():

    is_flight = (
        str(block["coarse_state"])
        == "FLIGHT"
    )

    if is_flight:
        if current_group:
            same_chapter = (
                int(current_group[-1]["chapter_index"])
                == int(block["chapter_index"])
            )

            if not same_chapter:
                flight_groups.append(
                    current_group
                )

                current_group = []

        current_group.append(
            block.to_dict()
        )

    else:
        if current_group:
            flight_groups.append(
                current_group
            )

            current_group = []

if current_group:
    flight_groups.append(
        current_group
    )

final_flight_segments = []

for group in flight_groups:

    first_block = group[0]
    last_block = group[-1]

    chapter_index = int(
        first_block["chapter_index"]
    )

    source_block_ids = [
        int(block["block_id"])
        for block in group
    ]

    coarse_start_sec = float(
        first_block["start_sec"]
    )

    coarse_end_sec = float(
        last_block["end_sec"]
    )

    final_start_sec = coarse_start_sec
    final_end_sec = coarse_end_sec

    start_source = "coarse_block_start"
    end_source = "coarse_block_end"

    overlapping_start_envelopes = []

    for _, freeze in freeze_envelopes_df.iterrows():
        envelope_start_sec = float(
            freeze["envelope_start_sec"]
        )

        envelope_end_sec = float(
            freeze["envelope_end_sec"]
        )

        overlaps_group_start = (
            envelope_start_sec
            <= coarse_start_sec
            < envelope_end_sec
        )

        if overlaps_group_start:
            overlapping_start_envelopes.append(
                envelope_end_sec
            )

    if overlapping_start_envelopes:
        refined_start_sec = max(
            overlapping_start_envelopes
        )

        if (
            refined_start_sec
            > final_start_sec + TIME_EPSILON
        ):
            final_start_sec = refined_start_sec
            start_source = "freeze_edit_envelope_end"

    end_candidates = [
        {
            "time_sec": coarse_end_sec,
            "source": "coarse_block_end"
        }
    ]

    for _, freeze in freeze_envelopes_df.iterrows():
        envelope_start_sec = float(
            freeze["envelope_start_sec"]
        )

        begins_inside_flight = (
            final_start_sec + TIME_EPSILON
            < envelope_start_sec
            < coarse_end_sec - TIME_EPSILON
        )

        if begins_inside_flight:
            end_candidates.append({
                "time_sec": envelope_start_sec,
                "source": "freeze_edit_envelope_start"
            })

    last_block_id = int(
        last_block["block_id"]
    )

    matching_refined_boundaries = (
        refined_boundaries_df[
            refined_boundaries_df[
                "previous_flight_block_id"
            ] == last_block_id
        ]
    )

    for _, boundary in matching_refined_boundaries.iterrows():
        refined_cut_sec = float(
            boundary["refined_cut_sec"]
        )

        if (
            final_start_sec + TIME_EPSILON
            < refined_cut_sec
            < coarse_end_sec - TIME_EPSILON
        ):
            end_candidates.append({
                "time_sec": refined_cut_sec,
                "source": "refined_transition_start"
            })

    matching_montages = (
        edit_montage_boundaries_df[
            edit_montage_boundaries_df[
                "flight_block_id"
            ] == last_block_id
        ]
    )

    for _, montage in matching_montages.iterrows():
        montage_start_sec = float(
            montage["montage_start_sec"]
        )

        if (
            final_start_sec + TIME_EPSILON
            < montage_start_sec
            < coarse_end_sec - TIME_EPSILON
        ):
            end_candidates.append({
                "time_sec": montage_start_sec,
                "source": "edit_montage_start"
            })

    selected_end = min(
        end_candidates,
        key=lambda candidate: candidate["time_sec"]
    )

    final_end_sec = float(
        selected_end["time_sec"]
    )

    end_source = str(
        selected_end["source"]
    )

    duration_sec = (
        final_end_sec
        - final_start_sec
    )

    if duration_sec < MIN_FINAL_FLIGHT_SECONDS:
        continue

    final_flight_segments.append({
        "segment_index": len(
            final_flight_segments
        ),
        "chapter_index": chapter_index,
        "source_block_ids": ",".join(
            str(block_id)
            for block_id in source_block_ids
        ),
        "start_sec": final_start_sec,
        "end_sec": final_end_sec,
        "duration_sec": duration_sec,
        "start_source": start_source,
        "end_source": end_source
    })

final_flight_segments_df = pd.DataFrame(
    final_flight_segments,
    columns=[
        "segment_index",
        "chapter_index",
        "source_block_ids",
        "start_sec",
        "end_sec",
        "duration_sec",
        "start_source",
        "end_source"
    ]
)


In [ ]:
FINAL_SEGMENTS_CSV_PATH = (
    OUTPUT_DIR
    / f"{VIDEO_PATH.stem}_flight_segments.csv"
)

final_flight_segments_df.to_csv(
    FINAL_SEGMENTS_CSV_PATH,
    index=False
)

DRIVE_FINAL_SEGMENTS_CSV_PATH = (
    DRIVE_OUTPUT_DIR
    / FINAL_SEGMENTS_CSV_PATH.name
)

shutil.copy2(
    FINAL_SEGMENTS_CSV_PATH,
    DRIVE_FINAL_SEGMENTS_CSV_PATH
)

display(final_flight_segments_df)


In [ ]:
if final_flight_segments_df.empty:
    raise ValueError(
        "לא נמצאו קטעי טיסה ליצירת הסרטון."
    )

FINAL_VIDEO_PATH = (
    OUTPUT_DIR
    / f"{VIDEO_PATH.stem}_flight_only.mp4"
)

filter_parts = []
video_labels = []

for segment_number, segment in (
    final_flight_segments_df
    .sort_values("segment_index")
    .reset_index(drop=True)
    .iterrows()
):
    start_sec = float(
        segment["start_sec"]
    )

    end_sec = float(
        segment["end_sec"]
    )

    label = f"v{segment_number}"

    filter_parts.append(
        (
            f"[0:v]"
            f"trim=start={start_sec:.6f}:end={end_sec:.6f},"
            f"setpts=PTS-STARTPTS"
            f"[{label}]"
        )
    )

    video_labels.append(
        f"[{label}]"
    )

segment_count = len(
    video_labels
)

filter_parts.append(
    (
        "".join(video_labels)
        + f"concat=n={segment_count}:v=1:a=0"
        + "[outv]"
    )
)

filter_complex = ";".join(
    filter_parts
)

if FFMPEG_PATH is None:
    raise RuntimeError(
        "FFmpeg לא נמצא בסביבת Colab."
    )

command = [
    FFMPEG_PATH,
    "-y",
    "-i",
    str(VIDEO_PATH),
    "-filter_complex",
    filter_complex,
    "-map",
    "[outv]",
    "-an",
    "-c:v",
    "libx264",
    "-preset",
    "medium",
    "-crf",
    "18",
    "-pix_fmt",
    "yuv420p",
    "-movflags",
    "+faststart",
    str(FINAL_VIDEO_PATH)
]

result = subprocess.run(
    command,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)

    raise RuntimeError(
        "FFmpeg לא הצליח ליצור את הסרטון."
    )

if (
    not FINAL_VIDEO_PATH.exists()
    or FINAL_VIDEO_PATH.stat().st_size == 0
):
    raise RuntimeError(
        "קובץ הפלט לא נוצר כראוי."
    )

FINAL_VIDEO_INFO = get_video_info(
    FINAL_VIDEO_PATH
)

expected_duration_sec = float(
    final_flight_segments_df[
        "duration_sec"
    ].sum()
)

actual_duration_sec = float(
    FINAL_VIDEO_INFO["duration_sec"]
)

display(pd.DataFrame([{
    "video": str(FINAL_VIDEO_PATH),
    "expected_sec": round(expected_duration_sec, 2),
    "actual_sec": round(actual_duration_sec, 2),
    "difference_sec": round(actual_duration_sec - expected_duration_sec, 3),
    "size_mb": round(FINAL_VIDEO_PATH.stat().st_size / (1024 ** 2), 2),
}]))


In [ ]:
DRIVE_FINAL_VIDEO_PATH = (
    DRIVE_OUTPUT_DIR
    / FINAL_VIDEO_PATH.name
)

shutil.copy2(
    FINAL_VIDEO_PATH,
    DRIVE_FINAL_VIDEO_PATH
)


In [ ]:
SEGMENTS_DIR = (
    OUTPUT_DIR
    / f"{VIDEO_PATH.stem}_segments"
)

DRIVE_SEGMENTS_DIR = (
    DRIVE_OUTPUT_DIR
    / f"{VIDEO_PATH.stem}_segments"
)

for folder in [
    SEGMENTS_DIR,
    DRIVE_SEGMENTS_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

manifest_rows = []
clean_timeline_cursor_sec = 0.0

for _, segment in (
    final_flight_segments_df
    .sort_values("segment_index")
    .iterrows()
):
    segment_index = int(
        segment["segment_index"]
    )

    source_start_sec = float(
        segment["start_sec"]
    )

    source_end_sec = float(
        segment["end_sec"]
    )

    segment_filename = (
        f"segment_{segment_index:03d}.mp4"
    )

    runtime_segment_path = (
        SEGMENTS_DIR
        / segment_filename
    )

    drive_segment_path = (
        DRIVE_SEGMENTS_DIR
        / segment_filename
    )

    segment_filter = (
        f"[0:v]"
        f"trim=start={source_start_sec:.6f}:"
        f"end={source_end_sec:.6f},"
        f"setpts=PTS-STARTPTS"
        f"[outv]"
    )

    segment_command = [
        FFMPEG_PATH,
        "-y",
        "-i",
        str(VIDEO_PATH),
        "-filter_complex",
        segment_filter,
        "-map",
        "[outv]",
        "-an",
        "-c:v",
        "libx264",
        "-preset",
        "medium",
        "-crf",
        "18",
        "-pix_fmt",
        "yuv420p",
        "-movflags",
        "+faststart",
        str(runtime_segment_path)
    ]

    segment_result = subprocess.run(
        segment_command,
        capture_output=True,
        text=True
    )

    if segment_result.returncode != 0:
        print(segment_result.stderr)
        raise RuntimeError(
            f"FFmpeg נכשל במקטע {segment_index}."
        )

    segment_info = get_video_info(
        runtime_segment_path
    )

    shutil.copy2(
        runtime_segment_path,
        drive_segment_path
    )

    segment_duration_sec = float(
        segment_info["duration_sec"]
    )

    manifest_rows.append({
        "segment_index": segment_index,
        "source_video_name": VIDEO_PATH.name,
        "source_start_sec": source_start_sec,
        "source_end_sec": source_end_sec,
        "source_start_frame": int(round(
            source_start_sec
            * VIDEO_INFO["fps"]
        )),
        "source_end_frame_exclusive": int(round(
            source_end_sec
            * VIDEO_INFO["fps"]
        )),
        "clean_timeline_start_sec": (
            clean_timeline_cursor_sec
        ),
        "clean_timeline_end_sec": (
            clean_timeline_cursor_sec
            + segment_duration_sec
        ),
        "segment_duration_sec": segment_duration_sec,
        "segment_fps": float(
            segment_info["fps"]
        ),
        "segment_frame_count": int(
            segment_info["frame_count"]
        ),
        "drive_segment_path": str(
            drive_segment_path
        )
    })

    clean_timeline_cursor_sec += (
        segment_duration_sec
    )

manifest_df = pd.DataFrame(
    manifest_rows
)

MANIFEST_CSV_PATH = (
    OUTPUT_DIR
    / f"{VIDEO_PATH.stem}_manifest.csv"
)

DRIVE_MANIFEST_CSV_PATH = (
    DRIVE_OUTPUT_DIR
    / MANIFEST_CSV_PATH.name
)

manifest_df.to_csv(
    MANIFEST_CSV_PATH,
    index=False
)

shutil.copy2(
    MANIFEST_CSV_PATH,
    DRIVE_MANIFEST_CSV_PATH
)


In [ ]:
quality_summary = {
    "schema_version": 1,
    "created_at_utc": (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    "stage1_version": STAGE1_VERSION,
    "parent_baseline_version": PARENT_BASELINE_VERSION,
    "parent_candidate_version": PARENT_CANDIDATE_VERSION,
    "parent_baseline_logic_sha256": (
        PARENT_BASELINE_LOGIC_SHA256
    ),
    "candidate_change": {
        "parameter": "MAX_REPLAY_FREEZE_OVERLAP_RATIO",
        "parent_value": None,
        "candidate_value": float(
            MAX_REPLAY_FREEZE_OVERLAP_RATIO
        )
    },
    "inherited_candidate_settings": {
        "MIN_REPLAY_TIME_GAP_SEC": float(
            MIN_REPLAY_TIME_GAP_SEC
        )
    },
    "source_video": VIDEO_INFO,
    "requested_sample_fps": SAMPLE_FPS,
    "actual_sample_fps": actual_sample_fps,
    "feature_rows": int(
        len(features_df)
    ),
    "accepted_replay_count": int(
        len(replay_intervals_df)
    ),
    "freeze_overlap_rejection_count": int(
        len(rejected_freeze_overlap_replays_df)
    ),
    "segment_count": int(
        len(manifest_df)
    ),
    "expected_total_duration_sec": float(
        final_flight_segments_df[
            "duration_sec"
        ].sum()
    ),
    "actual_final_video_duration_sec": (
        actual_duration_sec
    ),
    "duration_difference_sec": float(
        actual_duration_sec
        - expected_duration_sec
    ),
    "final_video_drive_path": str(
        DRIVE_FINAL_VIDEO_PATH
    ),
    "manifest_drive_path": str(
        DRIVE_MANIFEST_CSV_PATH
    ),
    "environment": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "opencv": cv2.__version__,
        "matplotlib": matplotlib.__version__
    }
}

QUALITY_JSON_PATH = (
    OUTPUT_DIR
    / f"{VIDEO_PATH.stem}_quality.json"
)

DRIVE_QUALITY_JSON_PATH = (
    DRIVE_OUTPUT_DIR
    / QUALITY_JSON_PATH.name
)

with open(
    QUALITY_JSON_PATH,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        quality_summary,
        file,
        ensure_ascii=False,
        indent=2
    )

shutil.copy2(
    QUALITY_JSON_PATH,
    DRIVE_QUALITY_JSON_PATH
)

display(manifest_df)

display(pd.DataFrame([{
    "final_video": str(DRIVE_FINAL_VIDEO_PATH),
    "manifest": str(DRIVE_MANIFEST_CSV_PATH),
    "quality": str(DRIVE_QUALITY_JSON_PATH),
    "segments_dir": str(DRIVE_SEGMENTS_DIR),
}]))

In [ ]:
PIPELINE_VERSION = "fpv_reconstruction_resilient_decode"
STAGE2_VERSION = "stage2_vggt_omega_resilient_decode"
STAGE1_SOURCE_VERSION = STAGE1_VERSION

STAGE1_MANIFEST_DRIVE_PATH = ""  # ריק = איתור אוטומטי
STAGE1_SOURCE_RUN_NAME = ""

# מאגר צפוף לבחירה; המודל עצמו אינו מקבל את כל המועמדים.
KEYFRAME_CANDIDATE_FPS = 4.0
KEYFRAME_MAX_CANDIDATES = 240
KEYFRAME_TARGET_FPS = 1.0
KEYFRAME_MAX_MODEL_FRAMES = 48
KEYFRAME_MIN_MODEL_FRAMES = 4
KEYFRAME_MIN_PER_STAGE1_SEGMENT = 1
KEYFRAME_ANALYSIS_WIDTH = 480
KEYFRAME_ORB_FEATURES = 1400
SAVE_KEYFRAME_CANDIDATES_TO_DRIVE = True
STAGE2_REUSE_CANDIDATE_CACHE = True

# VGGT-Ω הרשמי, ללא Text Alignment — הענף האיכותי לשחזור Camera + Depth.
OMEGA_REPO_URL = "https://github.com/facebookresearch/vggt-omega.git"
OMEGA_REPO_COMMIT = "39a0cb8af88554f15ddcb5354cd52bde588fa014"
OMEGA_MODEL_ID = "facebook/VGGT-Omega"
OMEGA_CHECKPOINT_FILENAME = "vggt_omega_1b_512.pt"
OMEGA_CHECKPOINT_CACHE_MODE = "runtime"  # runtime | drive
OMEGA_CHECKPOINT_DRIVE_PATH = ""  # אופציונלי: נתיב מפורש גובר על מצב ה-cache
OMEGA_IMAGE_RESOLUTION = 512
OMEGA_PREPROCESS_MODE = "balanced"  # זהה ל-demo הרשמי
OMEGA_PRECISION = "float32"  # איכות רשמית; אין FP16 שקט
OMEGA_MIN_FREE_GPU_MEMORY_GB = 10.0


def stage2_resize_for_analysis(frame, target_width):
    """Stage-2-local copy of the frozen Stage 1 resize contract."""
    height, width = frame.shape[:2]
    if width <= target_width:
        return frame.copy()
    scale = target_width / width
    target_height = int(round(height * scale))
    return cv2.resize(
        frame,
        (target_width, target_height),
        interpolation=cv2.INTER_AREA,
    )


def stage2_make_tracking_mask(height, width):
    """Stage-2-local copy of the frozen Stage 1 tracking mask."""
    mask = np.full((height, width), 255, dtype=np.uint8)
    mask[int(height * 0.62):height, 0:int(width * 0.22)] = 0
    mask[
        int(height * 0.42):int(height * 0.58),
        int(width * 0.42):int(width * 0.58),
    ] = 0
    mask[0:int(height * 0.25), int(width * 0.70):width] = 0
    return mask


if not 0.5 <= KEYFRAME_CANDIDATE_FPS <= 12.0:
    raise ValueError("KEYFRAME_CANDIDATE_FPS חייב להיות בין 0.5 ל-12.")
if not 0.5 <= KEYFRAME_TARGET_FPS <= 2.0:
    raise ValueError("KEYFRAME_TARGET_FPS חייב להיות בין 0.5 ל-2.0.")
if not 4 <= KEYFRAME_MAX_MODEL_FRAMES <= 100:
    raise ValueError("KEYFRAME_MAX_MODEL_FRAMES חייב להיות בין 4 ל-100.")
if KEYFRAME_MAX_CANDIDATES < KEYFRAME_MAX_MODEL_FRAMES:
    raise ValueError("מאגר המועמדים חייב להיות גדול ממספר פריימי המודל.")
if OMEGA_IMAGE_RESOLUTION != 512:
    raise ValueError("ה-checkpoint שנבחר מיועד לרזולוציה 512.")
if OMEGA_PREPROCESS_MODE not in {"balanced", "max_size"}:
    raise ValueError("OMEGA_PREPROCESS_MODE חייב להיות balanced או max_size.")
if OMEGA_PRECISION != "float32":
    raise ValueError("v1.2.2 מקבעת FP32 כדי לשחזר את איכות המימוש הרשמי.")
if OMEGA_CHECKPOINT_CACHE_MODE not in {"runtime", "drive"}:
    raise ValueError("OMEGA_CHECKPOINT_CACHE_MODE חייב להיות runtime או drive.")
if OMEGA_MIN_FREE_GPU_MEMORY_GB < 8.0:
    raise ValueError("סף זיכרון ה-GPU נמוך מדי ל-Baseline האיכותי.")


In [ ]:
required_bootstrap_names = [
    "MY_DRIVE_DIR",
    "PROJECT_FOLDER_NAME",
    "INPUT_MODE"
]

missing_bootstrap_names = [
    name
    for name in required_bootstrap_names
    if name not in globals()
]

if missing_bootstrap_names:
    raise RuntimeError(
        "לפני Stage 2 יש להריץ את התאים 1.1, 1.2 ו-1.3. "
        f"חסרים: {missing_bootstrap_names}"
    )

def make_stage2_safe_stem(text):

    safe_text = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(text)
    ).strip("._")

    return safe_text or "fpv_run"

def infer_stage1_run_name():

    if STAGE1_SOURCE_RUN_NAME.strip():
        return make_stage2_safe_stem(
            STAGE1_SOURCE_RUN_NAME
        )

    input_mode = str(
        globals().get("INPUT_MODE", "")
    )

    if input_mode == "url":
        source_text = str(
            globals().get("VIDEO_URL", "")
        ).strip()

        source_name = Path(
            urlparse(source_text).path
        ).name

    elif input_mode == "drive":
        source_text = str(
            globals().get("DRIVE_VIDEO_PATH", "")
        ).strip()

        source_name = Path(
            source_text
        ).name

    else:
        raise ValueError(
            "במצב upload אי אפשר להסיק את שם הריצה "
            "לפני העלאת הסרטון. יש למלא "
            "STAGE1_SOURCE_RUN_NAME או "
            "STAGE1_MANIFEST_DRIVE_PATH."
        )

    if not source_name:
        raise ValueError(
            "לא ניתן להסיק את שם הסרטון מה-CONFIG."
        )

    source_stem = make_stage2_safe_stem(
        Path(source_name).stem
    )

    run_parts = [
        source_stem
    ]

    run_suffix = str(
        globals().get("RUN_SUFFIX", "")
    ).strip()

    if run_suffix:
        run_parts.append(
            make_stage2_safe_stem(run_suffix)
        )

    return "_".join(run_parts)

manifest_path = None
manifest_source = None

if STAGE1_MANIFEST_DRIVE_PATH.strip():
    manifest_path = Path(
        STAGE1_MANIFEST_DRIVE_PATH
    ).expanduser()

    if not manifest_path.is_absolute():
        manifest_path = (
            MY_DRIVE_DIR
            / manifest_path
        )

    manifest_source = "explicit_path"

elif (
    "DRIVE_MANIFEST_CSV_PATH" in globals()
    and Path(
        globals()["DRIVE_MANIFEST_CSV_PATH"]
    ).is_file()
):
    manifest_path = Path(
        globals()["DRIVE_MANIFEST_CSV_PATH"]
    )

    manifest_source = "current_runtime"

else:
    inferred_run_name = infer_stage1_run_name()

    inferred_output_dir = (
        MY_DRIVE_DIR
        / PROJECT_FOLDER_NAME
        / "runs"
        / STAGE1_SOURCE_VERSION
        / inferred_run_name
        / "output"
    )

    manifest_candidates = sorted(
        inferred_output_dir.glob(
            "*_manifest.csv"
        )
    )

    if len(manifest_candidates) != 1:
        raise FileNotFoundError(
            "ציפיתי למצוא manifest אחד בדיוק בתוך:\n"
            f"{inferred_output_dir}\n"
            f"נמצאו: {len(manifest_candidates)}"
        )

    manifest_path = manifest_candidates[0]
    manifest_source = "inferred_from_config"

manifest_path = Path(
    manifest_path
).resolve()

if not manifest_path.is_file():
    raise FileNotFoundError(
        f"קובץ ה-manifest לא נמצא: {manifest_path}"
    )

stage1_manifest_df = pd.read_csv(
    manifest_path
)

required_manifest_columns = [
    "segment_index",
    "source_video_name",
    "source_start_sec",
    "source_end_sec",
    "source_start_frame",
    "source_end_frame_exclusive",
    "clean_timeline_start_sec",
    "clean_timeline_end_sec",
    "segment_duration_sec",
    "segment_fps",
    "segment_frame_count",
    "drive_segment_path"
]

missing_manifest_columns = [
    column
    for column in required_manifest_columns
    if column not in stage1_manifest_df.columns
]

if missing_manifest_columns:
    raise ValueError(
        "חסרות עמודות ב-manifest: "
        f"{missing_manifest_columns}"
    )

if stage1_manifest_df.empty:
    raise ValueError(
        "ה-manifest ריק."
    )

stage1_manifest_df = (
    stage1_manifest_df
    .sort_values("segment_index")
    .reset_index(drop=True)
    .copy()
)

if stage1_manifest_df[
    "segment_index"
].duplicated().any():
    raise ValueError(
        "נמצאו segment_index כפולים."
    )

numeric_positive_columns = [
    "segment_duration_sec",
    "segment_fps",
    "segment_frame_count"
]

for column in numeric_positive_columns:
    if (
        pd.to_numeric(
            stage1_manifest_df[column],
            errors="coerce"
        )
        <= 0
    ).any():
        raise ValueError(
            f"נמצא ערך לא תקין בעמודה {column}."
        )

resolved_segment_paths = []

for path_text in stage1_manifest_df[
    "drive_segment_path"
].astype(str):
    segment_path = Path(
        path_text
    ).expanduser()

    if not segment_path.is_file():
        fallback_path = (
            manifest_path.parent
            / segment_path.parent.name
            / segment_path.name
        )

        if fallback_path.is_file():
            segment_path = fallback_path

    resolved_segment_paths.append(
        segment_path.resolve()
    )

stage1_manifest_df[
    "stage2_segment_path"
] = [
    str(path)
    for path in resolved_segment_paths
]

missing_segment_paths = [
    path
    for path in resolved_segment_paths
    if not path.is_file()
]

if missing_segment_paths:
    raise FileNotFoundError(
        "חסרים קובצי segment:\n"
        + "\n".join(
            str(path)
            for path in missing_segment_paths
        )
    )

source_video_names = (
    stage1_manifest_df[
        "source_video_name"
    ]
    .astype(str)
    .unique()
)

if len(source_video_names) != 1:
    raise ValueError(
        "ה-manifest צריך להתייחס לסרטון מקור אחד."
    )

SOURCE_VIDEO_NAME = str(
    source_video_names[0]
)

SOURCE_VIDEO_STEM = make_stage2_safe_stem(
    Path(SOURCE_VIDEO_NAME).stem
)

STAGE1_MANIFEST_PATH = manifest_path
STAGE1_RUN_DIR = (
    STAGE1_MANIFEST_PATH
    .parent
    .parent
)

quality_candidates = sorted(
    STAGE1_MANIFEST_PATH.parent.glob(
        "*_quality.json"
    )
)

if len(quality_candidates) != 1:
    raise FileNotFoundError(
        "ציפיתי למצוא דוח איכות Stage 1 אחד בדיוק בתוך:\n"
        f"{STAGE1_MANIFEST_PATH.parent}\n"
        f"נמצאו: {len(quality_candidates)}"
    )

DRIVE_MANIFEST_CSV_PATH = STAGE1_MANIFEST_PATH
DRIVE_QUALITY_JSON_PATH = quality_candidates[0].resolve()

clean_video_candidates = sorted(
    STAGE1_MANIFEST_PATH.parent.glob("*_flight_only.mp4")
)
if len(clean_video_candidates) != 1:
    raise FileNotFoundError(
        "ציפיתי למצוא סרטון Stage 1 נקי אחד בדיוק בתוך:\n"
        f"{STAGE1_MANIFEST_PATH.parent}\n"
        f"נמצאו: {len(clean_video_candidates)}"
    )
DRIVE_FINAL_VIDEO_PATH = clean_video_candidates[0].resolve()

with open(
    DRIVE_QUALITY_JSON_PATH,
    "r",
    encoding="utf-8"
) as file:
    stage1_quality_summary = json.load(file)

# Backward-compatible aliases used by the final result cell after a resumed run.
manifest_df = stage1_manifest_df.copy()

STAGE1_SOURCE_FPS = float(
    stage1_quality_summary
    .get("source_video", {})
    .get(
        "fps",
        stage1_manifest_df[
            "segment_fps"
        ].median()
    )
)

stage1_resume_summary_df = pd.DataFrame([{
    "manifest_source": manifest_source,
    "segments": len(stage1_manifest_df),
    "duration_sec": round(float(stage1_manifest_df["segment_duration_sec"].sum()), 3),
    "clean_video": str(DRIVE_FINAL_VIDEO_PATH),
    "stage1_quality": str(DRIVE_QUALITY_JSON_PATH),
}])
display(stage1_resume_summary_df)
display(
    stage1_manifest_df[
        [
            "segment_index",
            "source_start_sec",
            "source_end_sec",
            "segment_duration_sec",
            "segment_frame_count",
            "stage2_segment_path"
        ]
    ]
)


In [ ]:
STAGE2_RUNTIME_DIR = (
    Path("/content/fpv_stage2_runtime")
    / SOURCE_VIDEO_STEM
    / STAGE2_VERSION
)
STAGE2_RUNTIME_FRAMES_DIR = STAGE2_RUNTIME_DIR / "candidate_frames"
STAGE2_RUNTIME_OUTPUT_DIR = STAGE2_RUNTIME_DIR / "output"

STAGE2_DRIVE_DIR = (
    STAGE1_RUN_DIR
    / "stage2"
    / STAGE2_VERSION
)
STAGE2_DRIVE_FRAMES_DIR = STAGE2_DRIVE_DIR / "candidate_frames"
STAGE2_DRIVE_OUTPUT_DIR = STAGE2_DRIVE_DIR / "output"

folders_to_create = [
    STAGE2_RUNTIME_DIR,
    STAGE2_RUNTIME_FRAMES_DIR,
    STAGE2_RUNTIME_OUTPUT_DIR,
    STAGE2_DRIVE_DIR,
    STAGE2_DRIVE_OUTPUT_DIR,
]
if SAVE_KEYFRAME_CANDIDATES_TO_DRIVE:
    folders_to_create.append(STAGE2_DRIVE_FRAMES_DIR)

for folder in folders_to_create:
    folder.mkdir(parents=True, exist_ok=True)

for stale_frame in STAGE2_RUNTIME_FRAMES_DIR.glob("candidate_*.png"):
    stale_frame.unlink()

# Drive candidates are derived cache. Preserve them only when resume is enabled.
if SAVE_KEYFRAME_CANDIDATES_TO_DRIVE and not STAGE2_REUSE_CANDIDATE_CACHE:
    for stale_frame in STAGE2_DRIVE_FRAMES_DIR.glob("candidate_*.png"):
        stale_frame.unlink()


In [ ]:
def make_even_frame_indices(frame_count, sample_count):
    frame_count = int(frame_count)
    sample_count = int(sample_count)
    if not 1 <= sample_count <= frame_count:
        raise ValueError("מספר הדגימות אינו מתאים למספר הפריימים.")
    if sample_count == 1:
        return np.array([frame_count // 2], dtype=int)
    indices = np.rint(
        np.linspace(0, frame_count - 1, sample_count)
    ).astype(int)
    if len(np.unique(indices)) != sample_count:
        raise RuntimeError("נוצרו אינדקסים כפולים בדגימת המועמדים.")
    return indices


total_stage1_duration = float(
    stage1_manifest_df["segment_duration_sec"].astype(float).sum()
)
desired_total_candidates = int(
    np.ceil(total_stage1_duration * KEYFRAME_CANDIDATE_FPS)
) + len(stage1_manifest_df)
desired_total_candidates = min(
    KEYFRAME_MAX_CANDIDATES,
    max(desired_total_candidates, len(stage1_manifest_df)),
)

durations = stage1_manifest_df["segment_duration_sec"].astype(float).to_numpy()
available = stage1_manifest_df["segment_frame_count"].astype(int).to_numpy()
desired_float = desired_total_candidates * durations / durations.sum()
allocation = np.minimum(available, np.maximum(1, np.floor(desired_float).astype(int)))

while int(allocation.sum()) < min(desired_total_candidates, int(available.sum())):
    can_add = allocation < available
    if not can_add.any():
        break
    priorities = desired_float - allocation
    priorities[~can_add] = -np.inf
    allocation[int(np.argmax(priorities))] += 1

while int(allocation.sum()) > desired_total_candidates:
    can_remove = allocation > 1
    if not can_remove.any():
        break
    excess = allocation - desired_float
    excess[~can_remove] = -np.inf
    allocation[int(np.argmax(excess))] -= 1

candidate_allocation_df = stage1_manifest_df[
    ["segment_index", "segment_duration_sec", "segment_frame_count"]
].copy()
candidate_allocation_df["candidate_count"] = allocation.astype(int)
display(candidate_allocation_df)


In [ ]:
candidate_rows = []
candidate_index = 0
candidate_cache_hits = 0
candidate_cache_misses = 0
candidate_decode_fallback_count = 0

for _, allocation_row in candidate_allocation_df.sort_values("segment_index").iterrows():
    segment_index = int(allocation_row["segment_index"])
    segment_row = stage1_manifest_df[
        stage1_manifest_df["segment_index"] == segment_index
    ].iloc[0]
    segment_path = Path(segment_row["stage2_segment_path"])
    segment_frame_count = int(segment_row["segment_frame_count"])
    segment_fps = float(segment_row["segment_fps"])
    local_indices = make_even_frame_indices(
        segment_frame_count,
        int(allocation_row["candidate_count"]),
    )

    # Ordered decoding is much more reliable for MP4 than repeated random seeks,
    # especially at the final container-reported frame.
    cap = cv2.VideoCapture(str(segment_path))
    if not cap.isOpened():
        raise ValueError(f"Cannot open segment: {segment_path}")

    current_decoded_index = -1
    last_decoded_frame = None
    decoder_exhausted = False
    terminal_tolerance_frames = max(2, int(np.ceil(segment_fps * 0.10)))
    used_actual_frame_indices = set()

    for local_order, requested_frame_index in enumerate(local_indices):
        requested_frame_index = int(requested_frame_index)
        frame_filename = (
            f"candidate_{candidate_index:04d}_"
            f"seg{segment_index:03d}_"
            f"local{requested_frame_index:06d}.png"
        )
        runtime_path = STAGE2_RUNTIME_FRAMES_DIR / frame_filename
        drive_cache_path = STAGE2_DRIVE_FRAMES_DIR / frame_filename

        cache_is_valid = False
        if (
            SAVE_KEYFRAME_CANDIDATES_TO_DRIVE
            and STAGE2_REUSE_CANDIDATE_CACHE
            and drive_cache_path.is_file()
            and drive_cache_path.stat().st_size > 0
        ):
            cached_frame = cv2.imread(str(drive_cache_path), cv2.IMREAD_COLOR)
            cache_is_valid = cached_frame is not None and cached_frame.size > 0

        actual_frame_index = requested_frame_index
        decode_fallback = "NONE"
        decode_fallback_frames = 0

        if cache_is_valid:
            shutil.copy2(drive_cache_path, runtime_path)
            candidate_cache_hits += 1
        else:
            while current_decoded_index < requested_frame_index and not decoder_exhausted:
                success, decoded_frame = cap.read()
                if not success or decoded_frame is None:
                    decoder_exhausted = True
                    break
                current_decoded_index += 1
                last_decoded_frame = decoded_frame

            if current_decoded_index == requested_frame_index and last_decoded_frame is not None:
                frame = last_decoded_frame
            else:
                missing_distance = requested_frame_index - current_decoded_index
                request_is_terminal = (
                    requested_frame_index
                    >= segment_frame_count - terminal_tolerance_frames - 1
                )
                can_clamp_terminal_frame = (
                    last_decoded_frame is not None
                    and request_is_terminal
                    and 0 < missing_distance <= terminal_tolerance_frames
                )
                if not can_clamp_terminal_frame:
                    cap.release()
                    raise RuntimeError(
                        "Segment decoding stopped too early. "
                        f"segment={segment_index}, requested={requested_frame_index}, "
                        f"last_decoded={current_decoded_index}, declared_frames={segment_frame_count}. "
                        "The segment may be damaged; this is not treated as a safe terminal-frame mismatch."
                    )
                frame = last_decoded_frame.copy()
                actual_frame_index = current_decoded_index
                decode_fallback = "TERMINAL_EOF_CLAMP"
                decode_fallback_frames = int(missing_distance)
                candidate_decode_fallback_count += 1

            if not cv2.imwrite(str(runtime_path), frame):
                cap.release()
                raise RuntimeError(f"Failed to save candidate PNG: {runtime_path}")

            # A clamped image must not masquerade as an exact cached frame later.
            if SAVE_KEYFRAME_CANDIDATES_TO_DRIVE and decode_fallback == "NONE":
                shutil.copy2(runtime_path, drive_cache_path)
            candidate_cache_misses += 1

        segment_time_sec = actual_frame_index / segment_fps
        source_time_sec = float(segment_row["source_start_sec"]) + segment_time_sec
        clean_time_sec = float(segment_row["clean_timeline_start_sec"]) + segment_time_sec

        if actual_frame_index in used_actual_frame_indices:
            if decode_fallback == "TERMINAL_EOF_CLAMP":
                runtime_path.unlink(missing_ok=True)
                print(
                    f"Skipped duplicate terminal candidate in segment {segment_index}: "
                    f"requested={requested_frame_index}, actual={actual_frame_index}."
                )
                continue
            cap.release()
            raise RuntimeError(
                f"Duplicate exact candidate frame in segment {segment_index}: {actual_frame_index}."
            )
        used_actual_frame_indices.add(actual_frame_index)
        candidate_rows.append({
            "candidate_index": candidate_index,
            "segment_index": segment_index,
            "segment_candidate_order": int(local_order),
            "requested_segment_frame_index": requested_frame_index,
            "segment_frame_index": int(actual_frame_index),
            "decode_fallback": decode_fallback,
            "decode_fallback_frames": int(decode_fallback_frames),
            "segment_time_sec": float(segment_time_sec),
            "source_time_sec": float(source_time_sec),
            "clean_time_sec": float(clean_time_sec),
            "frame_filename": frame_filename,
            "frame_runtime_path": str(runtime_path),
            "candidate_cache_hit": bool(cache_is_valid),
        })
        candidate_index += 1

    cap.release()

candidate_frames_df = pd.DataFrame(candidate_rows).sort_values(
    ["segment_index", "segment_frame_index"]
).reset_index(drop=True)
if candidate_frames_df.empty:
    raise RuntimeError("No keyframe candidates were extracted.")

duplicate_actual_frames = candidate_frames_df.duplicated(
    ["segment_index", "segment_frame_index"], keep=False
)
if duplicate_actual_frames.any():
    duplicate_rows = candidate_frames_df.loc[
        duplicate_actual_frames,
        ["segment_index", "requested_segment_frame_index", "segment_frame_index"],
    ]
    raise RuntimeError(
        "Resilient decoding produced duplicate candidate frames:\n"
        + duplicate_rows.to_string(index=False)
    )

missing_runtime_candidates = [
    path_text
    for path_text in candidate_frames_df["frame_runtime_path"]
    if not Path(path_text).is_file() or Path(path_text).stat().st_size == 0
]
if missing_runtime_candidates:
    raise RuntimeError(
        "Missing runtime keyframe candidates:\n" + "\n".join(missing_runtime_candidates)
    )

display(candidate_frames_df.head())
print(
    f"Prepared {len(candidate_frames_df)} lossless candidates; "
    f"cache hits={candidate_cache_hits}, misses={candidate_cache_misses}, "
    f"safe terminal clamps={candidate_decode_fallback_count}."
)

In [ ]:
def percentile_rank(values):
    series = pd.Series(values, dtype=float)
    return series.rank(method="average", pct=True).fillna(0.0).to_numpy()


def load_candidate_features(path_text):
    frame = cv2.imread(str(path_text), cv2.IMREAD_COLOR)
    if frame is None:
        raise RuntimeError(f"לא ניתן לקרוא מועמד: {path_text}")
    small = stage2_resize_for_analysis(frame, KEYFRAME_ANALYSIS_WIDTH)
    gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    mask = stage2_make_tracking_mask(*gray.shape)
    laplacian = cv2.Laplacian(gray, cv2.CV_32F)
    sharpness = float(laplacian.var())
    brightness = float(gray.mean())
    contrast = float(gray.std())
    clipped_ratio = float(np.mean((gray <= 8) | (gray >= 247)))

    orb = cv2.ORB_create(
        nfeatures=KEYFRAME_ORB_FEATURES,
        scaleFactor=1.2,
        nlevels=8,
        edgeThreshold=19,
        fastThreshold=12,
    )
    keypoints, descriptors = orb.detectAndCompute(gray, mask)
    points = np.empty((0, 2), dtype=np.float32)
    if keypoints:
        points = np.array([kp.pt for kp in keypoints], dtype=np.float32)
    return {
        "gray": gray,
        "keypoints": keypoints or [],
        "points": points,
        "descriptors": descriptors,
        "sharpness": sharpness,
        "brightness": brightness,
        "contrast": contrast,
        "clipped_ratio": clipped_ratio,
    }


def grid_coverage(points, image_shape, grid_size=4):
    if points is None or len(points) == 0:
        return 0.0
    height, width = image_shape[:2]
    x_bins = np.clip((points[:, 0] / max(width, 1) * grid_size).astype(int), 0, grid_size - 1)
    y_bins = np.clip((points[:, 1] / max(height, 1) * grid_size).astype(int), 0, grid_size - 1)
    occupied = len(set(zip(x_bins.tolist(), y_bins.tolist())))
    return float(occupied / (grid_size * grid_size))


def verified_overlap(feature_a, feature_b):
    desc_a = feature_a["descriptors"]
    desc_b = feature_b["descriptors"]
    if desc_a is None or desc_b is None or len(desc_a) < 2 or len(desc_b) < 2:
        return {"matches": 0, "inliers": 0, "inlier_ratio": 0.0, "coverage": 0.0, "score": 0.0}

    matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    raw_matches = matcher.knnMatch(desc_a, desc_b, k=2)
    good = [pair[0] for pair in raw_matches if len(pair) == 2 and pair[0].distance < 0.78 * pair[1].distance]
    if len(good) < 4:
        return {"matches": len(good), "inliers": 0, "inlier_ratio": 0.0, "coverage": 0.0, "score": 0.0}

    points_a = np.float32([feature_a["keypoints"][match.queryIdx].pt for match in good])
    points_b = np.float32([feature_b["keypoints"][match.trainIdx].pt for match in good])
    masks = []
    if len(good) >= 8:
        try:
            _, fundamental_mask = cv2.findFundamentalMat(
                points_a,
                points_b,
                cv2.FM_RANSAC,
                1.5,
                0.995,
            )
            if fundamental_mask is not None:
                masks.append(fundamental_mask.reshape(-1).astype(bool))
        except cv2.error:
            pass
    try:
        _, homography_mask = cv2.findHomography(
            points_a,
            points_b,
            cv2.RANSAC,
            3.0,
        )
        if homography_mask is not None:
            masks.append(homography_mask.reshape(-1).astype(bool))
    except cv2.error:
        pass

    if not masks:
        inlier_mask = np.zeros(len(good), dtype=bool)
    else:
        inlier_mask = max(masks, key=lambda item: int(item.sum()))
    inliers = int(inlier_mask.sum())
    inlier_ratio = float(inliers / max(len(good), 1))
    if inliers:
        coverage = min(
            grid_coverage(points_a[inlier_mask], feature_a["gray"].shape),
            grid_coverage(points_b[inlier_mask], feature_b["gray"].shape),
        )
    else:
        coverage = 0.0
    score = float(
        np.log1p(inliers)
        * np.sqrt(max(inlier_ratio, 0.0))
        * np.sqrt(max(coverage, 0.0))
    )
    return {
        "matches": int(len(good)),
        "inliers": inliers,
        "inlier_ratio": inlier_ratio,
        "coverage": float(coverage),
        "score": score,
    }


candidate_feature_map = {
    int(row.candidate_index): load_candidate_features(row.frame_runtime_path)
    for row in candidate_frames_df.itertuples()
}

for metric_name in ["sharpness", "brightness", "contrast", "clipped_ratio"]:
    candidate_frames_df[metric_name] = [
        candidate_feature_map[int(index)][metric_name]
        for index in candidate_frames_df["candidate_index"]
    ]

candidate_frames_df["incoming_local_inliers"] = np.nan
candidate_frames_df["incoming_local_ratio"] = np.nan
candidate_frames_df["incoming_local_coverage"] = np.nan
candidate_frames_df["incoming_local_score"] = np.nan
candidate_frames_df["outgoing_local_inliers"] = np.nan
candidate_frames_df["outgoing_local_ratio"] = np.nan
candidate_frames_df["outgoing_local_coverage"] = np.nan
candidate_frames_df["outgoing_local_score"] = np.nan

candidate_support_map = {
    int(index): []
    for index in candidate_frames_df["candidate_index"]
}
for _, segment_group in candidate_frames_df.groupby("segment_index", sort=True):
    positions = segment_group.index.to_numpy(dtype=int)
    for offset in (1, 2):
        for previous_position, current_position in zip(positions[:-offset], positions[offset:]):
            previous_index = int(candidate_frames_df.loc[previous_position, "candidate_index"])
            current_index = int(candidate_frames_df.loc[current_position, "candidate_index"])
            overlap = verified_overlap(
                candidate_feature_map[previous_index],
                candidate_feature_map[current_index],
            )
            candidate_support_map[previous_index].append(overlap)
            candidate_support_map[current_index].append(overlap)
            if offset == 1:
                candidate_frames_df.loc[current_position, "incoming_local_inliers"] = overlap["inliers"]
                candidate_frames_df.loc[current_position, "incoming_local_ratio"] = overlap["inlier_ratio"]
                candidate_frames_df.loc[current_position, "incoming_local_coverage"] = overlap["coverage"]
                candidate_frames_df.loc[current_position, "incoming_local_score"] = overlap["score"]
                candidate_frames_df.loc[previous_position, "outgoing_local_inliers"] = overlap["inliers"]
                candidate_frames_df.loc[previous_position, "outgoing_local_ratio"] = overlap["inlier_ratio"]
                candidate_frames_df.loc[previous_position, "outgoing_local_coverage"] = overlap["coverage"]
                candidate_frames_df.loc[previous_position, "outgoing_local_score"] = overlap["score"]


def robust_neighbor_support(items, key):
    values = sorted((float(item[key]) for item in items), reverse=True)
    if not values:
        return 0.0
    # שני פריימים משובשים סמוכים אינם רשאים לאשר זה את זה לבדם.
    return values[1] if len(values) >= 2 else values[0]


robust_local_inliers = np.array([
    robust_neighbor_support(candidate_support_map[int(index)], "inliers")
    for index in candidate_frames_df["candidate_index"]
], dtype=float)
robust_local_coverage = np.array([
    robust_neighbor_support(candidate_support_map[int(index)], "coverage")
    for index in candidate_frames_df["candidate_index"]
], dtype=float)
robust_local_score = np.array([
    robust_neighbor_support(candidate_support_map[int(index)], "score")
    for index in candidate_frames_df["candidate_index"]
], dtype=float)

candidate_frames_df["local_support_inliers"] = robust_local_inliers
candidate_frames_df["local_support_coverage"] = robust_local_coverage
candidate_frames_df["local_support_score"] = robust_local_score
candidate_frames_df["isolated_candidate"] = (
    (robust_local_inliers < 6)
    & (robust_local_coverage < 0.02)
)
candidate_frames_df["exposure_warning"] = (
    (candidate_frames_df["clipped_ratio"] > 0.72)
    | (candidate_frames_df["contrast"] < 6.0)
)
candidate_frames_df["candidate_trusted"] = ~(
    candidate_frames_df["isolated_candidate"]
    | candidate_frames_df["exposure_warning"]
)

exposure_quality = np.clip(1.0 - candidate_frames_df["clipped_ratio"].to_numpy(float), 0.0, 1.0)
candidate_frames_df["quality_score"] = (
    0.48 * percentile_rank(robust_local_score)
    + 0.24 * percentile_rank(candidate_frames_df["sharpness"])
    + 0.13 * percentile_rank(candidate_frames_df["contrast"])
    + 0.15 * exposure_quality
)

if len(stage1_manifest_df) > KEYFRAME_MAX_MODEL_FRAMES:
    raise RuntimeError(
        "מספר מקטעי Stage 1 גדול ממספר פריימי המודל המרבי; "
        "לא ניתן לשמור לפחות Keyframe אחד לכל רכיב בלי להוריד איכות."
    )

# בדחיסות הרגילה בוחרים 0s, 1s, 2s, ... בלי לכפות את פריים הסיום,
# בדיוק כמו sampler הווידאו ב-demo הרשמי.
regular_segment_counts = np.maximum(
    KEYFRAME_MIN_PER_STAGE1_SEGMENT,
    np.floor(durations * KEYFRAME_TARGET_FPS + 1e-9).astype(int) + 1,
)
candidate_counts = candidate_allocation_df["candidate_count"].astype(int).to_numpy()
requested_model_total = min(
    KEYFRAME_MAX_MODEL_FRAMES,
    max(KEYFRAME_MIN_MODEL_FRAMES, int(regular_segment_counts.sum())),
    int(candidate_counts.sum()),
)

desired_model_float = requested_model_total * durations / durations.sum()
model_allocation = np.maximum(
    KEYFRAME_MIN_PER_STAGE1_SEGMENT,
    np.floor(desired_model_float).astype(int),
)
model_allocation = np.minimum(model_allocation, candidate_counts)
while int(model_allocation.sum()) < requested_model_total:
    can_add = model_allocation < candidate_counts
    priorities = desired_model_float - model_allocation
    priorities[~can_add] = -np.inf
    if not np.isfinite(priorities).any():
        break
    model_allocation[int(np.argmax(priorities))] += 1
while int(model_allocation.sum()) > requested_model_total:
    can_remove = model_allocation > KEYFRAME_MIN_PER_STAGE1_SEGMENT
    excess = model_allocation - desired_model_float
    excess[~can_remove] = -np.inf
    if not np.isfinite(excess).any():
        break
    model_allocation[int(np.argmax(excess))] -= 1

adaptive_target_count = int(model_allocation.sum())
if adaptive_target_count != requested_model_total:
    raise RuntimeError(
        f"לא ניתן להקצות {requested_model_total} Keyframes למקטעי Stage 1; "
        f"הוקצו {adaptive_target_count}."
    )

selected_candidate_indices = []
forced_untrusted_indices = []
selection_target_by_candidate = {}
for allocation_position, segment_index in enumerate(
    candidate_allocation_df.sort_values("segment_index")["segment_index"].astype(int)
):
    segment_group = candidate_frames_df[
        candidate_frames_df["segment_index"] == segment_index
    ].sort_values("clean_time_sec").copy()
    sample_count = int(model_allocation[allocation_position])
    segment_duration = float(durations[allocation_position])
    regular_count = int(regular_segment_counts[allocation_position])
    if sample_count == regular_count and segment_duration >= 1.0 / KEYFRAME_TARGET_FPS:
        local_targets = (
            np.arange(sample_count, dtype=float)
            / KEYFRAME_TARGET_FPS
        )
    else:
        local_targets = segment_duration * (
            np.arange(1, sample_count + 1, dtype=float)
            / (sample_count + 1)
        )
    clean_start = float(
        stage1_manifest_df.loc[
            stage1_manifest_df["segment_index"] == segment_index,
            "clean_timeline_start_sec",
        ].iloc[0]
    )
    target_times = clean_start + local_targets
    start_time = float(segment_group["clean_time_sec"].min())
    end_time = float(segment_group["clean_time_sec"].max())
    target_times = np.clip(target_times, start_time, end_time)
    internal_edges = (target_times[:-1] + target_times[1:]) / 2.0
    edges = np.concatenate((
        np.array([start_time], dtype=float),
        internal_edges,
        np.array([end_time + 1e-9], dtype=float),
    ))

    for bin_index in range(sample_count):
        left = edges[bin_index]
        right = edges[bin_index + 1]
        if bin_index == sample_count - 1:
            in_bin = segment_group[
                (segment_group["clean_time_sec"] >= left)
                & (segment_group["clean_time_sec"] <= right)
            ].copy()
        else:
            in_bin = segment_group[
                (segment_group["clean_time_sec"] >= left)
                & (segment_group["clean_time_sec"] < right)
            ].copy()
        center = float(target_times[bin_index])
        if in_bin.empty:
            in_bin = segment_group.iloc[[
                int(np.argmin(np.abs(segment_group["clean_time_sec"].to_numpy(float) - center)))
            ]].copy()

        trusted_bin = in_bin[in_bin["candidate_trusted"]]
        forced = trusted_bin.empty
        pool = in_bin if forced else trusted_bin
        half_width = max((right - left) / 2.0, 1e-6)
        pool = pool.copy()
        pool["bin_center_bonus"] = 1.0 - np.clip(
            np.abs(pool["clean_time_sec"].to_numpy(float) - center) / half_width,
            0.0,
            1.0,
        )
        pool["selection_score"] = pool["quality_score"] + 0.08 * pool["bin_center_bonus"]
        chosen = pool.sort_values(
            ["selection_score", "local_support_score", "sharpness"],
            ascending=False,
        ).iloc[0]
        chosen_index = int(chosen["candidate_index"])
        selected_candidate_indices.append(chosen_index)
        selection_target_by_candidate[chosen_index] = center
        if forced:
            forced_untrusted_indices.append(chosen_index)

model_frames_df = candidate_frames_df[
    candidate_frames_df["candidate_index"].isin(selected_candidate_indices)
].sort_values(["clean_time_sec", "segment_index"]).reset_index(drop=True).copy()
if len(set(selected_candidate_indices)) != len(selected_candidate_indices):
    raise RuntimeError("אותו Keyframe נבחר ליותר מחלון זמן אחד.")
if len(model_frames_df) != adaptive_target_count:
    raise RuntimeError(
        f"מספר ה-Keyframes שנבחר ({len(model_frames_df)}) אינו תואם ליעד "
        f"({adaptive_target_count})."
    )
model_frames_df["model_frame_index"] = np.arange(len(model_frames_df), dtype=int)
model_frames_df["selection_target_clean_sec"] = model_frames_df["candidate_index"].map(
    selection_target_by_candidate
).astype(float)
model_frames_df["selection_target_error_sec"] = (
    model_frames_df["clean_time_sec"]
    - model_frames_df["selection_target_clean_sec"]
).abs()
model_frames_df["forced_untrusted_selection"] = model_frames_df["candidate_index"].isin(
    forced_untrusted_indices
)
model_frames_df["input_frame_trusted"] = (
    model_frames_df["candidate_trusted"]
    & ~model_frames_df["forced_untrusted_selection"]
)

selected_incoming = []
for row_index in range(len(model_frames_df)):
    if row_index == 0:
        selected_incoming.append(None)
        continue
    previous_index = int(model_frames_df.loc[row_index - 1, "candidate_index"])
    current_index = int(model_frames_df.loc[row_index, "candidate_index"])
    selected_incoming.append(
        verified_overlap(
            candidate_feature_map[previous_index],
            candidate_feature_map[current_index],
        )
    )

for column, key in [
    ("incoming_selected_matches", "matches"),
    ("incoming_selected_inliers", "inliers"),
    ("incoming_selected_ratio", "inlier_ratio"),
    ("incoming_selected_coverage", "coverage"),
    ("incoming_selected_score", "score"),
]:
    model_frames_df[column] = [
        np.nan if item is None else item[key]
        for item in selected_incoming
    ]

segment_change = model_frames_df["segment_index"].ne(
    model_frames_df["segment_index"].shift()
).to_numpy(bool)
segment_change[0] = False
model_frames_df["stage1_segment_changed"] = segment_change

continuity_warning = (
    (model_frames_df["incoming_selected_inliers"].fillna(999) < 6)
    & (model_frames_df["incoming_selected_coverage"].fillna(1.0) < 0.02)
).to_numpy(bool)
continuity_warning[segment_change] = False
model_frames_df["input_continuity_warning"] = continuity_warning

geometry_component = np.zeros(len(model_frames_df), dtype=int)
component_index = 0
for row_index in range(1, len(model_frames_df)):
    if segment_change[row_index]:
        previous_segment = int(model_frames_df.loc[row_index - 1, "segment_index"])
        current_segment = int(model_frames_df.loc[row_index, "segment_index"])
        previous_rows = model_frames_df[
            model_frames_df["segment_index"] == previous_segment
        ].tail(3)
        current_rows = model_frames_df[
            model_frames_df["segment_index"] == current_segment
        ].head(3)
        boundary_overlaps = []
        for previous_candidate in previous_rows["candidate_index"].astype(int):
            for current_candidate in current_rows["candidate_index"].astype(int):
                boundary_overlaps.append(
                    verified_overlap(
                        candidate_feature_map[previous_candidate],
                        candidate_feature_map[current_candidate],
                    )
                )
        best_boundary = max(
            boundary_overlaps,
            key=lambda item: (item["inliers"], item["coverage"], item["score"]),
        )
        boundary_is_disconnected = (
            best_boundary["inliers"] < 8
            and best_boundary["coverage"] < 0.02
        )
        if boundary_is_disconnected:
            component_index += 1
    geometry_component[row_index] = component_index

model_frames_df["geometry_segment_index"] = geometry_component

candidate_frames_df["selected_for_model"] = candidate_frames_df["candidate_index"].isin(
    model_frames_df["candidate_index"]
)
candidate_frames_df["selection_reason"] = np.where(
    candidate_frames_df["selected_for_model"],
    "SELECTED",
    np.where(
        candidate_frames_df["isolated_candidate"],
        "REJECTED_ISOLATED",
        np.where(
            candidate_frames_df["exposure_warning"],
            "REJECTED_EXPOSURE",
            "NOT_BEST_IN_TIME_BIN",
        ),
    ),
)

KEYFRAME_AUDIT_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_keyframe_selection_audit.csv"
)
KEYFRAME_AUDIT_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / KEYFRAME_AUDIT_RUNTIME_PATH.name
MODEL_FRAMES_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_selected_model_frames.csv"
)
MODEL_FRAMES_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / MODEL_FRAMES_RUNTIME_PATH.name

candidate_frames_df.drop(columns=[], errors="ignore").to_csv(KEYFRAME_AUDIT_RUNTIME_PATH, index=False)
model_frames_df.to_csv(MODEL_FRAMES_RUNTIME_PATH, index=False)
shutil.copy2(KEYFRAME_AUDIT_RUNTIME_PATH, KEYFRAME_AUDIT_DRIVE_PATH)
shutil.copy2(MODEL_FRAMES_RUNTIME_PATH, MODEL_FRAMES_DRIVE_PATH)

for required_keyframe_artifact in [
    KEYFRAME_AUDIT_DRIVE_PATH,
    MODEL_FRAMES_DRIVE_PATH,
]:
    if not required_keyframe_artifact.is_file() or required_keyframe_artifact.stat().st_size == 0:
        raise RuntimeError(f"תוצר Keyframes חסר או ריק: {required_keyframe_artifact}")

selection_summary_df = pd.DataFrame([{
    "candidate_frames": len(candidate_frames_df),
    "model_frames": len(model_frames_df),
    "target_fps": KEYFRAME_TARGET_FPS,
    "isolated_candidates": int(candidate_frames_df["isolated_candidate"].sum()),
    "forced_untrusted": int(model_frames_df["forced_untrusted_selection"].sum()),
    "continuity_warnings": int(model_frames_df["input_continuity_warning"].sum()),
    "geometry_segments": int(model_frames_df["geometry_segment_index"].nunique()),
}])
display(selection_summary_df)


In [ ]:
import gc
import importlib
import importlib.util
import os

try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch לא נמצא בסביבת Colab.") from error

if not torch.cuda.is_available():
    raise RuntimeError(
        "VGGT-Omega דורש CUDA GPU. בחר Runtime → Change runtime type → T4 GPU."
    )

gpu_free_bytes, gpu_total_bytes = torch.cuda.mem_get_info()
GPU_FREE_MEMORY_GB_AT_PREFLIGHT = gpu_free_bytes / (1024 ** 3)
GPU_TOTAL_MEMORY_GB_AT_PREFLIGHT = gpu_total_bytes / (1024 ** 3)
OMEGA_MODEL_REUSE_CANDIDATE = (
    "omega_model" in globals()
    and globals().get("_LOADED_OMEGA_COMMIT") == OMEGA_REPO_COMMIT
    and bool(globals().get("_LOADED_OMEGA_CHECKPOINT"))
)
if (
    GPU_FREE_MEMORY_GB_AT_PREFLIGHT < OMEGA_MIN_FREE_GPU_MEMORY_GB
    and not OMEGA_MODEL_REUSE_CANDIDATE
):
    raise RuntimeError(
        "אין מספיק זיכרון GPU פנוי ל-Baseline האיכותי: "
        f"{GPU_FREE_MEMORY_GB_AT_PREFLIGHT:.2f} GiB פנויים, "
        f"נדרשים לפחות {OMEGA_MIN_FREE_GPU_MEMORY_GB:.2f} GiB. "
        "לא תתבצע הורדת איכות אוטומטית."
    )

dependency_specs = {
    "huggingface_hub": "huggingface_hub>=0.30",
    "einops": "einops>=0.8",
    "safetensors": "safetensors>=0.4",
    "PIL": "Pillow>=10",
    "torchvision": "torchvision>=0.18",
    "scipy": "scipy>=1.11",
    "trimesh": "trimesh>=4.4",
}
missing_specs = [
    spec for module_name, spec in dependency_specs.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_specs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_specs],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("התקנת תלויות VGGT-Omega נכשלה.")
    importlib.invalidate_caches()

OMEGA_REPO_DIR = Path("/content/vggt_omega_stage2_repo")
if OMEGA_REPO_DIR.exists() and not (OMEGA_REPO_DIR / ".git").is_dir():
    shutil.rmtree(OMEGA_REPO_DIR)
if not OMEGA_REPO_DIR.exists():
    OMEGA_REPO_DIR.mkdir(parents=True, exist_ok=True)
    init_result = subprocess.run(
        ["git", "-C", str(OMEGA_REPO_DIR), "init"],
        capture_output=True,
        text=True,
    )
    if init_result.returncode != 0:
        raise RuntimeError(init_result.stderr or "git init נכשל.")

remote_result = subprocess.run(
    ["git", "-C", str(OMEGA_REPO_DIR), "remote", "get-url", "origin"],
    capture_output=True,
    text=True,
)
if remote_result.returncode != 0:
    remote_result = subprocess.run(
        ["git", "-C", str(OMEGA_REPO_DIR), "remote", "add", "origin", OMEGA_REPO_URL],
        capture_output=True,
        text=True,
    )
    if remote_result.returncode != 0:
        raise RuntimeError(remote_result.stderr or "הגדרת מקור Omega נכשלה.")
elif remote_result.stdout.strip() != OMEGA_REPO_URL:
    remote_result = subprocess.run(
        ["git", "-C", str(OMEGA_REPO_DIR), "remote", "set-url", "origin", OMEGA_REPO_URL],
        capture_output=True,
        text=True,
    )
    if remote_result.returncode != 0:
        raise RuntimeError(remote_result.stderr or "עדכון מקור Omega נכשל.")

current_result = subprocess.run(
    ["git", "-C", str(OMEGA_REPO_DIR), "rev-parse", "HEAD"],
    capture_output=True,
    text=True,
)
current_commit = current_result.stdout.strip() if current_result.returncode == 0 else ""
if current_commit != OMEGA_REPO_COMMIT:
    print("מוריד את קוד VGGT-Omega הרשמי...")
    fetch_result = subprocess.run(
        ["git", "-C", str(OMEGA_REPO_DIR), "fetch", "--depth", "1", "origin", OMEGA_REPO_COMMIT],
        capture_output=True,
        text=True,
    )
    if fetch_result.returncode != 0:
        print(fetch_result.stderr)
        raise RuntimeError("הורדת קוד VGGT-Omega נכשלה.")
    checkout_result = subprocess.run(
        ["git", "-C", str(OMEGA_REPO_DIR), "checkout", "--detach", "--force", "FETCH_HEAD"],
        capture_output=True,
        text=True,
    )
    if checkout_result.returncode != 0:
        raise RuntimeError(checkout_result.stderr or "checkout של Omega נכשל.")

if str(OMEGA_REPO_DIR) not in sys.path:
    sys.path.insert(0, str(OMEGA_REPO_DIR))
for module_name in list(sys.modules):
    if module_name == "vggt_omega" or module_name.startswith("vggt_omega."):
        del sys.modules[module_name]
if "visual_util" in sys.modules:
    del sys.modules["visual_util"]
importlib.invalidate_caches()

from huggingface_hub import hf_hub_download
from vggt_omega.models import VGGTOmega
from vggt_omega.utils.load_fn import load_and_preprocess_images as omega_load_images
from vggt_omega.utils.pose_enc import encoding_to_camera
from visual_util import predictions_to_glb as omega_predictions_to_glb

if OMEGA_CHECKPOINT_DRIVE_PATH.strip():
    omega_checkpoint_path = Path(OMEGA_CHECKPOINT_DRIVE_PATH).expanduser()
    if not omega_checkpoint_path.is_absolute():
        omega_checkpoint_path = MY_DRIVE_DIR / omega_checkpoint_path
    omega_checkpoint_cache_mode_used = "explicit_path"
elif OMEGA_CHECKPOINT_CACHE_MODE == "drive":
    omega_model_dir = MY_DRIVE_DIR / PROJECT_FOLDER_NAME / "models" / "vggt_omega"
    omega_model_dir.mkdir(parents=True, exist_ok=True)
    omega_checkpoint_path = omega_model_dir / OMEGA_CHECKPOINT_FILENAME
    omega_checkpoint_cache_mode_used = "drive"
else:
    omega_model_dir = Path("/content/fpv_omega_model")
    omega_model_dir.mkdir(parents=True, exist_ok=True)
    omega_checkpoint_path = omega_model_dir / OMEGA_CHECKPOINT_FILENAME
    omega_checkpoint_cache_mode_used = "runtime"

if not omega_checkpoint_path.is_file():
    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None

    if not hf_token:
        raise RuntimeError(
            "ה-checkpoint של VGGT-Omega עדיין לא נמצא. אשר גישה ב-"
            "https://huggingface.co/facebook/VGGT-Omega ואז הוסף ב-Colab "
            "Secret בשם HF_TOKEN והריץ שוב. אין להדביק טוקן בתוך המחברת."
        )
    try:
        downloaded_path = hf_hub_download(
            repo_id=OMEGA_MODEL_ID,
            filename=OMEGA_CHECKPOINT_FILENAME,
            token=hf_token,
            local_dir=str(omega_checkpoint_path.parent),
        )
        omega_checkpoint_path = Path(downloaded_path)
    except Exception as error:
        raise RuntimeError(
            "הורדת VGGT-Omega נכשלה. ודא שהגישה אושרה וש-HF_TOKEN תקין."
        ) from error

if omega_checkpoint_path.stat().st_size < 1_000_000_000:
    raise RuntimeError("קובץ ה-checkpoint קטן מדי ונראה חלקי.")

VGGT_DEVICE = torch.device("cuda")
gpu_properties = torch.cuda.get_device_properties(0)
GPU_TOTAL_MEMORY_GB = gpu_properties.total_memory / (1024 ** 3)

display(pd.DataFrame([{
    "gpu": gpu_properties.name,
    "gpu_memory_gb": round(GPU_TOTAL_MEMORY_GB, 2),
    "precision": OMEGA_PRECISION,
    "model": "VGGT-Omega-1B-512",
    "repo_commit": OMEGA_REPO_COMMIT,
    "checkpoint_cached": True,
    "checkpoint_cache_mode": omega_checkpoint_cache_mode_used,
    "gpu_free_memory_gb_at_preflight": round(GPU_FREE_MEMORY_GB_AT_PREFLIGHT, 2),
}]))


In [ ]:
reuse_loaded_model = (
    "omega_model" in globals()
    and globals().get("_LOADED_OMEGA_CHECKPOINT") == str(omega_checkpoint_path.resolve())
    and globals().get("_LOADED_OMEGA_COMMIT") == OMEGA_REPO_COMMIT
)

if reuse_loaded_model:
    print("משתמש ב-VGGT-Omega שכבר נטען ב-Runtime.")
else:
    if "omega_model" in globals():
        del omega_model
    gc.collect()
    torch.cuda.empty_cache()
    print("טוען את משקולות VGGT-Omega-1B-512 ב-FP32...")
    state_dict = torch.load(
        str(omega_checkpoint_path),
        map_location="cpu",
        weights_only=True,
    )
    omega_model = VGGTOmega().eval()
    omega_model.load_state_dict(state_dict, strict=True)
    del state_dict
    omega_model = omega_model.to(VGGT_DEVICE)
    _LOADED_OMEGA_CHECKPOINT = str(omega_checkpoint_path.resolve())
    _LOADED_OMEGA_COMMIT = OMEGA_REPO_COMMIT
    gc.collect()
    torch.cuda.empty_cache()

# The pinned upstream forward enables BF16/FP16 autocast internally.  v1.2.1
# makes the notebook's FP32 contract real and explicit on every GPU.
from types import MethodType

def omega_forward_true_fp32(self, images):
    if len(images.shape) == 4:
        images = images.unsqueeze(0)

    images = images.float()
    with torch.autocast(device_type="cuda", enabled=False):
        aggregated_tokens_list, patch_token_start = self.aggregator(images)
        final_tokens = aggregated_tokens_list[-1]
        if final_tokens is None:
            raise ValueError("Aggregator did not cache the final layer, which VGGTOmega needs.")

        predictions = {
            "camera_and_register_tokens": final_tokens[
                :, :, :patch_token_start
            ].contiguous(),
        }
        if self.camera_head is not None:
            predictions["pose_enc"] = self.camera_head(
                aggregated_tokens_list,
                patch_token_start=patch_token_start,
            )
        if self.dense_head is not None:
            depth, depth_conf = self.dense_head(
                aggregated_tokens_list,
                images=images,
                patch_token_start=patch_token_start,
            )
            predictions["depth"] = depth
            predictions["depth_conf"] = depth_conf
        if self.text_alignment_head is not None:
            predictions.update(
                self.text_alignment_head(
                    aggregated_tokens_list,
                    patch_token_start=patch_token_start,
                )
            )
        if not self.training:
            predictions["images"] = images
    return predictions

omega_model.forward = MethodType(omega_forward_true_fp32, omega_model)
OMEGA_FORWARD_POLICY = "true_fp32_no_autocast"
OMEGA_RUNTIME_COMPUTE_DTYPE = "torch.float32"
if next(omega_model.parameters()).dtype != torch.float32:
    raise RuntimeError("VGGT-Omega model parameters are not FP32.")

model_parameter_count = sum(parameter.numel() for parameter in omega_model.parameters())
display(pd.DataFrame([{
    "parameters": model_parameter_count,
    "gpu_memory_after_load_gb": round(torch.cuda.memory_allocated() / (1024 ** 3), 2),
    "dtype": str(next(omega_model.parameters()).dtype),
    "forward_policy": OMEGA_FORWARD_POLICY,
}]))


In [ ]:
preview_count = min(20, len(model_frames_df))
preview_positions = np.unique(
    np.rint(np.linspace(0, len(model_frames_df) - 1, preview_count)).astype(int)
)
columns = 4
rows = int(np.ceil(len(preview_positions) / columns))
fig, axes = plt.subplots(rows, columns, figsize=(16, rows * 3.1))
axes = np.asarray(axes).reshape(-1)
for axis in axes:
    axis.axis("off")
for axis, position in zip(axes, preview_positions):
    row = model_frames_df.iloc[int(position)]
    image_bgr = cv2.imread(str(row["frame_runtime_path"]))
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    axis.imshow(image_rgb)
    trusted_text = "trusted" if bool(row["input_frame_trusted"]) else "CHECK"
    axis.set_title(
        f"#{int(row['model_frame_index'])} · clean {float(row['clean_time_sec']):.2f}s\n"
        f"seg {int(row['segment_index'])} · {trusted_text}"
    )
    axis.axis("off")
plt.tight_layout()
KEYFRAME_CONTACT_SHEET_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_selected_keyframes.png"
)
KEYFRAME_CONTACT_SHEET_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / KEYFRAME_CONTACT_SHEET_RUNTIME_PATH.name
fig.savefig(KEYFRAME_CONTACT_SHEET_RUNTIME_PATH, dpi=170, bbox_inches="tight")
shutil.copy2(KEYFRAME_CONTACT_SHEET_RUNTIME_PATH, KEYFRAME_CONTACT_SHEET_DRIVE_PATH)
plt.show()

print(
    "בחירת Keyframes הסתיימה:",
    f"{len(model_frames_df)} פריימים למודל,",
    f"{int(model_frames_df['input_frame_trusted'].sum())} מהימנים בקלט."
)


In [ ]:
#@title Reconstruction settings { display-mode: "form" }
STAGE2B_VERSION = "stage2_vggt_omega_resilient_decode"

OMEGA_ALLOW_DEGRADED_GPU_RESULT = False #@param {type:"boolean"}
DENSE_CONFIDENCE_PERCENTILE = 50.0 #@param {type:"number"}
DENSE_FILTER_DEPTH_EDGES = True #@param {type:"boolean"}
DENSE_DEPTH_EDGE_RTOL = 0.03 #@param {type:"number"}
DENSE_MAX_POINTS_K = 750 #@param {type:"integer"}
DENSE_MAX_PLOT_POINTS = 140_000
DENSE_MAX_INTERACTIVE_POINTS_K = 80
DENSE_SHOW_CAMERAS_IN_GLB = True #@param {type:"boolean"}
DENSE_FILTER_SKY = False #@param {type:"boolean"}
DENSE_MASK_BLACK_BACKGROUND = False #@param {type:"boolean"}
DENSE_MASK_WHITE_BACKGROUND = False #@param {type:"boolean"}

if not 0 <= DENSE_CONFIDENCE_PERCENTILE < 100:
    raise ValueError("DENSE_CONFIDENCE_PERCENTILE חייב להיות בין 0 ל-100.")
if not 0.001 <= DENSE_DEPTH_EDGE_RTOL <= 0.25:
    raise ValueError("DENSE_DEPTH_EDGE_RTOL אינו בטווח סביר.")
if not 10 <= DENSE_MAX_POINTS_K <= 10_000:
    raise ValueError("DENSE_MAX_POINTS_K חייב להיות בין 10 ל-10,000.")
if not 10 <= DENSE_MAX_INTERACTIVE_POINTS_K <= 250:
    raise ValueError("DENSE_MAX_INTERACTIVE_POINTS_K חייב להיות בין 10 ל-250.")

DENSE_MAX_EXPORT_POINTS = int(DENSE_MAX_POINTS_K * 1000)
DENSE_MAX_INTERACTIVE_POINTS = int(DENSE_MAX_INTERACTIVE_POINTS_K * 1000)

display(pd.DataFrame([{
    "stage2_version": STAGE2B_VERSION,
    "model_frames": len(model_frames_df),
    "resolution": OMEGA_IMAGE_RESOLUTION,
    "preprocess": OMEGA_PREPROCESS_MODE,
    "precision": OMEGA_PRECISION,
    "confidence_percentile": DENSE_CONFIDENCE_PERCENTILE,
    "depth_edge_filter": DENSE_FILTER_DEPTH_EDGES,
    "gpu_quality_guard": not OMEGA_ALLOW_DEGRADED_GPU_RESULT,
}]))


In [ ]:
try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
except ImportError as error:
    raise RuntimeError(
        "This PyTorch build does not expose explicit SDPA backend control."
    ) from error

OMEGA_ATTENTION_BACKEND = "EFFICIENT_ATTENTION"

def verify_omega_attention_backend():
    probe = torch.zeros(
        (1, 1, 128, 64),
        device=VGGT_DEVICE,
        dtype=torch.float32,
    )
    try:
        with torch.inference_mode(), sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
            probe_output = torch.nn.functional.scaled_dot_product_attention(
                probe,
                probe,
                probe,
            )
        if probe_output.dtype != torch.float32:
            raise RuntimeError(
                f"Efficient attention returned {probe_output.dtype}, expected torch.float32."
            )
    except Exception as error:
        raise RuntimeError(
            "The current GPU/PyTorch combination cannot run FP32 efficient attention. "
            "No lower-quality fallback was used."
        ) from error
    finally:
        if "probe_output" in locals():
            del probe_output
        del probe
        torch.cuda.empty_cache()

verify_omega_attention_backend()

def choose_quality_subset(frame_table, requested_count):
    requested_count = int(requested_count)
    if requested_count >= len(frame_table):
        selected = frame_table.copy()
    else:
        positions = np.unique(
            np.rint(np.linspace(0, len(frame_table) - 1, requested_count)).astype(int)
        )
        selected = frame_table.iloc[positions].copy()
    selected = selected.sort_values("clean_time_sec").reset_index(drop=True)
    selected["model_frame_index"] = np.arange(len(selected), dtype=int)
    return selected


def tensor_to_numpy(value):
    if value is None:
        return None
    array = value.detach().float().cpu().numpy()
    if array.ndim > 0 and array.shape[0] == 1:
        array = array[0]
    return array


def unproject_omega_depth(depth_map, extrinsic, intrinsic):
    depth = np.asarray(depth_map, dtype=np.float32)
    if depth.ndim == 4 and depth.shape[-1] == 1:
        depth = depth[..., 0]
    frame_count, height, width = depth.shape
    y, x = np.meshgrid(
        np.arange(height, dtype=np.float32),
        np.arange(width, dtype=np.float32),
        indexing="ij",
    )
    x = np.broadcast_to(x[None], (frame_count, height, width))
    y = np.broadcast_to(y[None], (frame_count, height, width))
    fx = intrinsic[:, 0, 0][:, None, None]
    fy = intrinsic[:, 1, 1][:, None, None]
    cx = intrinsic[:, 0, 2][:, None, None]
    cy = intrinsic[:, 1, 2][:, None, None]
    camera_points = np.stack([
        (x - cx) / fx * depth,
        (y - cy) / fy * depth,
        depth,
    ], axis=-1)
    rotation = extrinsic[:, :3, :3]
    translation = extrinsic[:, :3, 3]
    return np.einsum(
        "sij,shwj->shwi",
        np.transpose(rotation, (0, 2, 1)),
        camera_points - translation[:, None, None, :],
    )


def run_omega_inference(model, frames_df):
    image_paths = [str(path) for path in frames_df["frame_runtime_path"]]
    images_cpu = omega_load_images(
        image_paths,
        mode=OMEGA_PREPROCESS_MODE,
        image_resolution=OMEGA_IMAGE_RESOLUTION,
    )
    if images_cpu.ndim != 4 or images_cpu.shape[0] != len(frames_df):
        raise RuntimeError(f"צורת קלט Omega אינה תקינה: {tuple(images_cpu.shape)}")
    input_height = int(images_cpu.shape[-2])
    input_width = int(images_cpu.shape[-1])
    images_gpu = images_cpu.to(VGGT_DEVICE, non_blocking=True)
    predictions_gpu = None
    torch.cuda.reset_peak_memory_stats()
    try:
        with torch.inference_mode(), sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
            # המימוש הרשמי של Omega רץ ב-FP32; אין autocast.
            predictions_gpu = model(images_gpu)
        fp32_outputs = {
            name: value
            for name, value in predictions_gpu.items()
            if isinstance(value, torch.Tensor)
        }
        non_fp32_outputs = {
            name: str(value.dtype)
            for name, value in fp32_outputs.items()
            if value.is_floating_point() and value.dtype != torch.float32
        }
        if non_fp32_outputs:
            raise RuntimeError(
                "Omega returned non-FP32 tensors despite the strict precision policy: "
                f"{non_fp32_outputs}"
            )
        if "pose_enc" not in predictions_gpu or "depth" not in predictions_gpu:
            raise RuntimeError("פלט Omega חסר Pose או Depth.")
        extrinsic_gpu, intrinsic_gpu = encoding_to_camera(
            predictions_gpu["pose_enc"],
            predictions_gpu["images"].shape[-2:],
        )
        predictions = {
            "images": tensor_to_numpy(predictions_gpu["images"]),
            "pose_enc": tensor_to_numpy(predictions_gpu["pose_enc"]),
            "depth": tensor_to_numpy(predictions_gpu["depth"]),
            "depth_conf": tensor_to_numpy(predictions_gpu["depth_conf"]),
            "extrinsic": tensor_to_numpy(extrinsic_gpu),
            "intrinsic": tensor_to_numpy(intrinsic_gpu),
        }
        predictions["world_points_from_depth"] = unproject_omega_depth(
            predictions["depth"],
            predictions["extrinsic"],
            predictions["intrinsic"],
        )
        peak_gpu_memory_gb = float(torch.cuda.max_memory_allocated() / (1024 ** 3))
        return {
            "predictions": predictions,
            "input_height": input_height,
            "input_width": input_width,
            "peak_gpu_memory_gb": peak_gpu_memory_gb,
            "attention_backend": OMEGA_ATTENTION_BACKEND,
            "compute_dtype": OMEGA_RUNTIME_COMPUTE_DTYPE,
        }
    finally:
        if predictions_gpu is not None:
            del predictions_gpu
        del images_gpu
        del images_cpu
        if "extrinsic_gpu" in locals():
            del extrinsic_gpu
        if "intrinsic_gpu" in locals():
            del intrinsic_gpu
        gc.collect()
        torch.cuda.empty_cache()


initial_omega_count = len(model_frames_df)
minimum_quality_count = initial_omega_count
inference_counts = [initial_omega_count]
if OMEGA_ALLOW_DEGRADED_GPU_RESULT:
    for retry_count in [40, 32, 24, 16, 12, 8, 4]:
        if KEYFRAME_MIN_MODEL_FRAMES <= retry_count < initial_omega_count:
            inference_counts.append(retry_count)

omega_run_result = None
omega_used_frames_df = None
omega_last_oom = None
omega_inference_attempts = []

for requested_count in inference_counts:
    candidate_subset = choose_quality_subset(model_frames_df, requested_count)
    print(f"מריץ VGGT-Omega ב-FP32 עם {len(candidate_subset)} Keyframes...")
    try:
        omega_run_result = run_omega_inference(omega_model, candidate_subset)
        omega_used_frames_df = candidate_subset.copy()
        omega_inference_attempts.append({
            "requested_frame_count": int(requested_count),
            "status": "SUCCESS",
            "peak_gpu_memory_gb": float(omega_run_result["peak_gpu_memory_gb"]),
        })
        break
    except (torch.cuda.OutOfMemoryError, RuntimeError) as error:
        is_oom = isinstance(error, torch.cuda.OutOfMemoryError) or "out of memory" in str(error).lower()
        if not is_oom:
            raise
        omega_last_oom = error
        omega_inference_attempts.append({
            "requested_frame_count": int(requested_count),
            "status": "GPU_OUT_OF_MEMORY",
            "error_type": type(error).__name__,
        })
        gc.collect()
        torch.cuda.empty_cache()
        if not OMEGA_ALLOW_DEGRADED_GPU_RESULT:
            break

if omega_run_result is None or omega_used_frames_df is None:
    raise RuntimeError(
        "VGGT-Omega לא הצליח להריץ את כל Keyframes ב-FP32. "
        "התוצאה לא הונמכה בשקט; נסה Runtime חדש או GPU חזק יותר."
    ) from omega_last_oom

omega_frame_fallback_used = len(omega_used_frames_df) < initial_omega_count
omega_gpu_quality_degraded = len(omega_used_frames_df) < minimum_quality_count
if omega_gpu_quality_degraded and not OMEGA_ALLOW_DEGRADED_GPU_RESULT:
    raise RuntimeError("מספר פריימי Omega ירד מתחת לסף האיכות.")

omega_predictions = omega_run_result["predictions"]
OMEGA_USED_FRAMES_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_used_frames.csv"
)
OMEGA_USED_FRAMES_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / OMEGA_USED_FRAMES_RUNTIME_PATH.name
omega_used_frames_df.to_csv(OMEGA_USED_FRAMES_RUNTIME_PATH, index=False)
shutil.copy2(OMEGA_USED_FRAMES_RUNTIME_PATH, OMEGA_USED_FRAMES_DRIVE_PATH)

display(pd.DataFrame([{
    "frames": len(omega_used_frames_df),
    "input_width": omega_run_result["input_width"],
    "input_height": omega_run_result["input_height"],
    "precision": OMEGA_PRECISION,
    "compute_dtype": omega_run_result["compute_dtype"],
    "attention_backend": omega_run_result["attention_backend"],
    "peak_gpu_memory_gb": round(omega_run_result["peak_gpu_memory_gb"], 2),
    "gpu_fallback_used": omega_frame_fallback_used,
}]))


In [ ]:
dense_dependency_specs = {
    "trimesh": "trimesh>=4.4",
    "plotly": "plotly>=5.20",
}
missing_dense_specs = [
    spec for module_name, spec in dense_dependency_specs.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_dense_specs:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing_dense_specs],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("התקנת תלויות התלת-ממד נכשלה.")
    importlib.invalidate_caches()

import trimesh
import plotly.graph_objects as go
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from matplotlib.colors import LinearSegmentedColormap


def omega_depth_edge_mask(depth, rtol=0.03, kernel_size=3):
    depth = np.asarray(depth)
    if depth.ndim == 4 and depth.shape[-1] == 1:
        depth = depth[..., 0]
    original_shape = depth.shape
    flat = depth.reshape(-1, *original_shape[-2:])
    pad = kernel_size // 2
    padded = np.pad(flat, ((0, 0), (pad, pad), (pad, pad)), mode="edge")
    local_max = np.full_like(flat, -np.inf)
    local_min = np.full_like(flat, np.inf)
    for y_offset in range(kernel_size):
        for x_offset in range(kernel_size):
            window = padded[
                :,
                y_offset:y_offset + flat.shape[-2],
                x_offset:x_offset + flat.shape[-1],
            ]
            local_max = np.maximum(local_max, window)
            local_min = np.minimum(local_min, window)
    relative_jump = (local_max - local_min) / np.maximum(np.abs(flat), 1e-6)
    return (relative_jump > rtol).reshape(original_shape)


def exact_percentile_keep_mask(confidence, eligible_mask, percentile):
    confidence = np.asarray(confidence, dtype=np.float64).reshape(-1)
    eligible_mask = np.asarray(eligible_mask, dtype=bool).reshape(-1)
    eligible_indices = np.flatnonzero(eligible_mask)
    if len(eligible_indices) == 0:
        raise RuntimeError("אין נקודות כשירות לסינון Confidence.")
    keep_count = max(
        1,
        int(np.ceil(len(eligible_indices) * (100.0 - float(percentile)) / 100.0)),
    )
    eligible_confidence = confidence[eligible_indices]
    kth_position = len(eligible_confidence) - keep_count
    threshold = float(np.partition(eligible_confidence, kth_position)[kth_position])
    above_indices = eligible_indices[eligible_confidence > threshold]
    tied_indices = eligible_indices[eligible_confidence == threshold]
    remaining = keep_count - len(above_indices)
    if remaining > 0:
        if remaining >= len(tied_indices):
            chosen_ties = tied_indices
        else:
            tie_positions = np.unique(
                np.rint(np.linspace(0, len(tied_indices) - 1, remaining)).astype(int)
            )
            if len(tie_positions) < remaining:
                missing = remaining - len(tie_positions)
                unused = np.setdiff1d(
                    np.arange(len(tied_indices)),
                    tie_positions,
                    assume_unique=True,
                )
                tie_positions = np.concatenate([tie_positions, unused[:missing]])
            chosen_ties = tied_indices[tie_positions[:remaining]]
        selected_indices = np.concatenate([above_indices, chosen_ties])
    else:
        selected_indices = above_indices[:keep_count]
    keep_mask = np.zeros_like(eligible_mask, dtype=bool)
    keep_mask[selected_indices] = True
    return keep_mask, threshold, keep_count, int(len(tied_indices))


def robust_upper_threshold(values, median_multiplier=4.0, mad_multiplier=8.0):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    robust_sigma = 1.4826 * mad
    threshold = max(
        median * median_multiplier,
        median + mad_multiplier * robust_sigma,
        1e-9,
    )
    return median, mad, float(threshold)


dense_points_world = np.asarray(
    omega_predictions["world_points_from_depth"],
    dtype=np.float32,
)
dense_confidence = np.asarray(omega_predictions["depth_conf"], dtype=np.float32)
if dense_confidence.ndim == 4 and dense_confidence.shape[-1] == 1:
    dense_confidence = dense_confidence[..., 0]
dense_images = np.asarray(omega_predictions["images"], dtype=np.float32)
if dense_images.ndim == 4 and dense_images.shape[1] == 3:
    dense_images_rgb = np.transpose(dense_images, (0, 2, 3, 1))
else:
    dense_images_rgb = dense_images

if dense_points_world.shape[:-1] != dense_confidence.shape:
    raise RuntimeError(
        f"צורות Point/Confidence אינן תואמות: {dense_points_world.shape} מול {dense_confidence.shape}"
    )
if dense_points_world.shape[:-1] != dense_images_rgb.shape[:-1]:
    raise RuntimeError("צורות Point/Image אינן תואמות.")

flat_points_world = dense_points_world.reshape(-1, 3)
flat_confidence = dense_confidence.reshape(-1)
flat_colors = np.clip(dense_images_rgb.reshape(-1, 3), 0.0, 1.0)
flat_colors_uint8 = np.rint(flat_colors * 255.0).astype(np.uint8)

eligible_mask = (
    np.isfinite(flat_points_world).all(axis=1)
    & np.isfinite(flat_confidence)
    & np.isfinite(flat_colors).all(axis=1)
    & (flat_confidence > 1e-5)
)
dense_depth_edge_count = 0
if DENSE_FILTER_DEPTH_EDGES:
    depth_edge_mask = omega_depth_edge_mask(
        omega_predictions["depth"],
        rtol=DENSE_DEPTH_EDGE_RTOL,
    ).reshape(-1)
    dense_depth_edge_count = int((eligible_mask & depth_edge_mask).sum())
    eligible_mask &= ~depth_edge_mask
if DENSE_MASK_BLACK_BACKGROUND:
    eligible_mask &= flat_colors_uint8.sum(axis=1) >= 16
if DENSE_MASK_WHITE_BACKGROUND:
    eligible_mask &= ~(
        (flat_colors_uint8[:, 0] > 240)
        & (flat_colors_uint8[:, 1] > 240)
        & (flat_colors_uint8[:, 2] > 240)
    )

dense_keep_mask, dense_confidence_threshold, dense_target_keep_count, dense_tie_count = (
    exact_percentile_keep_mask(
        flat_confidence,
        eligible_mask,
        DENSE_CONFIDENCE_PERCENTILE,
    )
)
filtered_points_world = flat_points_world[dense_keep_mask]
filtered_colors = flat_colors_uint8[dense_keep_mask]
if len(filtered_points_world) == 0:
    raise RuntimeError("סינון Confidence הסיר את כל נקודות Omega.")

dense_rng = np.random.default_rng(20260809)
if len(filtered_points_world) > DENSE_MAX_EXPORT_POINTS:
    export_indices = dense_rng.choice(
        len(filtered_points_world),
        size=DENSE_MAX_EXPORT_POINTS,
        replace=False,
    )
    export_indices.sort()
else:
    export_indices = np.arange(len(filtered_points_world))
export_points_world = filtered_points_world[export_indices]
export_colors = filtered_colors[export_indices]

omega_extrinsics = np.asarray(omega_predictions["extrinsic"], dtype=np.float64)
omega_intrinsics = np.asarray(omega_predictions["intrinsic"], dtype=np.float64)
rotations_world_to_camera = omega_extrinsics[:, :3, :3]
translations_world_to_camera = omega_extrinsics[:, :3, 3]
camera_centers_world = -np.einsum(
    "sji,sj->si",
    rotations_world_to_camera,
    translations_world_to_camera,
)
first_rotation = rotations_world_to_camera[0]
first_translation = translations_world_to_camera[0]
export_points_first = export_points_world @ first_rotation.T + first_translation
camera_centers_first = camera_centers_world @ first_rotation.T + first_translation

OMEGA_VISUAL_DIR = STAGE2_RUNTIME_DIR / "omega_visual"
OMEGA_VISUAL_IMAGES_DIR = OMEGA_VISUAL_DIR / "images"
if OMEGA_VISUAL_DIR.exists():
    shutil.rmtree(OMEGA_VISUAL_DIR)
OMEGA_VISUAL_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
for image_index, path_text in enumerate(omega_used_frames_df["frame_runtime_path"]):
    source_path = Path(path_text)
    shutil.copy2(
        source_path,
        OMEGA_VISUAL_IMAGES_DIR / f"{image_index:04d}{source_path.suffix.lower()}",
    )

DENSE_GLB_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_official_scene.glb"
)
official_scene = omega_predictions_to_glb(
    omega_predictions,
    conf_thres=DENSE_CONFIDENCE_PERCENTILE,
    mask_black_bg=DENSE_MASK_BLACK_BACKGROUND,
    mask_white_bg=DENSE_MASK_WHITE_BACKGROUND,
    show_cam=DENSE_SHOW_CAMERAS_IN_GLB,
    mask_sky=DENSE_FILTER_SKY,
    target_dir=str(OMEGA_VISUAL_DIR),
    max_points=DENSE_MAX_EXPORT_POINTS,
    filter_depth_edges=DENSE_FILTER_DEPTH_EDGES,
    depth_edge_rtol=DENSE_DEPTH_EDGE_RTOL,
)
official_scene.export(str(DENSE_GLB_RUNTIME_PATH))

DENSE_PLY_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_point_cloud.ply"
)
trimesh.points.PointCloud(
    vertices=export_points_first,
    colors=export_colors,
).export(str(DENSE_PLY_RUNTIME_PATH))

frame_count = len(camera_centers_first)
geometry_segment_index = omega_used_frames_df["geometry_segment_index"].to_numpy(dtype=int)
retained_segment_index = omega_used_frames_df["segment_index"].to_numpy(dtype=int)
source_times = omega_used_frames_df["source_time_sec"].to_numpy(dtype=float)
source_time_gap = np.full(frame_count, np.nan, dtype=float)
if frame_count > 1:
    source_time_gap[1:] = np.diff(source_times)
component_change = np.zeros(frame_count, dtype=bool)
retained_segment_change = np.zeros(frame_count, dtype=bool)
if frame_count > 1:
    component_change[1:] = geometry_segment_index[1:] != geometry_segment_index[:-1]
    retained_segment_change[1:] = retained_segment_index[1:] != retained_segment_index[:-1]
motion_break = component_change | retained_segment_change
source_time_gap[motion_break] = np.nan
motion_edge_comparable = (~motion_break) & np.isfinite(source_time_gap) & (source_time_gap > 0)

steps = np.full(frame_count, np.nan, dtype=float)
if frame_count > 1:
    steps[1:] = np.linalg.norm(np.diff(camera_centers_first, axis=0), axis=1)
steps[~motion_edge_comparable] = np.nan
translation_speed = steps / source_time_gap

rotation_angle_deg = np.full(frame_count, np.nan, dtype=float)
if frame_count > 1:
    relative_rotations = np.einsum(
        "sij,skj->sik",
        rotations_world_to_camera[1:],
        rotations_world_to_camera[:-1],
    )
    trace_values = np.trace(relative_rotations, axis1=1, axis2=2)
    cos_angles = np.clip((trace_values - 1.0) / 2.0, -1.0, 1.0)
    rotation_angle_deg[1:] = np.degrees(np.arccos(cos_angles))
rotation_angle_deg[~motion_edge_comparable] = np.nan
angular_speed_deg = rotation_angle_deg / source_time_gap

valid_edge_mask = motion_edge_comparable & np.isfinite(translation_speed)
translation_median, translation_mad, translation_threshold = robust_upper_threshold(
    translation_speed[valid_edge_mask]
)
rotation_median, rotation_mad, rotation_threshold = robust_upper_threshold(
    angular_speed_deg[valid_edge_mask],
    median_multiplier=4.5,
    mad_multiplier=9.0,
)
translation_edge_warning = (
    np.isfinite(translation_threshold)
    & (translation_speed > translation_threshold)
)
rotation_edge_warning = (
    np.isfinite(rotation_threshold)
    & (angular_speed_deg > rotation_threshold)
)
translation_edge_warning[~motion_edge_comparable] = False
rotation_edge_warning[~motion_edge_comparable] = False

input_continuity_warning = omega_used_frames_df[
    "input_continuity_warning"
].fillna(False).to_numpy(dtype=bool)
input_frame_trusted = omega_used_frames_df[
    "input_frame_trusted"
].fillna(False).to_numpy(dtype=bool)

isolated_translation_pose = np.zeros(frame_count, dtype=bool)
for pose_index in range(1, frame_count - 1):
    if not motion_edge_comparable[pose_index] or not motion_edge_comparable[pose_index + 1]:
        continue
    incoming_edge = translation_edge_warning[pose_index]
    outgoing_edge = translation_edge_warning[pose_index + 1]
    if incoming_edge and outgoing_edge:
        chord = float(np.linalg.norm(
            camera_centers_first[pose_index + 1]
            - camera_centers_first[pose_index - 1]
        ))
        detour = float(steps[pose_index] + steps[pose_index + 1])
        if detour > 0 and chord / detour < 0.65:
            isolated_translation_pose[pose_index] = True

endpoint_pose_warning = np.zeros(frame_count, dtype=bool)
if frame_count >= 2:
    endpoint_pose_warning[-1] = bool(
        translation_edge_warning[-1]
        and (
            input_continuity_warning[-1]
            or translation_speed[-1] > max(translation_median * 8.0, 1e-9)
        )
    )

pose_finite = np.isfinite(camera_centers_first).all(axis=1)
pose_trusted = (
    pose_finite
    & input_frame_trusted
    & ~isolated_translation_pose
    & ~endpoint_pose_warning
)

severe_edge_warning = translation_edge_warning & input_continuity_warning
# Never draw or interpolate a straight line across a warned translation jump.
# The direct poses remain in the raw CSV; only the unsupported connecting edge is broken.
path_break_before = motion_break | translation_edge_warning
for pose_index in range(frame_count):
    if not pose_trusted[pose_index]:
        path_break_before[pose_index] = True
        if pose_index + 1 < frame_count:
            path_break_before[pose_index + 1] = True
    if severe_edge_warning[pose_index]:
        path_break_before[pose_index] = True
path_break_before[0] = True

warning_reasons = []
for pose_index in range(frame_count):
    reasons = []
    if not input_frame_trusted[pose_index]:
        reasons.append("INPUT_FRAME_UNTRUSTED")
    if input_continuity_warning[pose_index]:
        reasons.append("LOW_INPUT_OVERLAP")
    if translation_edge_warning[pose_index]:
        reasons.append("TRANSLATION_SPEED_OUTLIER")
    if rotation_edge_warning[pose_index]:
        reasons.append("ROTATION_SPEED_OUTLIER")
    if isolated_translation_pose[pose_index]:
        reasons.append("ISOLATED_POSE_SPIKE")
    if endpoint_pose_warning[pose_index]:
        reasons.append("UNRELIABLE_ENDPOINT")
    warning_reasons.append("|".join(reasons))

trajectory_df = omega_used_frames_df[
    [
        "model_frame_index",
        "candidate_index",
        "segment_index",
        "geometry_segment_index",
        "source_time_sec",
        "clean_time_sec",
        "input_frame_trusted",
        "input_continuity_warning",
        "incoming_selected_inliers",
        "incoming_selected_ratio",
        "incoming_selected_coverage",
    ]
].copy()
trajectory_df["x_raw_relative"] = camera_centers_first[:, 0]
trajectory_df["y_raw_relative"] = camera_centers_first[:, 1]
trajectory_df["z_raw_relative"] = camera_centers_first[:, 2]
trajectory_df["incoming_time_gap_sec"] = source_time_gap
trajectory_df["incoming_edge_comparable"] = motion_edge_comparable
trajectory_df["incoming_step_distance"] = steps
trajectory_df["incoming_translation_speed"] = translation_speed
trajectory_df["incoming_rotation_deg"] = rotation_angle_deg
trajectory_df["incoming_angular_speed_deg_s"] = angular_speed_deg
trajectory_df["translation_edge_warning"] = translation_edge_warning
trajectory_df["rotation_edge_warning"] = rotation_edge_warning
trajectory_df["pose_trusted"] = pose_trusted
trajectory_df["pose_warning_reason"] = warning_reasons
trajectory_df["path_break_before"] = path_break_before
trajectory_df["x_trusted_relative"] = np.where(
    pose_trusted, camera_centers_first[:, 0], np.nan
)
trajectory_df["y_trusted_relative"] = np.where(
    pose_trusted, camera_centers_first[:, 1], np.nan
)
trajectory_df["z_trusted_relative"] = np.where(
    pose_trusted, camera_centers_first[:, 2], np.nan
)

trusted_indices = np.flatnonzero(pose_trusted)
if len(trusted_indices) == 0:
    raise RuntimeError("לא נשאר אף Pose מהימן לאחר בדיקות האיכות.")
last_trusted_index = int(trusted_indices[-1])

DENSE_TRAJECTORY_RUNTIME_CSV_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_camera_trajectory_raw_trusted.csv"
)
trajectory_df.to_csv(DENSE_TRAJECTORY_RUNTIME_CSV_PATH, index=False)

if len(export_points_first) > DENSE_MAX_PLOT_POINTS:
    plot_indices = dense_rng.choice(
        len(export_points_first),
        size=DENSE_MAX_PLOT_POINTS,
        replace=False,
    )
else:
    plot_indices = np.arange(len(export_points_first))
plot_points = export_points_first[plot_indices]
plot_colors = export_colors[plot_indices] / 255.0
plot_points_xyz = np.column_stack([
    plot_points[:, 0],
    plot_points[:, 2],
    -plot_points[:, 1],
])
trajectory_plot_xyz = np.column_stack([
    camera_centers_first[:, 0],
    camera_centers_first[:, 2],
    -camera_centers_first[:, 1],
])

point_lower = np.percentile(plot_points_xyz, 2, axis=0)
point_upper = np.percentile(plot_points_xyz, 98, axis=0)
point_span = np.maximum(point_upper - point_lower, 1e-6)
visible_points = np.all(
    (plot_points_xyz >= point_lower - 0.25 * point_span)
    & (plot_points_xyz <= point_upper + 0.25 * point_span),
    axis=1,
)

fig = plt.figure(figsize=(15, 10))
axis = fig.add_subplot(111, projection="3d")
axis.set_facecolor("#071018")
fig.patch.set_facecolor("#071018")
axis.scatter(
    plot_points_xyz[visible_points, 0],
    plot_points_xyz[visible_points, 1],
    plot_points_xyz[visible_points, 2],
    c=plot_colors[visible_points],
    s=0.35,
    alpha=0.68,
    linewidths=0,
    depthshade=False,
)

raw_plot_run = np.cumsum(path_break_before.astype(np.int64))
raw_label_used = False
for run_id in np.unique(raw_plot_run):
    run_positions = np.flatnonzero(raw_plot_run == run_id)
    if len(run_positions) < 2:
        continue
    run_xyz = trajectory_plot_xyz[run_positions]
    axis.plot(
        run_xyz[:, 0],
        run_xyz[:, 1],
        run_xyz[:, 2],
        color="#94a3b8",
        linestyle=":",
        linewidth=1.5,
        alpha=0.75,
        label="raw trajectory" if not raw_label_used else None,
    )
    raw_label_used = True

progress_colors = LinearSegmentedColormap.from_list(
    "trusted_path", ["#00d8ff", "#22d3a7", "#ffd166"]
)(np.linspace(0.0, 1.0, frame_count))
for pose_index in range(1, frame_count):
    if (
        pose_trusted[pose_index - 1]
        and pose_trusted[pose_index]
        and not path_break_before[pose_index]
    ):
        segment = np.stack([
            trajectory_plot_xyz[pose_index - 1],
            trajectory_plot_xyz[pose_index],
        ])[None, ...]
        axis.add_collection3d(Line3DCollection(
            segment,
            colors=[progress_colors[pose_index - 1]],
            linewidths=3.4,
            alpha=0.98,
        ))

axis.scatter(
    trajectory_plot_xyz[pose_trusted, 0],
    trajectory_plot_xyz[pose_trusted, 1],
    trajectory_plot_xyz[pose_trusted, 2],
    color="#22d3a7",
    s=24,
    label="trusted poses",
)
untrusted_mask = ~pose_trusted
if untrusted_mask.any():
    axis.scatter(
        trajectory_plot_xyz[untrusted_mask, 0],
        trajectory_plot_xyz[untrusted_mask, 1],
        trajectory_plot_xyz[untrusted_mask, 2],
        color="#ff3b30",
        marker="x",
        s=80,
        linewidths=2.0,
        label="untrusted pose",
    )
first_trusted_index = int(trusted_indices[0])
axis.scatter(
    *trajectory_plot_xyz[first_trusted_index],
    color="#39ff88",
    marker="o",
    s=95,
    edgecolors="black",
    label="first trusted pose",
)
axis.scatter(
    *trajectory_plot_xyz[last_trusted_index],
    color="#ffd166",
    marker="*",
    s=150,
    edgecolors="black",
    label="last trusted pose",
)

trusted_xyz = trajectory_plot_xyz[pose_trusted]
combined_lower = np.minimum(point_lower, trusted_xyz.min(axis=0))
combined_upper = np.maximum(point_upper, trusted_xyz.max(axis=0))
combined_span = np.maximum(combined_upper - combined_lower, 1e-6)
combined_center = (combined_lower + combined_upper) / 2.0
half_span = float(combined_span.max()) * 0.55
axis.set_xlim(combined_center[0] - half_span, combined_center[0] + half_span)
axis.set_ylim(combined_center[1] - half_span, combined_center[1] + half_span)
axis.set_zlim(combined_center[2] - half_span, combined_center[2] + half_span)
axis.set_xlabel("X — right", color="white")
axis.set_ylabel("Z — forward", color="white")
axis.set_zlabel("Height = -Y", color="white")
axis.tick_params(colors="white")
axis.set_title(
    "VGGT-Omega — point cloud + raw/trusted camera trajectory",
    color="white",
    fontsize=15,
    pad=18,
)
axis.legend(loc="upper left")
axis.view_init(elev=24, azim=-58)
plt.tight_layout()
DENSE_PLOT_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_point_cloud_trajectory.png"
)
fig.savefig(DENSE_PLOT_RUNTIME_PATH, dpi=210, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()

if len(plot_points_xyz) > DENSE_MAX_INTERACTIVE_POINTS:
    interactive_indices = dense_rng.choice(
        len(plot_points_xyz),
        size=DENSE_MAX_INTERACTIVE_POINTS,
        replace=False,
    )
else:
    interactive_indices = np.arange(len(plot_points_xyz))
interactive_points = plot_points_xyz[interactive_indices]
interactive_colors = plot_colors[interactive_indices]
color_strings = [
    f"rgb({int(r * 255)},{int(g * 255)},{int(b * 255)})"
    for r, g, b in interactive_colors
]
hover_text = [
    (
        f"frame {int(row.model_frame_index)}"
        f"<br>clean {float(row.clean_time_sec):.2f}s"
        f"<br>trusted {bool(row.pose_trusted)}"
        f"<br>{row.pose_warning_reason or 'OK'}"
    )
    for row in trajectory_df.itertuples()
]
interactive_figure = go.Figure()
interactive_figure.add_trace(go.Scatter3d(
    x=interactive_points[:, 0],
    y=interactive_points[:, 1],
    z=interactive_points[:, 2],
    mode="markers",
    marker={"size": 1.2, "color": color_strings, "opacity": 0.62},
    name="Omega point cloud",
    hoverinfo="skip",
))
for run_id in np.unique(raw_plot_run):
    positions = np.flatnonzero(raw_plot_run == run_id)
    if len(positions) < 2:
        continue
    xyz = trajectory_plot_xyz[positions]
    interactive_figure.add_trace(go.Scatter3d(
        x=xyz[:, 0], y=xyz[:, 1], z=xyz[:, 2],
        mode="lines",
        line={"color": "#94a3b8", "width": 2, "dash": "dot"},
        name=f"raw path run {int(run_id)}",
        hoverinfo="skip",
    ))
trusted_line_x, trusted_line_y, trusted_line_z = [], [], []
for pose_index in range(1, frame_count):
    if pose_trusted[pose_index - 1] and pose_trusted[pose_index] and not path_break_before[pose_index]:
        for coordinate_list, axis_index in [
            (trusted_line_x, 0), (trusted_line_y, 1), (trusted_line_z, 2)
        ]:
            coordinate_list.extend([
                trajectory_plot_xyz[pose_index - 1, axis_index],
                trajectory_plot_xyz[pose_index, axis_index],
                None,
            ])
interactive_figure.add_trace(go.Scatter3d(
    x=trusted_line_x, y=trusted_line_y, z=trusted_line_z,
    mode="lines",
    line={"color": "#22d3a7", "width": 6},
    name="trusted path",
    hoverinfo="skip",
))
interactive_figure.add_trace(go.Scatter3d(
    x=trajectory_plot_xyz[:, 0],
    y=trajectory_plot_xyz[:, 1],
    z=trajectory_plot_xyz[:, 2],
    mode="markers",
    marker={
        "size": 5,
        "color": ["#22d3a7" if trusted else "#ff3b30" for trusted in pose_trusted],
    },
    text=hover_text,
    hovertemplate="%{text}<extra></extra>",
    name="camera poses",
))
interactive_figure.update_layout(
    title="VGGT-Omega — Raw and Trusted trajectory",
    paper_bgcolor="#071018",
    plot_bgcolor="#071018",
    font={"color": "#e6edf3"},
    height=780,
    margin={"l": 0, "r": 0, "t": 70, "b": 0},
    scene={
        "xaxis_title": "X — right",
        "yaxis_title": "Z — forward",
        "zaxis_title": "Height = -Y",
        "aspectmode": "data",
        "bgcolor": "#071018",
    },
)
DENSE_INTERACTIVE_HTML_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_interactive_viewer.html"
)
interactive_figure.write_html(
    str(DENSE_INTERACTIVE_HTML_RUNTIME_PATH),
    include_plotlyjs=True,
    full_html=True,
)

fx_values = omega_intrinsics[:, 0, 0]
fy_values = omega_intrinsics[:, 1, 1]
focal_length_cv = float(np.std(fx_values) / max(np.mean(fx_values), 1e-9))

stage2_artifact_runtime_paths = [
    KEYFRAME_AUDIT_RUNTIME_PATH,
    MODEL_FRAMES_RUNTIME_PATH,
    KEYFRAME_CONTACT_SHEET_RUNTIME_PATH,
    OMEGA_USED_FRAMES_RUNTIME_PATH,
    DENSE_TRAJECTORY_RUNTIME_CSV_PATH,
    DENSE_GLB_RUNTIME_PATH,
    DENSE_PLY_RUNTIME_PATH,
    DENSE_PLOT_RUNTIME_PATH,
    DENSE_INTERACTIVE_HTML_RUNTIME_PATH,
]
for runtime_path in stage2_artifact_runtime_paths:
    if not runtime_path.is_file() or runtime_path.stat().st_size == 0:
        raise RuntimeError(f"קובץ Stage 2 חסר או ריק: {runtime_path}")
    shutil.copy2(runtime_path, STAGE2_DRIVE_OUTPUT_DIR / runtime_path.name)

display(trajectory_df)


In [ ]:
stage1_logic_sha256_for_dense = (
    stage1_quality_summary.get("baseline_logic_sha256")
    or stage1_quality_summary.get("parent_baseline_logic_sha256")
)

geometry_segment_count = int(
    omega_used_frames_df["geometry_segment_index"].nunique()
)
untrusted_pose_count = int((~pose_trusted).sum())
pose_warning_count = int(sum(bool(reason) for reason in warning_reasons))
continuity_warning_count = int(input_continuity_warning.sum())
severe_edge_warning_count = int(severe_edge_warning.sum())
translation_warning_count = int(translation_edge_warning.sum())
rotation_warning_count = int(rotation_edge_warning.sum())

quality_warning_reasons = []
if candidate_decode_fallback_count > 0:
    quality_warning_reasons.append("CANDIDATE_TERMINAL_DECODE_CLAMP")
if geometry_segment_count > 1:
    quality_warning_reasons.append("MULTIPLE_GEOMETRY_SEGMENTS")
if continuity_warning_count > 0:
    quality_warning_reasons.append("LOW_INPUT_OVERLAP")
if untrusted_pose_count > 0:
    quality_warning_reasons.append("UNTRUSTED_POSES")
if translation_warning_count > 0:
    quality_warning_reasons.append("TRANSLATION_SPEED_OUTLIERS")
if rotation_warning_count > 0:
    quality_warning_reasons.append("ROTATION_SPEED_OUTLIERS")
if focal_length_cv > 0.20:
    quality_warning_reasons.append("UNSTABLE_INTRINSICS")
if omega_frame_fallback_used:
    quality_warning_reasons.append("GPU_FRAME_FALLBACK")

untrusted_fraction = float(untrusted_pose_count / max(len(pose_trusted), 1))
if omega_gpu_quality_degraded:
    dense_quality_status = "FAIL_GPU_QUALITY_FLOOR"
elif geometry_segment_count > 1:
    dense_quality_status = "NEEDS_REVIEW_DISCONNECTED_INPUT"
elif severe_edge_warning_count > 0 or untrusted_fraction > 0.10:
    dense_quality_status = "NEEDS_REVIEW"
elif quality_warning_reasons:
    dense_quality_status = "PASS_WITH_WARNINGS"
else:
    dense_quality_status = "PASS"

STAGE2_SUMMARY_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_stage2_summary.json"
)
STAGE2_SUMMARY_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / STAGE2_SUMMARY_RUNTIME_PATH.name

stage2_output_files = {
    path.name: str(STAGE2_DRIVE_OUTPUT_DIR / path.name)
    for path in stage2_artifact_runtime_paths
}
stage2_output_files[STAGE2_SUMMARY_RUNTIME_PATH.name] = str(STAGE2_SUMMARY_DRIVE_PATH)

stage2_summary = {
    "schema_version": 5,
    "pipeline_version": PIPELINE_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "stage2_version": STAGE2B_VERSION,
    "stage1_source_version": STAGE1_SOURCE_VERSION,
    "stage1_logic_sha256": stage1_logic_sha256_for_dense,
    "source_video_name": SOURCE_VIDEO_NAME,
    "model": {
        "family": "VGGT-Omega",
        "model_id": OMEGA_MODEL_ID,
        "checkpoint_filename": OMEGA_CHECKPOINT_FILENAME,
        "code_repository": OMEGA_REPO_URL,
        "code_commit": OMEGA_REPO_COMMIT,
        "image_resolution": OMEGA_IMAGE_RESOLUTION,
        "preprocess_mode": OMEGA_PREPROCESS_MODE,
        "precision": OMEGA_PRECISION,
        "compute_dtype": omega_run_result["compute_dtype"],
        "forward_policy": OMEGA_FORWARD_POLICY,
        "attention_backend": omega_run_result["attention_backend"],
        "checkpoint_cache_mode": omega_checkpoint_cache_mode_used,
        "stage2_resume_safe": True,
    },
    "keyframe_selection": {
    "profile": "official_video_sampler_targets_quality_aware_v2",
        "candidate_fps": float(KEYFRAME_CANDIDATE_FPS),
        "target_fps": float(KEYFRAME_TARGET_FPS),
        "candidate_count": int(len(candidate_frames_df)),
        "selected_count": int(len(model_frames_df)),
        "model_used_count": int(len(omega_used_frames_df)),
        "isolated_candidate_count": int(candidate_frames_df["isolated_candidate"].sum()),
        "exposure_warning_candidate_count": int(candidate_frames_df["exposure_warning"].sum()),
        "forced_untrusted_selection_count": int(
            model_frames_df["forced_untrusted_selection"].sum()
        ),
        "input_continuity_warning_count": continuity_warning_count,
        "geometry_segment_count": geometry_segment_count,
        "lossless_candidate_format": "PNG",
        "candidate_cache_hits": int(candidate_frames_df["candidate_cache_hit"].sum()),
        "candidate_cache_misses": int((~candidate_frames_df["candidate_cache_hit"] ).sum()),
        "decode_fallback_count": int(candidate_decode_fallback_count),
        "max_decode_fallback_frames": int(candidate_frames_df["decode_fallback_frames"].max()),
    },
    "gpu": {
        "name": gpu_properties.name,
        "total_memory_gb": float(GPU_TOTAL_MEMORY_GB),
        "peak_allocated_memory_gb": float(omega_run_result["peak_gpu_memory_gb"]),
        "torch_version": torch.__version__,
        "precision": OMEGA_PRECISION,
        "compute_dtype": omega_run_result["compute_dtype"],
        "attention_backend": omega_run_result["attention_backend"],
        "frame_fallback_used": bool(omega_frame_fallback_used),
        "quality_degraded": bool(omega_gpu_quality_degraded),
        "quality_guard_enabled": bool(not OMEGA_ALLOW_DEGRADED_GPU_RESULT),
        "inference_attempts": omega_inference_attempts,
    },
    "model_input_width": int(omega_run_result["input_width"]),
    "model_input_height": int(omega_run_result["input_height"]),
    "point_cloud": {
        "source": "depth_unprojection",
        "finite_input_point_count": int(np.isfinite(flat_points_world).all(axis=1).sum()),
        "eligible_point_count_after_depth_edges": int(eligible_mask.sum()),
        "depth_edge_rejected_count": dense_depth_edge_count,
        "confidence_percentile": float(DENSE_CONFIDENCE_PERCENTILE),
        "confidence_threshold_value": float(dense_confidence_threshold),
        "confidence_threshold_tie_count": int(dense_tie_count),
        "confidence_filtered_point_count": int(len(filtered_points_world)),
        "exported_point_count": int(len(export_points_first)),
        "exact_percentile_tie_policy": "deterministic_even_tie_sampling",
    },
    "intrinsics_diagnostics": {
        "fx_min": float(fx_values.min()),
        "fx_median": float(np.median(fx_values)),
        "fx_max": float(fx_values.max()),
        "fy_min": float(fy_values.min()),
        "fy_median": float(np.median(fy_values)),
        "fy_max": float(fy_values.max()),
        "focal_length_cv": float(focal_length_cv),
        "camera_is_physically_fixed": True,
        "intrinsics_were_not_forced_or_smoothed": True,
    },
    "trajectory_quality": {
        "raw_pose_count": int(len(pose_trusted)),
        "trusted_pose_count": int(pose_trusted.sum()),
        "untrusted_pose_count": untrusted_pose_count,
        "pose_warning_count": pose_warning_count,
        "translation_edge_warning_count": translation_warning_count,
        "rotation_edge_warning_count": rotation_warning_count,
        "severe_edge_warning_count": severe_edge_warning_count,
        "last_trusted_model_frame_index": last_trusted_index,
        "last_trusted_source_time_sec": float(
            trajectory_df.loc[last_trusted_index, "source_time_sec"]
        ),
        "last_trusted_clean_time_sec": float(
            trajectory_df.loc[last_trusted_index, "clean_time_sec"]
        ),
        "translation_speed_diagnostics": {
            "median": None if not np.isfinite(translation_median) else float(translation_median),
            "mad": None if not np.isfinite(translation_mad) else float(translation_mad),
            "outlier_threshold": None if not np.isfinite(translation_threshold) else float(translation_threshold),
        },
        "angular_speed_diagnostics_deg_s": {
            "median": None if not np.isfinite(rotation_median) else float(rotation_median),
            "mad": None if not np.isfinite(rotation_mad) else float(rotation_mad),
            "outlier_threshold": None if not np.isfinite(rotation_threshold) else float(rotation_threshold),
        },
    },
    "coordinate_system": {
        "origin": "first Omega camera center",
        "x": "right in the first camera frame",
        "y": "down in the first camera frame",
        "z": "forward in the first camera frame",
        "scale": "relative model units; not meters",
    },
    "quality_status": dense_quality_status,
    "quality_warning_reasons": quality_warning_reasons,
    "limitations": [
        "Raw and trusted camera coordinates are relative, not metric.",
        "No GPS latitude, longitude, or absolute altitude is inferred.",
        "Trusted coordinates are masked, never smoothed or invented.",
        "Warned translation edges remain in the raw CSV but are not connected or interpolated in trajectory views.",
        "Multiple disconnected geometry segments require review before interpreting one global path.",
        "Per-frame dense trajectory interpolation is a later stage; this file contains selected keyframes.",
    ],
    "output_files": stage2_output_files,
}

STAGE2_SUMMARY_RUNTIME_PATH.write_text(
    json.dumps(stage2_summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
shutil.copy2(STAGE2_SUMMARY_RUNTIME_PATH, STAGE2_SUMMARY_DRIVE_PATH)
stage2_artifact_runtime_paths.append(STAGE2_SUMMARY_RUNTIME_PATH)

stage2_export_table = pd.DataFrame([
    {
        "file": path.name,
        "drive_path": str(STAGE2_DRIVE_OUTPUT_DIR / path.name),
        "size_mb": round(path.stat().st_size / (1024 ** 2), 3),
    }
    for path in stage2_artifact_runtime_paths
])
display(pd.DataFrame([{
    "quality_status": dense_quality_status,
    "raw_poses": len(pose_trusted),
    "trusted_poses": int(pose_trusted.sum()),
    "geometry_segments": geometry_segment_count,
    "warnings": ", ".join(quality_warning_reasons) or "none",
}]))


In [ ]:
#@title Frame trajectory { display-mode: "form" }
from scipy.spatial.transform import Rotation, Slerp

STAGE3_VERSION = "stage3_per_frame_research_decode_aware"
PIPELINE_VERSION = "fpv_reconstruction_resilient_decode"
PER_FRAME_ANALYSIS_WIDTH = 320

# Build an exact frame/time map from the lossless Stage 1 manifest.
per_frame_rows = []
clean_frame_index = 0
for segment in stage1_manifest_df.itertuples(index=False):
    segment_index = int(segment.segment_index)
    fps = float(segment.segment_fps)
    frame_count = int(segment.segment_frame_count)
    source_start_frame = int(segment.source_start_frame)
    clean_start = float(segment.clean_timeline_start_sec)
    for local_frame_index in range(frame_count):
        source_frame_index = source_start_frame + local_frame_index
        per_frame_rows.append({
            "clean_frame_index": clean_frame_index,
            "segment_index": segment_index,
            "segment_frame_index": local_frame_index,
            "source_frame_index": source_frame_index,
            "source_time_sec": source_frame_index / fps,
            "clean_time_sec": clean_start + local_frame_index / fps,
            "fps": fps,
        })
        clean_frame_index += 1

per_frame_trajectory_df = pd.DataFrame(per_frame_rows)
if per_frame_trajectory_df.empty:
    raise RuntimeError("Stage 3 cannot run because the Stage 1 manifest produced no frames.")

# Lightweight visual evidence for every retained frame. These are observations,
# not 3D coordinates, and are useful for future dense visual-odometry work.
visual_feature_rows = []
for segment in stage1_manifest_df.itertuples(index=False):
    segment_index = int(segment.segment_index)
    expected_frames = int(segment.segment_frame_count)
    segment_path = Path(segment.stage2_segment_path)
    capture = cv2.VideoCapture(str(segment_path))
    previous_gray = None
    for local_frame_index in range(expected_frames):
        ok, frame_bgr = capture.read()
        if not ok or frame_bgr is None:
            visual_feature_rows.append({
                "segment_index": segment_index,
                "segment_frame_index": local_frame_index,
                "frame_read_ok": False,
                "brightness_mean": np.nan,
                "contrast_std": np.nan,
                "sharpness_laplacian": np.nan,
                "clipped_ratio": np.nan,
                "visual_motion_mean_absdiff": np.nan,
            })
            previous_gray = None
            continue

        height, width = frame_bgr.shape[:2]
        if width > PER_FRAME_ANALYSIS_WIDTH:
            analysis_height = max(1, int(round(height * PER_FRAME_ANALYSIS_WIDTH / width)))
            frame_bgr = cv2.resize(
                frame_bgr,
                (PER_FRAME_ANALYSIS_WIDTH, analysis_height),
                interpolation=cv2.INTER_AREA,
            )
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        brightness = float(gray.mean())
        contrast = float(gray.std())
        sharpness = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        clipped_ratio = float(np.mean((gray <= 5) | (gray >= 250)))
        motion_absdiff = (
            float(cv2.absdiff(previous_gray, gray).mean())
            if previous_gray is not None and previous_gray.shape == gray.shape
            else np.nan
        )
        visual_feature_rows.append({
            "segment_index": segment_index,
            "segment_frame_index": local_frame_index,
            "frame_read_ok": True,
            "brightness_mean": brightness,
            "contrast_std": contrast,
            "sharpness_laplacian": sharpness,
            "clipped_ratio": clipped_ratio,
            "visual_motion_mean_absdiff": motion_absdiff,
        })
        previous_gray = gray
    capture.release()

visual_features_df = pd.DataFrame(visual_feature_rows)
per_frame_trajectory_df = per_frame_trajectory_df.merge(
    visual_features_df,
    on=["segment_index", "segment_frame_index"],
    how="left",
    validate="one_to_one",
)

# Coordinate and provenance columns. Absolute geographic fields are deliberately empty.
for column in [
    "x_raw_relative", "y_raw_relative", "z_raw_relative",
    "x_relative", "y_relative", "z_relative",
    "qx_relative", "qy_relative", "qz_relative", "qw_relative",
    "roll_relative_deg", "pitch_relative_deg", "yaw_relative_deg",
    "incoming_step_relative", "speed_relative_units_s",
    "latitude_deg", "longitude_deg", "altitude_m",
]:
    per_frame_trajectory_df[column] = np.nan

per_frame_trajectory_df["is_omega_keyframe"] = False
per_frame_trajectory_df["omega_model_frame_index"] = pd.Series(
    pd.array([pd.NA] * len(per_frame_trajectory_df), dtype="Int64")
)
per_frame_trajectory_df["geometry_segment_index"] = pd.Series(
    pd.array([pd.NA] * len(per_frame_trajectory_df), dtype="Int64")
)
per_frame_trajectory_df["omega_pose_trusted"] = False
per_frame_trajectory_df["position_available"] = False
per_frame_trajectory_df["trajectory_method"] = "UNAVAILABLE"
per_frame_trajectory_df["position_quality"] = "NO_TRUSTED_OMEGA_BRACKET"
per_frame_trajectory_df["coordinate_frame"] = "first_omega_camera_relative"
per_frame_trajectory_df["metric_scale_available"] = False
per_frame_trajectory_df["absolute_geolocation_available"] = False
per_frame_trajectory_df["path_break_before"] = True

# Express each Omega camera orientation in the first Omega camera frame.
camera_rotations_first = np.stack([
    first_rotation @ rotations_world_to_camera[index].T
    for index in range(len(rotations_world_to_camera))
])
camera_rotation_objects = Rotation.from_matrix(camera_rotations_first)
camera_quaternions_xyzw = camera_rotation_objects.as_quat()
camera_euler_xyz_deg = camera_rotation_objects.as_euler("xyz", degrees=True)

# Map each direct Omega pose to its exact/nearest retained frame.
anchor_dense_indices = []
for omega_index, omega_row in trajectory_df.reset_index(drop=True).iterrows():
    same_segment = per_frame_trajectory_df["segment_index"].eq(int(omega_row["segment_index"]))
    candidate_indices = per_frame_trajectory_df.index[same_segment].to_numpy()
    if len(candidate_indices) == 0:
        raise RuntimeError(f"No per-frame rows for Omega segment {omega_row['segment_index']}.")
    time_errors = np.abs(
        per_frame_trajectory_df.loc[candidate_indices, "source_time_sec"].to_numpy(float)
        - float(omega_row["source_time_sec"])
    )
    dense_index = int(candidate_indices[int(np.argmin(time_errors))])
    anchor_dense_indices.append(dense_index)

    trusted = bool(omega_row["pose_trusted"])
    per_frame_trajectory_df.loc[dense_index, [
        "x_raw_relative", "y_raw_relative", "z_raw_relative"
    ]] = [
        float(omega_row["x_raw_relative"]),
        float(omega_row["y_raw_relative"]),
        float(omega_row["z_raw_relative"]),
    ]
    per_frame_trajectory_df.loc[dense_index, "is_omega_keyframe"] = True
    per_frame_trajectory_df.loc[dense_index, "omega_model_frame_index"] = int(
        omega_row["model_frame_index"]
    )
    per_frame_trajectory_df.loc[dense_index, "geometry_segment_index"] = int(
        omega_row["geometry_segment_index"]
    )
    per_frame_trajectory_df.loc[dense_index, "omega_pose_trusted"] = trusted
    per_frame_trajectory_df.loc[dense_index, "trajectory_method"] = (
        "DIRECT_OMEGA" if trusted else "DIRECT_OMEGA_UNTRUSTED"
    )
    per_frame_trajectory_df.loc[dense_index, "position_quality"] = (
        "DIRECT_MODEL_TRUSTED" if trusted else "DIRECT_MODEL_UNTRUSTED"
    )
    if trusted:
        per_frame_trajectory_df.loc[dense_index, [
            "x_relative", "y_relative", "z_relative"
        ]] = camera_centers_first[omega_index]
        per_frame_trajectory_df.loc[dense_index, [
            "qx_relative", "qy_relative", "qz_relative", "qw_relative"
        ]] = camera_quaternions_xyzw[omega_index]
        per_frame_trajectory_df.loc[dense_index, [
            "roll_relative_deg", "pitch_relative_deg", "yaw_relative_deg"
        ]] = camera_euler_xyz_deg[omega_index]
        per_frame_trajectory_df.loc[dense_index, "position_available"] = True
        per_frame_trajectory_df.loc[dense_index, "path_break_before"] = bool(
            omega_row["path_break_before"]
        )

# Densify only between two trusted Omega anchors inside the same retained source segment.
# We intentionally do not bridge Stage 1 cuts, source-time gaps, or geometry components.
for left_index in range(len(trajectory_df) - 1):
    right_index = left_index + 1
    left = trajectory_df.iloc[left_index]
    right = trajectory_df.iloc[right_index]
    interval_is_trusted = (
        bool(left["pose_trusted"])
        and bool(right["pose_trusted"])
        and int(left["segment_index"]) == int(right["segment_index"])
        and int(left["geometry_segment_index"]) == int(right["geometry_segment_index"])
        and not bool(right["path_break_before"])
    )
    if not interval_is_trusted:
        continue

    left_time = float(left["clean_time_sec"])
    right_time = float(right["clean_time_sec"])
    if not right_time > left_time:
        continue

    same_segment = per_frame_trajectory_df["segment_index"].eq(int(left["segment_index"]))
    in_interval = (
        same_segment
        & per_frame_trajectory_df["clean_time_sec"].ge(left_time - 1e-9)
        & per_frame_trajectory_df["clean_time_sec"].le(right_time + 1e-9)
    )
    dense_indices = per_frame_trajectory_df.index[in_interval].to_numpy()
    if len(dense_indices) == 0:
        continue

    interval_times = per_frame_trajectory_df.loc[dense_indices, "clean_time_sec"].to_numpy(float)
    alpha = np.clip((interval_times - left_time) / (right_time - left_time), 0.0, 1.0)
    left_position = camera_centers_first[left_index]
    right_position = camera_centers_first[right_index]
    interpolated_positions = (
        (1.0 - alpha[:, None]) * left_position[None, :]
        + alpha[:, None] * right_position[None, :]
    )

    interval_rotation = Rotation.from_matrix(
        np.stack([camera_rotations_first[left_index], camera_rotations_first[right_index]])
    )
    slerp = Slerp([0.0, 1.0], interval_rotation)
    interpolated_rotation = slerp(alpha)
    interpolated_quaternions = interpolated_rotation.as_quat()
    interpolated_euler = interpolated_rotation.as_euler("xyz", degrees=True)

    per_frame_trajectory_df.loc[dense_indices, [
        "x_relative", "y_relative", "z_relative"
    ]] = interpolated_positions
    per_frame_trajectory_df.loc[dense_indices, [
        "qx_relative", "qy_relative", "qz_relative", "qw_relative"
    ]] = interpolated_quaternions
    per_frame_trajectory_df.loc[dense_indices, [
        "roll_relative_deg", "pitch_relative_deg", "yaw_relative_deg"
    ]] = interpolated_euler
    per_frame_trajectory_df.loc[dense_indices, "geometry_segment_index"] = int(
        left["geometry_segment_index"]
    )
    per_frame_trajectory_df.loc[dense_indices, "position_available"] = True
    per_frame_trajectory_df.loc[dense_indices, "path_break_before"] = False

    not_direct = ~per_frame_trajectory_df.loc[dense_indices, "is_omega_keyframe"].to_numpy(bool)
    interpolation_indices = dense_indices[not_direct]
    per_frame_trajectory_df.loc[interpolation_indices, "trajectory_method"] = "SE3_INTERPOLATION"
    per_frame_trajectory_df.loc[interpolation_indices, "position_quality"] = (
        "INTERPOLATED_BETWEEN_TRUSTED_OMEGA_ANCHORS"
    )

# Restore direct-anchor labels after interval fills.
for omega_index, dense_index in enumerate(anchor_dense_indices):
    trusted = bool(trajectory_df.iloc[omega_index]["pose_trusted"])
    per_frame_trajectory_df.loc[dense_index, "trajectory_method"] = (
        "DIRECT_OMEGA" if trusted else "DIRECT_OMEGA_UNTRUSTED"
    )
    per_frame_trajectory_df.loc[dense_index, "position_quality"] = (
        "DIRECT_MODEL_TRUSTED" if trusted else "DIRECT_MODEL_UNTRUSTED"
    )

# Never assign an inferred pose to a frame the decoder could not actually read.
frame_decode_ok = per_frame_trajectory_df["frame_read_ok"].fillna(False).to_numpy(bool)
decode_failed = ~frame_decode_ok
decode_guard_columns = [
    "x_relative", "y_relative", "z_relative",
    "qx_relative", "qy_relative", "qz_relative", "qw_relative",
    "roll_relative_deg", "pitch_relative_deg", "yaw_relative_deg",
]
per_frame_trajectory_df.loc[decode_failed, decode_guard_columns] = np.nan
per_frame_trajectory_df.loc[decode_failed, "omega_pose_trusted"] = False
per_frame_trajectory_df.loc[decode_failed, "position_available"] = False
per_frame_trajectory_df.loc[decode_failed, "trajectory_method"] = "UNAVAILABLE"
per_frame_trajectory_df.loc[decode_failed, "position_quality"] = "FRAME_DECODE_FAILED"
per_frame_trajectory_df.loc[decode_failed, "path_break_before"] = True
# Step/speed diagnostics only for consecutive available frames inside one retained segment.
xyz = per_frame_trajectory_df[["x_relative", "y_relative", "z_relative"]].to_numpy(float)
available = per_frame_trajectory_df["position_available"].to_numpy(bool)
same_segment_as_previous = per_frame_trajectory_df["segment_index"].eq(
    per_frame_trajectory_df["segment_index"].shift()
).to_numpy(bool)
valid_step = available & np.roll(available, 1) & same_segment_as_previous
valid_step[0] = False
step_values = np.full(len(per_frame_trajectory_df), np.nan, dtype=float)
step_values[valid_step] = np.linalg.norm(
    xyz[valid_step] - xyz[np.flatnonzero(valid_step) - 1], axis=1
)
per_frame_trajectory_df["incoming_step_relative"] = step_values
per_frame_trajectory_df["speed_relative_units_s"] = step_values * per_frame_trajectory_df["fps"].to_numpy(float)
per_frame_trajectory_df.loc[~valid_step, "path_break_before"] = True

PER_FRAME_TRAJECTORY_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_per_frame_research_trajectory.csv"
)
PER_FRAME_TRAJECTORY_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / PER_FRAME_TRAJECTORY_RUNTIME_PATH.name
per_frame_trajectory_df.to_csv(PER_FRAME_TRAJECTORY_RUNTIME_PATH, index=False)
shutil.copy2(PER_FRAME_TRAJECTORY_RUNTIME_PATH, PER_FRAME_TRAJECTORY_DRIVE_PATH)

# Truthful trajectory views: no smoothing, no false connections, and true data proportions.
PER_FRAME_PLOT_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_per_frame_trajectory.png"
)
PER_FRAME_PLOT_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / PER_FRAME_PLOT_RUNTIME_PATH.name

plot_xyz = np.column_stack([
    per_frame_trajectory_df["x_relative"].to_numpy(float),
    per_frame_trajectory_df["z_relative"].to_numpy(float),
    -per_frame_trajectory_df["y_relative"].to_numpy(float),
])
plot_available = (
    per_frame_trajectory_df["position_available"].to_numpy(bool)
    & np.isfinite(plot_xyz).all(axis=1)
)
plot_break = (
    per_frame_trajectory_df["path_break_before"].to_numpy(bool)
    | ~plot_available
)
plot_run_id = np.cumsum(plot_break.astype(np.int64))
segment_values = per_frame_trajectory_df["segment_index"].to_numpy(int)

if not plot_available.any():
    raise RuntimeError("No available coordinates remain for the trajectory visualization.")

segment_colors = {
    int(segment_index): plt.get_cmap("tab10")(color_index % 10)
    for color_index, segment_index in enumerate(sorted(np.unique(segment_values)))
}

figure = plt.figure(figsize=(17, 9), constrained_layout=True)
grid = figure.add_gridspec(2, 2, width_ratios=[1.35, 1.0])
axis_3d = figure.add_subplot(grid[:, 0], projection="3d")
axis_top = figure.add_subplot(grid[0, 1])
axis_side = figure.add_subplot(grid[1, 1])

labelled_segments = set()
for run_id in np.unique(plot_run_id[plot_available]):
    positions = np.flatnonzero(plot_available & (plot_run_id == run_id))
    if len(positions) == 0:
        continue
    segment_index = int(segment_values[positions[0]])
    color = segment_colors[segment_index]
    label = f"retained segment {segment_index}" if segment_index not in labelled_segments else None
    labelled_segments.add(segment_index)
    axis_3d.plot(
        plot_xyz[positions, 0], plot_xyz[positions, 1], plot_xyz[positions, 2],
        color=color, linewidth=2.4, label=label,
    )
    axis_top.plot(
        plot_xyz[positions, 0], plot_xyz[positions, 1],
        color=color, linewidth=2.2,
    )
    axis_side.plot(
        plot_xyz[positions, 1], plot_xyz[positions, 2],
        color=color, linewidth=2.2,
    )

direct_mask = (
    per_frame_trajectory_df["is_omega_keyframe"].to_numpy(bool)
    & per_frame_trajectory_df["omega_pose_trusted"].to_numpy(bool)
    & plot_available
)
axis_3d.scatter(
    plot_xyz[direct_mask, 0], plot_xyz[direct_mask, 1], plot_xyz[direct_mask, 2],
    s=28, c="black", depthshade=False, label="direct VGGT-Omega pose",
)
axis_top.scatter(plot_xyz[direct_mask, 0], plot_xyz[direct_mask, 1], s=20, c="black", zorder=3)
axis_side.scatter(plot_xyz[direct_mask, 1], plot_xyz[direct_mask, 2], s=20, c="black", zorder=3)

available_positions = np.flatnonzero(plot_available)
first_position = int(available_positions[0])
last_position = int(available_positions[-1])
for position, color, marker, label in [
    (first_position, "#16a34a", "o", "start"),
    (last_position, "#d97706", "*", "end"),
]:
    axis_3d.scatter(*plot_xyz[position], s=105, c=color, marker=marker, edgecolors="white", linewidths=0.8, label=label)
    axis_top.scatter(plot_xyz[position, 0], plot_xyz[position, 1], s=70, c=color, marker=marker, edgecolors="white", linewidths=0.7, zorder=4)
    axis_side.scatter(plot_xyz[position, 1], plot_xyz[position, 2], s=70, c=color, marker=marker, edgecolors="white", linewidths=0.7, zorder=4)

# Matplotlib's default 3D box stretches axes independently. Match the box to the
# actual coordinate ranges so small lateral/vertical noise is not visually magnified.
available_xyz = plot_xyz[plot_available]
data_span = np.ptp(available_xyz, axis=0)
largest_span = max(float(data_span.max()), 1e-9)
display_span = np.maximum(data_span, largest_span * 0.02)
axis_3d.set_box_aspect(display_span)
axis_3d.set_proj_type("ortho")
axis_3d.view_init(elev=24, azim=-58)
axis_3d.set_xlabel("X — right (relative)")
axis_3d.set_ylabel("Z — forward (relative)")
axis_3d.set_zlabel("Up = -Y (relative)")
axis_3d.set_title("3D camera path — true relative proportions", pad=16)
axis_3d.legend(loc="upper left", fontsize=9)

axis_top.set_aspect("equal", adjustable="datalim")
axis_top.set_xlabel("X — right (relative)")
axis_top.set_ylabel("Z — forward (relative)")
axis_top.set_title("Top view — equal scale")
axis_top.grid(True, alpha=0.25)

axis_side.set_aspect("equal", adjustable="datalim")
axis_side.set_xlabel("Z — forward (relative)")
axis_side.set_ylabel("Up = -Y (relative)")
axis_side.set_title("Side view — equal scale")
axis_side.grid(True, alpha=0.25)

figure.suptitle(
    "Relative camera trajectory — direct poses plus within-segment interpolation",
    fontsize=16,
)
figure.text(
    0.5, 0.01,
    "No smoothing. Removed intervals, unavailable poses, and warned translation jumps are not connected.",
    ha="center", fontsize=10, color="#475569",
)
figure.savefig(PER_FRAME_PLOT_RUNTIME_PATH, dpi=190, bbox_inches="tight")
plt.show()
shutil.copy2(PER_FRAME_PLOT_RUNTIME_PATH, PER_FRAME_PLOT_DRIVE_PATH)

PER_FRAME_SCHEMA_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_omega_per_frame_schema.json"
)
PER_FRAME_SCHEMA_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / PER_FRAME_SCHEMA_RUNTIME_PATH.name
per_frame_summary = {
    "schema_version": 1,
    "stage3_version": STAGE3_VERSION,
    "source_video_name": SOURCE_VIDEO_NAME,
    "row_count": int(len(per_frame_trajectory_df)),
    "direct_omega_pose_count": int(per_frame_trajectory_df["is_omega_keyframe"].sum()),
    "trusted_direct_omega_pose_count": int(
        (per_frame_trajectory_df["is_omega_keyframe"] & per_frame_trajectory_df["omega_pose_trusted"]).sum()
    ),
    "interpolated_frame_count": int(
        per_frame_trajectory_df["trajectory_method"].eq("SE3_INTERPOLATION").sum()
    ),
    "unavailable_frame_count": int((~per_frame_trajectory_df["position_available"] ).sum()),
    "frame_decode_failed_count": int(decode_failed.sum()),
    "coordinate_system": {
        "origin": "first Omega camera center",
        "x": "right in the first Omega camera frame",
        "y": "down in the first Omega camera frame",
        "z": "forward in the first Omega camera frame",
        "scale": "relative model units; not meters",
    },
    "metric_scale_available": False,
    "absolute_geolocation_available": False,
    "method_contract": {
        "DIRECT_OMEGA": "Direct model pose at a selected trusted keyframe.",
        "SE3_INTERPOLATION": "Interpolation only between trusted Omega anchors in the same retained source segment.",
        "UNAVAILABLE": "No defensible trusted bracket; coordinates intentionally left empty.",
    },
    "limitations": [
        "Interpolated rows are not independent Omega predictions.",
        "No interpolation crosses Stage 1 cuts, source gaps, disconnected geometry, untrusted anchors, or warned translation jumps.",
        "Trajectory figures use true coordinate proportions and do not smooth the direct model poses.",
        "Relative coordinates cannot be converted to latitude/longitude/absolute altitude without external evidence.",
    ],
    "output_files": {
        "per_frame_csv": str(PER_FRAME_TRAJECTORY_DRIVE_PATH),
        "per_frame_plot": str(PER_FRAME_PLOT_DRIVE_PATH),
    },
}
with open(PER_FRAME_SCHEMA_RUNTIME_PATH, "w", encoding="utf-8") as handle:
    json.dump(per_frame_summary, handle, indent=2, ensure_ascii=False)
shutil.copy2(PER_FRAME_SCHEMA_RUNTIME_PATH, PER_FRAME_SCHEMA_DRIVE_PATH)

display(pd.DataFrame([{
    "all retained frames": len(per_frame_trajectory_df),
    "direct Omega keyframes": int(per_frame_trajectory_df["is_omega_keyframe"].sum()),
    "interpolated trusted frames": int(per_frame_trajectory_df["trajectory_method"].eq("SE3_INTERPOLATION").sum()),
    "coordinates intentionally unavailable": int((~per_frame_trajectory_df["position_available"]).sum()),
}]))

In [ ]:
#@title Save results { display-mode: "form" }
from html import escape
from IPython.display import HTML

PIPELINE_RESULT_RUNTIME_PATH = STAGE2_RUNTIME_OUTPUT_DIR / (
    f"{SOURCE_VIDEO_STEM}_pipeline_result.json"
)
PIPELINE_RESULT_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / PIPELINE_RESULT_RUNTIME_PATH.name

pipeline_output_paths = {
    "clean_video": Path(DRIVE_FINAL_VIDEO_PATH),
    "segments_manifest": Path(DRIVE_MANIFEST_CSV_PATH),
    "stage1_quality": Path(DRIVE_QUALITY_JSON_PATH),
    "keyframe_audit": KEYFRAME_AUDIT_DRIVE_PATH,
    "selected_keyframes": MODEL_FRAMES_DRIVE_PATH,
    "keyframe_contact_sheet": KEYFRAME_CONTACT_SHEET_DRIVE_PATH,
    "trajectory_raw_trusted_csv": STAGE2_DRIVE_OUTPUT_DIR / DENSE_TRAJECTORY_RUNTIME_CSV_PATH.name,
    "trajectory_per_frame_csv": PER_FRAME_TRAJECTORY_DRIVE_PATH,
    "trajectory_per_frame_plot": PER_FRAME_PLOT_DRIVE_PATH,
    "trajectory_per_frame_schema": PER_FRAME_SCHEMA_DRIVE_PATH,
    "omega_official_scene_glb": STAGE2_DRIVE_OUTPUT_DIR / DENSE_GLB_RUNTIME_PATH.name,
    "point_cloud_ply": STAGE2_DRIVE_OUTPUT_DIR / DENSE_PLY_RUNTIME_PATH.name,
    "trajectory_plot": STAGE2_DRIVE_OUTPUT_DIR / DENSE_PLOT_RUNTIME_PATH.name,
    "interactive_viewer": STAGE2_DRIVE_OUTPUT_DIR / DENSE_INTERACTIVE_HTML_RUNTIME_PATH.name,
    "stage2_summary": STAGE2_SUMMARY_DRIVE_PATH,
}
missing_pipeline_outputs = [
    name for name, path in pipeline_output_paths.items()
    if not Path(path).is_file() or Path(path).stat().st_size == 0
]
if missing_pipeline_outputs:
    raise RuntimeError(
        "הריצה לא הושלמה: חסרים תוצרי חובה — " + ", ".join(missing_pipeline_outputs)
    )

pipeline_status = (
    "COMPLETED"
    if dense_quality_status in {"PASS", "PASS_WITH_WARNINGS"}
    else "COMPLETED_REVIEW_REQUIRED"
)
pipeline_result = {
    "schema_version": 4,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "pipeline_version": PIPELINE_VERSION,
    "status": pipeline_status,
    "source_video_name": SOURCE_VIDEO_NAME,
    "stage1_version": STAGE1_VERSION,
    "stage2_version": STAGE2B_VERSION,
    "model_family": "VGGT-Omega",
    "model_id": OMEGA_MODEL_ID,
    "model_checkpoint": OMEGA_CHECKPOINT_FILENAME,
    "model_code_commit": OMEGA_REPO_COMMIT,
    "model_precision": OMEGA_PRECISION,
    "checkpoint_cache_mode": omega_checkpoint_cache_mode_used,
    "stage2_resume_safe": True,
    "candidate_frame_count": int(len(candidate_frames_df)),
    "model_frame_count": int(len(omega_used_frames_df)),
    "trusted_pose_count": int(pose_trusted.sum()),
    "untrusted_pose_count": int((~pose_trusted).sum()),
    "geometry_segment_count": geometry_segment_count,
    "gpu_frame_fallback_used": bool(omega_frame_fallback_used),
    "gpu_quality_degraded": bool(omega_gpu_quality_degraded),
    "gpu": stage2_summary["gpu"],
    "quality_status": dense_quality_status,
    "quality_warning_reasons": quality_warning_reasons,
    "coordinate_system": stage2_summary["coordinate_system"],
    "absolute_geolocation_available": False,
    "output_files": {name: str(path) for name, path in pipeline_output_paths.items()},
}
PIPELINE_RESULT_RUNTIME_PATH.write_text(
    json.dumps(pipeline_result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
shutil.copy2(PIPELINE_RESULT_RUNTIME_PATH, PIPELINE_RESULT_DRIVE_PATH)

pipeline_result_table = pd.DataFrame([
    {
        "output": name,
        "drive_path": str(path),
        "size_mb": round(Path(path).stat().st_size / (1024 ** 2), 3),
    }
    for name, path in pipeline_output_paths.items()
] + [{
    "output": "pipeline_result",
    "drive_path": str(PIPELINE_RESULT_DRIVE_PATH),
    "size_mb": round(PIPELINE_RESULT_DRIVE_PATH.stat().st_size / (1024 ** 2), 3),
}])

display(HTML(f"""
<div dir="rtl" style="border:1px solid #16a34a;border-radius:14px;background:#f0fdf4;
    padding:16px 18px;font-family:Arial,sans-serif">
  <div style="font-size:21px;font-weight:700;color:#166534">✓ הריצה המלאה הושלמה</div>
  <div style="margin-top:8px;color:#365747;line-height:1.7">
    Stage 1: {len(manifest_df)} מקטעים · Omega: {len(omega_used_frames_df)} Keyframes ·
    Pose מהימן: {int(pose_trusted.sum())}/{len(pose_trusted)} ·
    GPU fallback: {"כן" if omega_frame_fallback_used else "לא"} ·
    איכות: {escape(dense_quality_status)}
  </div>
  <div style="margin-top:8px;color:#596b66;overflow-wrap:anywhere">
    קובץ הסיכום: <code>{escape(str(PIPELINE_RESULT_DRIVE_PATH))}</code>
  </div>
</div>
"""))
display(pipeline_result_table.style.hide(axis="index").set_caption("כל תוצרי המערכת"))


## Results

The important downloads and reconstruction views appear first. Detailed XYZ and quality tables follow below them.


In [ ]:
#@title Results { display-mode: "form" }
DOWNLOAD_GLB = False #@param {type:"boolean"}
DOWNLOAD_MAIN_PNG = False #@param {type:"boolean"}
DOWNLOAD_XYZ_CSV = False #@param {type:"boolean"}
DOWNLOAD_PER_FRAME_CSV = False #@param {type:"boolean"}
DOWNLOAD_ALL_ZIP = False #@param {type:"boolean"}

from html import escape
from IPython.display import FileLink, HTML, Image as IPythonImage, Video, display

required_result_names = [
    "pipeline_result",
    "trajectory_df",
    "per_frame_trajectory_df",
    "DENSE_GLB_RUNTIME_PATH",
    "DENSE_PLOT_RUNTIME_PATH",
    "DENSE_TRAJECTORY_RUNTIME_CSV_PATH",
    "PER_FRAME_TRAJECTORY_RUNTIME_PATH",
]
missing_result_names = [name for name in required_result_names if name not in globals()]
if missing_result_names:
    raise RuntimeError("Run the notebook from the beginning. Missing: " + ", ".join(missing_result_names))

DELIVERABLES_RUNTIME_DIR = STAGE2_RUNTIME_OUTPUT_DIR / "deliverables"
DELIVERABLES_RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

deliverable_paths = [
    Path(DRIVE_FINAL_VIDEO_PATH),
    STAGE2_DRIVE_OUTPUT_DIR / DENSE_GLB_RUNTIME_PATH.name,
    STAGE2_DRIVE_OUTPUT_DIR / DENSE_PLY_RUNTIME_PATH.name,
    STAGE2_DRIVE_OUTPUT_DIR / DENSE_PLOT_RUNTIME_PATH.name,
    STAGE2_DRIVE_OUTPUT_DIR / DENSE_TRAJECTORY_RUNTIME_CSV_PATH.name,
    Path(PER_FRAME_TRAJECTORY_DRIVE_PATH),
    Path(PER_FRAME_PLOT_DRIVE_PATH),
    Path(PER_FRAME_SCHEMA_DRIVE_PATH),
    Path(STAGE2_SUMMARY_DRIVE_PATH),
    Path(PIPELINE_RESULT_DRIVE_PATH),
    Path(MODEL_FRAMES_DRIVE_PATH),
    Path(KEYFRAME_CONTACT_SHEET_DRIVE_PATH),
]

for artifact_path in deliverable_paths:
    artifact_path = Path(artifact_path)
    if artifact_path.is_file():
        shutil.copy2(artifact_path, DELIVERABLES_RUNTIME_DIR / artifact_path.name)

DELIVERABLES_ZIP_RUNTIME_PATH = Path(shutil.make_archive(
    str(STAGE2_RUNTIME_OUTPUT_DIR / f"{SOURCE_VIDEO_STEM}_fpv_reconstruction_deliverables"),
    "zip",
    root_dir=DELIVERABLES_RUNTIME_DIR,
))
DELIVERABLES_ZIP_DRIVE_PATH = STAGE2_DRIVE_OUTPUT_DIR / DELIVERABLES_ZIP_RUNTIME_PATH.name
shutil.copy2(DELIVERABLES_ZIP_RUNTIME_PATH, DELIVERABLES_ZIP_DRIVE_PATH)

trusted_count = int(trajectory_df["pose_trusted"].sum())
keyframe_count = int(len(trajectory_df))
geometry_count = int(trajectory_df["geometry_segment_index"].nunique())
direct_count = int(per_frame_trajectory_df["trajectory_method"].eq("DIRECT_OMEGA").sum())
interpolated_count = int(per_frame_trajectory_df["trajectory_method"].eq("SE3_INTERPOLATION").sum())
unavailable_count = int((~per_frame_trajectory_df["position_available"] ).sum())
decode_fallback_count = int(candidate_frames_df["decode_fallback"].ne("NONE").sum())
frame_decode_failed_count = int((~per_frame_trajectory_df["frame_read_ok"].fillna(False)).sum())
quality_status = str(pipeline_result.get("quality_status", "UNKNOWN"))
quality_warnings = pipeline_result.get("quality_warning_reasons", [])
warning_text = ", ".join(quality_warnings) if quality_warnings else "None"
status_color = "#15803d" if quality_status == "PASS" else "#b45309"

display(HTML(f"""
<div style="font-family:Arial,sans-serif;border:1px solid #dbe3ea;border-radius:16px;padding:18px;background:#f8fafc">
  <div style="font-size:24px;font-weight:750;color:#0f172a">Reconstruction complete</div>
  <div style="margin-top:5px;color:#475569;overflow-wrap:anywhere">{escape(str(SOURCE_VIDEO_NAME))}</div>
  <div style="display:flex;flex-wrap:wrap;gap:10px;margin-top:16px">
    <div style="background:white;border-radius:12px;padding:12px 16px;min-width:130px"><div style="color:#64748b">Quality</div><b style="font-size:20px;color:{status_color}">{escape(quality_status)}</b></div>
    <div style="background:white;border-radius:12px;padding:12px 16px;min-width:130px"><div style="color:#64748b">Trusted poses</div><b style="font-size:20px">{trusted_count}/{keyframe_count}</b></div>
    <div style="background:white;border-radius:12px;padding:12px 16px;min-width:130px"><div style="color:#64748b">3D components</div><b style="font-size:20px">{geometry_count}</b></div>
    <div style="background:white;border-radius:12px;padding:12px 16px;min-width:130px"><div style="color:#64748b">Frame rows</div><b style="font-size:20px">{len(per_frame_trajectory_df):,}</b></div>
    <div style="background:white;border-radius:12px;padding:12px 16px;min-width:130px"><div style="color:#64748b">Decode clamps</div><b style="font-size:20px">{decode_fallback_count}</b></div>
  </div>
  <div style="margin-top:14px;padding:11px 13px;border-radius:10px;background:#fff7ed;color:#9a3412">
    <b>Warnings:</b> {escape(warning_text)}<br>
    XYZ is relative and non-metric. It is not latitude, longitude, altitude, or distance in meters.
  </div>
</div>
"""))

result_files = pd.DataFrame([
    {"output": "3D scene (GLB)", "drive_path": str(STAGE2_DRIVE_OUTPUT_DIR / DENSE_GLB_RUNTIME_PATH.name)},
    {"output": "Main reconstruction (PNG)", "drive_path": str(STAGE2_DRIVE_OUTPUT_DIR / DENSE_PLOT_RUNTIME_PATH.name)},
    {"output": "Selected keyframes (PNG)", "drive_path": str(KEYFRAME_CONTACT_SHEET_DRIVE_PATH)},
    {"output": "Trajectory views — true scale (PNG)", "drive_path": str(PER_FRAME_PLOT_DRIVE_PATH)},
    {"output": "Trusted keyframe XYZ (CSV)", "drive_path": str(STAGE2_DRIVE_OUTPUT_DIR / DENSE_TRAJECTORY_RUNTIME_CSV_PATH.name)},
    {"output": "Per-frame XYZ and features (CSV)", "drive_path": str(PER_FRAME_TRAJECTORY_DRIVE_PATH)},
    {"output": "Clean video", "drive_path": str(DRIVE_FINAL_VIDEO_PATH)},
    {"output": "All important files (ZIP)", "drive_path": str(DELIVERABLES_ZIP_DRIVE_PATH)},
])
display(HTML("<h3>Main files</h3><p>Everything is saved permanently in Google Drive. Use the direct links below or download one ZIP containing the important deliverables.</p>"))
display(result_files.style.hide(axis="index"))
display(FileLink(str(DENSE_GLB_RUNTIME_PATH), result_html_prefix="GLB: "))
display(FileLink(str(DENSE_PLOT_RUNTIME_PATH), result_html_prefix="Main reconstruction PNG: "))
display(FileLink(str(KEYFRAME_CONTACT_SHEET_RUNTIME_PATH), result_html_prefix="Selected keyframes PNG: "))
display(FileLink(str(PER_FRAME_PLOT_RUNTIME_PATH), result_html_prefix="True-scale trajectory PNG: "))
display(FileLink(str(DENSE_TRAJECTORY_RUNTIME_CSV_PATH), result_html_prefix="Trusted XYZ CSV: "))
display(FileLink(str(PER_FRAME_TRAJECTORY_RUNTIME_PATH), result_html_prefix="Per-frame CSV: "))
display(FileLink(str(DELIVERABLES_ZIP_RUNTIME_PATH), result_html_prefix="All results ZIP: "))

display(HTML("<h3>Main reconstruction PNG</h3>"))
display(IPythonImage(filename=str(DENSE_PLOT_RUNTIME_PATH), width=1100))

display(HTML("<h3>Interactive 3D reconstruction</h3><p>Drag to rotate and scroll to zoom.</p>"))
display(interactive_figure)

display(HTML("<h3>Selected keyframes</h3><p>These are the source views used by VGGT-Omega.</p>"))
display(IPythonImage(filename=str(KEYFRAME_CONTACT_SHEET_RUNTIME_PATH), width=1100))

display(HTML("<h3>Relative trajectory — true scale</h3><p>Direct VGGT poses are not smoothed. Unsupported gaps and warned translation jumps are not connected.</p>"))
display(IPythonImage(filename=str(PER_FRAME_PLOT_RUNTIME_PATH), width=1100))

display(HTML("<h3>Clean flight video</h3>"))
display(Video(str(FINAL_VIDEO_PATH), embed=True, width=900))

trusted_xyz = trajectory_df.loc[
    trajectory_df["pose_trusted"],
    [
        "model_frame_index",
        "segment_index",
        "source_time_sec",
        "clean_time_sec",
        "x_trusted_relative",
        "y_trusted_relative",
        "z_trusted_relative",
        "incoming_translation_speed",
        "incoming_rotation_deg",
        "incoming_angular_speed_deg_s",
        "geometry_segment_index",
    ],
].copy()
trusted_xyz.columns = [
    "keyframe",
    "video_segment",
    "source_sec",
    "clean_sec",
    "x_relative",
    "y_relative",
    "z_relative",
    "relative_speed_units_s",
    "rotation_step_deg",
    "angular_speed_deg_s",
    "3d_component",
]
display(HTML("<h3>Trusted relative XYZ keyframes</h3>"))
display(trusted_xyz.round(6).style.hide(axis="index"))
continuous_motion = trajectory_df[
    trajectory_df["pose_trusted"]
    & (~trajectory_df["path_break_before"])
    & np.isfinite(trajectory_df["incoming_translation_speed"])
    & np.isfinite(trajectory_df["incoming_angular_speed_deg_s"])
].copy()

def motion_summary_row(label, values, unit):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {"measurement": label, "samples": 0, "median": np.nan, "p95": np.nan, "maximum": np.nan, "unit": unit}
    return {
        "measurement": label,
        "samples": int(len(values)),
        "median": float(np.percentile(values, 50)),
        "p95": float(np.percentile(values, 95)),
        "maximum": float(values.max()),
        "unit": unit,
    }

motion_sanity_table = pd.DataFrame([
    motion_summary_row(
        "Relative translation speed",
        continuous_motion["incoming_translation_speed"],
        "relative units / second",
    ),
    motion_summary_row(
        "Camera angular speed",
        continuous_motion["incoming_angular_speed_deg_s"],
        "degrees / second",
    ),
])
motion_warning_table = pd.DataFrame([{
    "translation outlier edges": int(trajectory_df["translation_edge_warning"].sum()),
    "rotation outlier edges": int(trajectory_df["rotation_edge_warning"].sum()),
    "continuity warnings": int(trajectory_df["input_continuity_warning"].sum()),
    "trajectory breaks": int(trajectory_df["path_break_before"].sum()),
}])

display(HTML(
    "<h3>Motion sanity</h3>"
    "<p>Angular speed is physically interpretable. Translation speed is only in relative model units until metric scale is recovered. "
    "Outlier counts detect jumps; they do not prove the reconstruction is correct.</p>"
))
display(motion_sanity_table.round(4).style.hide(axis="index"))
display(motion_warning_table.style.hide(axis="index"))

per_frame_preview_columns = [
    "clean_frame_index",
    "source_time_sec",
    "clean_time_sec",
    "x_relative",
    "y_relative",
    "z_relative",
    "trajectory_method",
    "position_quality",
]
display(HTML(
    f"<h3>Per-frame XYZ preview</h3><p>Direct: <b>{direct_count}</b> &nbsp; Interpolated: <b>{interpolated_count}</b> &nbsp; Intentionally unavailable: <b>{unavailable_count}</b> &nbsp; Decode failures: <b>{frame_decode_failed_count}</b>. The CSV contains every retained frame.</p>"
))
display(per_frame_trajectory_df[per_frame_preview_columns].head(30).round(6).style.hide(axis="index"))

if DOWNLOAD_GLB:
    files.download(str(DENSE_GLB_RUNTIME_PATH))
if DOWNLOAD_MAIN_PNG:
    files.download(str(DENSE_PLOT_RUNTIME_PATH))
if DOWNLOAD_XYZ_CSV:
    files.download(str(DENSE_TRAJECTORY_RUNTIME_CSV_PATH))
if DOWNLOAD_PER_FRAME_CSV:
    files.download(str(PER_FRAME_TRAJECTORY_RUNTIME_PATH))
if DOWNLOAD_ALL_ZIP:
    files.download(str(DELIVERABLES_ZIP_RUNTIME_PATH))